In [ ]:
# ============================================================
# 我们用Yfinance下载先前整理好的市值最大的100支在标普500里的股票的日内数据
# 其中包括收盘价，开盘价，最高价，最低价，调整后收盘价以及交易量
# ============================================================


import urllib.request
import json
import pandas as pd
import time
import datetime
import numpy as np
import matplotlib.pyplot as plt

tickers = pd.read_csv("selected_tickers.csv")["Ticker"].tolist()
t1 = int(datetime.datetime(2010, 1, 1).timestamp())
t2 = int(datetime.datetime(2026, 1, 1).timestamp())

print(f"股票池: {len(tickers)} 只")
print(f"字段: Open, High, Low, Close, Adj Close, Volume\n")

all_data = {}   # {ticker: DataFrame}
fail_list = []

for i, t in enumerate(tickers):
    try:
        url = (
            f"https://query1.finance.yahoo.com/v8/finance/chart/{t}"
            f"?period1={t1}&period2={t2}&interval=1d&events=history"
        )
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        resp = urllib.request.urlopen(req, timeout=30)
        data = json.loads(resp.read())

        result = data["chart"]["result"][0]
        ts = pd.to_datetime(result["timestamp"], unit="s")
        quote = result["indicators"]["quote"][0]

        df = pd.DataFrame({
            "Date": ts,
            "Open":   quote["open"],
            "High":   quote["high"],
            "Low":    quote["low"],
            "Close":  quote["close"],
            "Volume": quote["volume"],
        })

        # Adj Close（部分股票可能没有）
        if "adjclose" in result["indicators"]:
            df["Adj Close"] = result["indicators"]["adjclose"][0]["adjclose"]
        else:
            df["Adj Close"] = quote["close"]  # 无调整时等于 Close

        df = df.dropna(subset=["Close"]).set_index("Date")

        if len(df) > 200:
            all_data[t] = df
            print(f"  [{i+1:>3}/{len(tickers)}] ✓ {t:<6} ({len(df)} 天)")
        else:
            fail_list.append(t)
            print(f"  [{i+1:>3}/{len(tickers)}] ✗ {t:<6} 数据不足 ({len(df)} 天)")

    except Exception as e:
        fail_list.append(t)
        print(f"  [{i+1:>3}/{len(tickers)}] ✗ {t:<6} {str(e)[:50]}")

    time.sleep(1.2)

# ============================================================
# 合并 & 保存
# ============================================================
print(f"\n{'='*50}")
print(f"成功: {len(all_data)}/{len(tickers)}, 失败: {len(fail_list)}")

if not all_data:
    print("全部失败")
    exit(1)

# 合并为 MultiIndex 列: (Ticker, Field)
panel = pd.concat(all_data, axis=1)  # 自动变成 (ticker, field) MultiIndex
panel.index.name = "Date"
panel = panel.sort_index()

# 保存
panel.to_csv("stock_data.csv")
panel.to_parquet("stock_data.parquet")

# 提取 Adj Close 矩阵单独保存一份
adj_close = panel.xs("Adj Close", axis=1, level=1)
adj_close.to_csv("adj_close_prices.csv")

# 保存成功列表
pd.Series(list(all_data.keys()), name="Ticker").to_csv("valid_tickers.csv", index=False)

print(f"stock_data.csv / stock_data.parquet  — 全字段 (Open/High/Low/Close/Adj Close/Volume)")
print(f"adj_close_prices.csv               — 仅 Adj Close 矩阵")
print(f"形状: {panel.shape[0]} 天 × {panel.shape[1]} 列")


In [ ]:
stock_data = pd.read_csv("stock_data.csv", header=[0, 1], index_col=0,
parse_dates=True, low_memory=False)
adj_close = stock_data.xs("Adj Close", axis=1, level=1)

In [ ]:
full_days = len(adj_close)
print(f"交易日总数: {full_days}")
print(f"日期范围: {adj_close.index[0].date()} → {adj_close.index[-1].date()}")
print(f"股票数量: {adj_close.shape[1]}\n")

# 每只股票的缺失情况
missing = pd.DataFrame({
    "有效天数": adj_close.count(),
    "缺失天数": adj_close.isnull().sum(),
    "缺失比例": (adj_close.isnull().sum() / full_days * 100).round(1),
    "起始日期": [str(adj_close[c].first_valid_index()).split(" ")[0] if
adj_close[c].first_valid_index() else None for c in adj_close.columns],
    "结束日期": [str(adj_close[c].last_valid_index()).split(" ")[0] if
adj_close[c].last_valid_index() else None for c in adj_close.columns],
})

# ============================================================
# 我们将检测连续缺失 >= 30 天的股票
# ============================================================
print("=" * 60)
print("连续缺失 >= 30 天的股票")
print("=" * 60)

problem_stocks = []

for ticker in adj_close.columns:
    series = adj_close[ticker]
    mask = series.isnull()
    if mask.sum() == 0:
        continue

    # 标记每个缺失段的起止
    is_missing = mask.astype(int)
    change = is_missing.diff()
    starts = mask.index[(change == 1)].tolist()
    if mask.iloc[0]:
        starts = [mask.index[0]] + starts
    ends = mask.index[(change == -1)].tolist()
    if mask.iloc[-1]:
        ends = ends + [mask.index[-1]]

    for s, e in zip(starts, ends):
        days = (e - s).days
        if days >= 30:
            problem_stocks.append({"Ticker": ticker, "开始": s.date(), "结束":
e.date(), "天数": days})

if problem_stocks:
    df_prob = pd.DataFrame(problem_stocks).sort_values("天数",
ascending=False)
    print(f"共 {len(df_prob)} 段连续缺失 >= 30 天，涉及{df_prob['Ticker'].nunique()} 只股票\n")
    for _, r in df_prob.iterrows():
        reason = ""
        if r["开始"].year == 2010:
            reason = "← 2010 年后才上市"
        elif r["结束"].year == 2025:
            reason = "← 2025 年底前已退市"
        print(f"{r['Ticker']:<8} {r['开始']} → {r['结束']}  缺{r['天数']:>4} 天  {reason}")

      # 按股票汇总
    print(f"\n--- 按股票汇总 ---")
    for t in sorted(df_prob["Ticker"].unique()):
        sub = df_prob[df_prob["Ticker"] == t]
        first_date = missing.loc[t, "起始日期"]
        print(f"{t:<8} {len(sub)} 段缺失, 最长 {sub['天数'].max()} 天,数据从 {first_date} 开始")
else:
    print("未发现连续缺失 >= 30 天的股票")

# ============================================================
# 将缺失原因分类
# ============================================================
print(f"\n{'='*60}")
print("缺失原因分类")
print("=" * 60)
print("""
  1. 2010 年后 IPO  — 数据起始日期晚于 2010-01-01（最常见的"缺失"）
     → PYPL(2015), ABBV(2013), META(2012), NOW(2012), ZTS(2013) 等

  2. 数据源缺失      — Yahoo Finance 偶尔丢个别交易日
     → 通常只缺 1-2 天，不是连续大段

  3. 代码/名称变更    — BRK-B、GOOGL 等发生过 ticker 变更
     → 新旧代码各有一段数据

  4. 退市/被收购      — 数据在结束日期前中断
     → 如果在 2025-12-31 之前终止

  结论: 连续大段缺失几乎全是 IPO 时间晚于 2010 年造成的，
       属于"未上市"而非"数据缺失"，不影响分析。
  """)

print(f"缺失分布:")
print(f"  0 缺失:     {(missing['缺失天数'] == 0).sum()} 只")
print(f"  0-1% 缺失:  {((missing['缺失天数'] > 0) & (missing['缺失比例'] <= 
1)).sum()} 只")
print(f"  1-5% 缺失:  {((missing['缺失比例'] > 1) & (missing['缺失比例'] <= 
5)).sum()} 只")
print(f"  5-20% 缺失: {((missing['缺失比例'] > 5) & (missing['缺失比例'] <= 
20)).sum()} 只")
print(f"  >20% 缺失:  {(missing['缺失比例'] > 20).sum()} 只")

In [ ]:
adj_close = stock_data.xs("Adj Close", axis=1, level=1).copy()

print(f"数据: {adj_close.shape[0]} 天 × {adj_close.shape[1]} 只")
print(f"日期: {adj_close.index[0].date()} → {adj_close.index[-1].date()}")

# ============================================================
# 计算日收益率
# ============================================================
returns = adj_close.pct_change()
returns = returns.iloc[1:]  # 去掉第一天（全 NaN）
print(f"日收益率: {returns.shape[0]} 天 × {returns.shape[1]} 只")

# ============================================================
# 3. MAD 法标记极端值（由于股票数据存在极端值，如：下载的数据有错误等等，
#    为此，我们需要用特定的方法来处理极端值，所以在这里，我们先把极端值标记出来）
# ============================================================
THRESHOLD_ABS = 0.30     # 绝对值 > 30%
THRESHOLD_MAD = 8        # 偏离中位数 > 8 × MAD

extreme_list = []

for ticker in returns.columns:
    r = returns[ticker].dropna()
    if len(r) == 0:
        continue

    median = r.median()
    mad = (r - median).abs().median()

    for date, val in r.items():
        reasons = []

          # 绝对阈值
        if abs(val) > THRESHOLD_ABS:
            reasons.append(f"|ret|={val*100:+.1f}%")

          # MAD 阈值
        if mad > 0:
            mad_score = abs(val - median) / mad
            if mad_score > THRESHOLD_MAD:
                reasons.append(f"MAD={mad_score:.0f}")

        if reasons:
            price_before = adj_close.loc[returns.index[returns.index <
date][-1], ticker] if len(returns.index[returns.index < date]) > 0 else np.nan
            extreme_list.append({
                "Ticker": ticker,
                "Date": date,
                "Return%": round(val * 100, 2),
                "MAD_Score": round(abs(val - median) / mad, 1) if mad > 0 else
np.nan,
                "Price": round(adj_close.loc[date, ticker], 2),
                "Reasons": " | ".join(reasons),
            })

df_ext = pd.DataFrame(extreme_list).sort_values("Date")

def classify(row):
      t = row["Ticker"]
      d = row["Date"]
      ret = row["Return%"] / 100
      price = row["Price"]

      # 规则 1: 涨跌幅超出物理范围
      if ret > 1.0 or ret < -0.8:
          return "数据错误", "涨跌幅超物理范围"

      # 规则 2: 孤立跳变（前后都正常，仅单日异常）
      try:
          idx = returns.index.get_loc(d)
          if 1 < idx < len(returns) - 2:
              before = returns[t].iloc[idx-2:idx].abs().mean()
              after = returns[t].iloc[idx+1:idx+3].abs().mean()
              if before < 0.02 and after < 0.02 and abs(ret) > 0.2:
                  return "数据错误", "孤立跳变-前后正常仅单日异常"
      except:
          pass

      # 规则 3: 同日 >=3 只极端 -> 市场事件
      if (df_ext["Date"] == d).sum() >= 3:
          return "真实事件", f"同日{(df_ext['Date']==d).sum()}只极端-市场事件"

      # 规则 4: IPO 首周
      fv = adj_close[t].first_valid_index()
      if fv is not None and d <= fv + pd.Timedelta(days=5):
          return "真实事件", "IPO初期剧烈波动"

      # 规则 5: 股价 <$1
      if price < 1.0:
          return "可能错误", "股价<$1-精度问题"

      return "真实事件", "财报/公告/行业冲击(默认)"

if len(df_ext) > 0:
      df_ext[["Judgment", "Evidence"]] = df_ext.apply(
          lambda r: pd.Series(classify(r)), axis=1
      )

In [ ]:
# 我们将极端值设置为空值，然后采取用前一日的数据来填充今日数据的方法
# ============================================================
print(f"\n{'='*60}")
print("极端值处理")
print("=" * 60)

# 统计待处理数量
errors = df_ext[df_ext["Judgment"] == "数据错误"]
possible = df_ext[df_ext["Judgment"] == "可能错误"]
real_events = df_ext[df_ext["Judgment"] == "真实事件"]

print(f"  数据错误:  {len(errors)} 个 → 设为 NaN")
print(f"  可能错误:  {len(possible)} 个 → 设为 NaN（保守处理）")
print(f"  真实事件:  {len(real_events)} 个 → 保留\n")

# 对错误值：在 returns 和 adj_close 中对应位置设为 NaN
to_clean = pd.concat([errors, possible])

for _, row in to_clean.iterrows():
    t, d = row["Ticker"], row["Date"]
    returns.loc[d, t] = np.nan
    adj_close.loc[d, t] = np.nan

# 被清除的 NaN 用前一日价格填充（收益率层面等价于当日无变化）
adj_close = adj_close.ffill(limit=1)
returns = adj_close.pct_change().iloc[1:]

print(f"处理后极端值重新检测:")
# 快速验证 — 用同样阈值再扫一遍
check_list = []
for ticker in returns.columns:
    r = returns[ticker].dropna()
    if len(r) == 0:
        continue
    median = r.median()
    mad = (r - median).abs().median()
    for date, val in r.items():
        if abs(val) > THRESHOLD_ABS or (mad > 0 and abs(val - median) / mad >
THRESHOLD_MAD):
            check_list.append({"Ticker": ticker, "Date": date, "Return%":
round(val * 100, 2)})

if len(check_list) > 0:
    df_check = pd.DataFrame(check_list)
    print(f"  剩余极端值: {len(df_check)} 个（以下为真实事件，已保留）")

      # 再次自动判断
    def reclassify(row):
        t, d, ret = row["Ticker"], row["Date"], row["Return%"] / 100
        if ret > 1.0 or ret < -0.8:
            return "数据错误-仍有残留"
        idx_check = returns.index.get_loc(d)
        if 1 < idx_check < len(returns) - 2:
            b = returns[t].iloc[idx_check-2:idx_check].abs().mean()
            a = returns[t].iloc[idx_check+1:idx_check+3].abs().mean()
            if b < 0.02 and a < 0.02 and abs(ret) > 0.2:
                return "数据错误-仍有残留"
        if (df_check["Date"] == d).sum() >= 3:
            return "真实事件-市场冲击"
        return "真实事件-个股"

    df_check["判断"] = df_check.apply(reclassify, axis=1)
    print(df_check.to_string())
else:
    print("  无剩余极端值 ✓")

In [ ]:
print(f"数据: {adj_close.shape[0]} 天 x {adj_close.shape[1]} 只")

# ============================================================
# 由于我们在用MAD方法时，有可能会引入前视偏差，所以在这里逐项审查
# ============================================================
print("=" * 60)
print("前视偏差审查")
print("=" * 60)

print("\n1. pct_change() -- 仅用 t-1 和 t，无前视偏差 ✓")
print("2. ffill()       -- 仅用历史数据前向填充，无前视偏差 ✓")
print("3. fillna(0)     -- 仅填充残留 NaN，无前视偏差 ✓")
print("4. MAD 检测      -- 原方案用全样本统计量，存在前视偏差")
print("   -> 改用 expanding window 重新检测，每时点仅用历史数据")

# ============================================================
# Expanding Window MAD 检测（无前视偏差）
# ============================================================
print(f"\n{'='*60}")
print("Expanding Window MAD 极端值检测")
print("=" * 60)

THRESHOLD_ABS = 0.30
THRESHOLD_MAD = 8
MIN_WINDOW = 252  # 至少 1 年数据才开始检测

extreme_expanding = []

for ticker in returns.columns:
    r = returns[ticker].dropna()
    if len(r) < MIN_WINDOW:
        continue

    for i in range(MIN_WINDOW, len(r)):
        window = r.iloc[:i]       # 只用时刻 i 之前的数据（关键！）
        val = r.iloc[i]           # 当前值
        date = r.index[i]

        median = window.median()
        mad = (window - median).abs().median()

        flagged = False
        if abs(val) > THRESHOLD_ABS:
            flagged = True
        if mad > 0 and abs(val - median) / mad > THRESHOLD_MAD:
            flagged = True

        if flagged:
            extreme_expanding.append({
                "Ticker": ticker,
                "Date": date,
                "Return%": round(val * 100, 2),
                "MAD_Score": round(abs(val - median) / mad, 1) if mad > 0 else
np.nan,
            })

df_ext_exp = pd.DataFrame(extreme_expanding).sort_values("Date")

print(f"Expanding window 检出: {len(df_ext_exp)} 个极端值")
print(f"涉及股票: {df_ext_exp['Ticker'].nunique() if len(df_ext_exp) > 0 else 
0} 只")

if len(df_ext_exp) > 0:
    print("\n--- 按日期分布 ---")
    print(df_ext_exp.groupby(df_ext_exp["Date"].dt.year).size().to_string())
    print(f"\n--- Top 20 (按 MAD 得分) ---")
    print(df_ext_exp.sort_values("MAD_Score",
ascending=False).head(20).to_string())

  # ============================================================
  # 4. 审查结论
  # ============================================================
print(f"\n{'='*60}")
print("前视偏差审查结论")
print("=" * 60)
print("""
  步骤                方法                     前视偏差
  -----------------------------------------------------
  缺失值处理          跳过(无交易日中间缺失)    无 ✓
  极端值检测(改进后)  Expanding window MAD     无 ✓
  极端值处理          NaN + ffill + fillna(0)  无 ✓
  收益率计算          pct_change()             无 ✓
  -----------------------------------------------------
  结论: 当前所有清洗步骤均无前视偏差
  """)

In [ ]:
adj_close.to_csv("adj_close_clean.csv")
returns.to_csv("daily_returns_clean.csv")
print(f"\n已保存: adj_close_clean.csv / daily_returns_clean.csv")
# 清洗后的 Adj Close
adj_close_clean = pd.read_csv("adj_close_clean.csv", index_col=0,
parse_dates=True)

# 替换 stock_data 里的 Adj Close
stock_data.loc[:, (slice(None), "Adj Close")] = adj_close_clean.values

# 保存
stock_data.to_csv("stock_data_clean.csv")
print("已保存 stock_data_clean.csv（全字段 + 清洗后 Adj Close）")
stock_data_clean = pd.read_csv("stock_data_clean.csv", header=[0, 1], index_col=0, parse_dates=True, low_memory=False)

In [ ]:
# ============================================================
# 【数据修复·第一步】把 Open/High/Low 复权，统一到 Adj Close 体系
# 复权因子 = Adj Close / Close，逐股票调整 OHL
# NaN位置(上市前)会保持NaN，正确跳过
# ============================================================
import numpy as np
import pandas as pd

# 从硬盘读回清洗后的数据（不管kernel状态都能跑）
stock_data_clean = pd.read_csv("stock_data_clean.csv", header=[0,1], index_col=0,
                                parse_dates=True, low_memory=False)

tickers_list = stock_data_clean.columns.get_level_values(0).unique()
n_adjusted = 0

for tk in tickers_list:
    close = stock_data_clean[(tk, "Close")].replace(0, np.nan)
    adjc = stock_data_clean[(tk, "Adj Close")]
    factor = adjc / close                       # 复权因子
    for fld in ["Open", "High", "Low"]:
        stock_data_clean[(tk, fld)] = stock_data_clean[(tk, fld)] * factor
    stock_data_clean[(tk, "Close")] = adjc      # Close也换成Adj Close，全体系一致
    if (factor.dropna() - 1).abs().max() > 0.05:
        n_adjusted += 1

print(f"已复权 {len(tickers_list)} 只股票的 OHL")
print(f"其中 {n_adjusted} 只有明显复权调整（拆股/大额分红）")

# 保存（新文件名，不覆盖原始清洗数据，保留可追溯）
stock_data_clean.to_csv("stock_data_clean_adjusted.csv")
print("✓ 已保存 stock_data_clean_adjusted.csv")

# 抽查一只有明显复权的股票，确认OHLC同体系
for tk in tickers_list:
    factor = stock_data_clean[(tk,"Adj Close")] / stock_data_clean[(tk,"Close")].replace(0,np.nan)
    # 这里Close已被替换,重新从比例看不出来了,改抽查OHLC数值是否量级一致
    sample = stock_data_clean[tk][["Open","High","Low","Close"]].dropna().tail(2)
    print(f"\n抽查 {tk} 最近2天（OHLC应量级一致，High≥Low）:")
    print(sample.round(2))
    break

In [ ]:
# ---- 读入清洗后的数据 ----
# adj_close: 清洗后的复权收盘价（第2步产物）
adj_close = pd.read_csv("adj_close_clean.csv", index_col=0, parse_dates=True)

# volume: 从原始 stock_data 里取（成交量不需要清洗）
stock_data = pd.read_csv("stock_data.csv", header=[0,1], index_col=0,
                         parse_dates=True, low_memory=False)
volume = stock_data.xs("Volume", axis=1, level=1)
# 对齐到 adj_close 的日期和股票
volume = volume.reindex(index=adj_close.index, columns=adj_close.columns)

print(f"数据: {adj_close.shape[0]} 天 × {adj_close.shape[1]} 只")

# ============================================================
# 计算4个因子（在 日期×股票 宽表上算，每列是一只股票，天然不串数据）
# ============================================================
# 1. 20日动量：当日收盘价 / 20个交易日前收盘价 - 1
momentum_20d = adj_close / adj_close.shift(20) - 1

# 2. 5日短期反转：过去5日收益率（当日/5日前 - 1）
reversal_5d = adj_close / adj_close.shift(5) - 1

# 3. 20日波动率：过去20日日收益率的标准差
daily_ret = adj_close.pct_change()
volatility_20d = daily_ret.rolling(20).std()

# 4. 成交量比：过去20日均成交量 / 当日成交量（按作业清单，均量在分子）
vol_ma20 = volume.rolling(20).mean()
volume_ratio = volume / vol_ma20 

# ============================================================
# 未来5日收益（预测目标）：(未来第5日 / 当日) - 1
# 用 shift(-5) 取未来价格。t日的值 = 持有t到t+5的收益。
# 最后5天因无未来数据自动为NaN——这正是杜绝未来泄漏的体现。
# ============================================================
future_ret_5d = adj_close.shift(-5) / adj_close - 1

# ============================================================
# 整合成一张长表：索引(Date, Ticker)，列=4因子+未来收益
# ============================================================
def to_long(df, name):
    return df.stack().rename(name)

factor_table = pd.concat([
    to_long(momentum_20d,   "momentum_20d"),
    to_long(reversal_5d,    "reversal_5d"),
    to_long(volatility_20d, "volatility_20d"),
    to_long(volume_ratio,   "volume_ratio"),
    to_long(future_ret_5d,  "future_ret_5d"),
], axis=1)
factor_table.index.names = ["Date", "Ticker"]

print(f"\n整合后因子表: {factor_table.shape[0]} 行 × {factor_table.shape[1]} 列")
print(f"列: {factor_table.columns.tolist()}")

# 去掉含NaN的行（开头窗口不足 + 结尾无未来收益的行）
factor_table_clean = factor_table.dropna()
print(f"去NaN后: {factor_table_clean.shape[0]} 行")

# 看几行
print(f"\n前5行:")
print(factor_table_clean.head().round(4).to_string())

# 保存
factor_table_clean.to_pickle("factor_table.pkl")
print(f"\n✓ 已保存 factor_table.pkl（4因子 + 未来5日收益）")

In [ ]:
# ============================================================
# 【时间切分】建立 train/val/test（带 embargo，全程统一使用）
# 划分按任务清单：训练2010-2018 / 验证2019-2020 / 测试2021-2025
# embargo=5：每段末尾砍5个交易日，因为这些日的 future_ret_5d 会
#            看到下一段的价格，砍掉可杜绝跨段未来泄漏
# ============================================================

# 读入第3步的因子表（索引 Date, Ticker）
factor_table = pd.read_pickle("factor_table.pkl")

EMBARGO = 5
date_level = factor_table.index.get_level_values("Date")

def slice_period(start, end):
    m = (date_level >= pd.Timestamp(start)) & (date_level <= pd.Timestamp(end))
    return factor_table[m]

def drop_tail_days(df, k):
    """删掉每段最后k个交易日：它们的 future_ret_5d 会看到下一段价格"""
    if len(df) == 0:
        return df
    uniq = df.index.get_level_values("Date").unique().sort_values()
    if len(uniq) <= k:
        return df
    keep_until = uniq[-(k + 1)]
    return df[df.index.get_level_values("Date") <= keep_until]

train = drop_tail_days(slice_period("2010-01-01", "2018-12-31"), EMBARGO)
val   = drop_tail_days(slice_period("2019-01-01", "2020-12-31"), EMBARGO)
test  = slice_period("2021-01-01", "2025-12-31")   # 最后一段，尾部无标签的行已在第3步dropna

print("=== 时间切分（带 embargo=5）===")
for name, d in [("train", train), ("val", val), ("test", test)]:
    dd = d.index.get_level_values("Date")
    print(f"{name:6}: {len(d):>7} 行 | {dd.min().date()} → {dd.max().date()}")

# 保存三段，后面第4步、第5步都用
train.to_pickle("factor_train.pkl")
val.to_pickle("factor_val.pkl")
test.to_pickle("factor_test.pkl")
print("\n✓ 已保存 factor_train/val/test.pkl")

In [ ]:
# ============================================================
# 【第4步·A】单因子IC评价
# 全样本IC统计 + 分三区间IC(用带embargo的train/val/test,口径与后续统一)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

factor_table = pd.read_pickle("factor_table.pkl")
train = pd.read_pickle("factor_train.pkl")
val   = pd.read_pickle("factor_val.pkl")
test  = pd.read_pickle("factor_test.pkl")

FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]

def compute_daily_ic(ft, factor_cols):
    """逐日截面Spearman IC，返回 日期×因子 的DataFrame"""
    dates_idx = ft.index.get_level_values("Date")
    ic_records = {}
    for factor in factor_cols:
        ics = {}
        for date, group in ft.groupby(dates_idx):
            f = group[factor].values
            r = group["future_ret_5d"].values
            mask = np.isfinite(f) & np.isfinite(r)
            if mask.sum() >= 5 and np.std(f[mask]) > 1e-12:
                ic, _ = spearmanr(f[mask], r[mask])
                if not np.isnan(ic):
                    ics[date] = ic
        ic_records[factor] = pd.Series(ics)
    return pd.DataFrame(ic_records).sort_index()

# 全样本日度IC（用完整因子表）
daily_ic = compute_daily_ic(factor_table, FACTOR_COLS)
print(f"日度IC序列: {daily_ic.shape[0]} 天 × {daily_ic.shape[1]} 因子")

# 全样本4指标
summary = pd.DataFrame({
    "IC均值": daily_ic.mean(),
    "IC标准差": daily_ic.std(),
    "IR": daily_ic.mean() / daily_ic.std(),
    "IC>0占比": (daily_ic > 0).mean(),
})
print("\n" + "="*60)
print("单因子IC统计（全样本 2010-2025）")
print("="*60)
print(summary.round(4).to_string())

# 分区间IC均值（用带embargo的三段，口径与第5步一致）
period_ic = {}
for name, seg in [("训练(2010-2018)", train), ("验证(2019-2020)", val), ("测试(2021-2025)", test)]:
    ic_seg = compute_daily_ic(seg, FACTOR_COLS)
    period_ic[name] = ic_seg.mean()
split_result = pd.DataFrame(period_ic).T

print("\n" + "="*60)
print("分区间IC均值（带embargo，观察因子稳定性）")
print("="*60)
print(split_result.round(4).to_string())

daily_ic.to_pickle("daily_ic.pkl")
print("\n✓ 已保存 daily_ic.pkl")

In [ ]:
# ============================================================
# 【第4步·B】累计IC曲线
# 把每个因子的日度IC逐日累加，观察其稳定性和持续性
# 曲线越接近直线（斜率恒定）→ 因子越稳定；
# 方向不重要（正/负都可以有效），关键看是否平滑单调
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# 中文字体（本地Mac有效）
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

daily_ic = pd.read_pickle("daily_ic.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]

# 累计IC = 日度IC逐日累加
cum_ic = daily_ic.cumsum()

# ---- 画图 ----
fig, ax = plt.subplots(figsize=(13, 6))
colors = {"momentum_20d": "tab:blue", "reversal_5d": "tab:orange",
          "volatility_20d": "tab:green", "volume_ratio": "tab:red"}
labels = {"momentum_20d": "20日动量", "reversal_5d": "5日反转",
          "volatility_20d": "20日波动率", "volume_ratio": "成交量比"}

for col in FACTOR_COLS:
    ax.plot(cum_ic.index, cum_ic[col], label=f"{labels[col]} (IC均值={daily_ic[col].mean():+.4f})",
            linewidth=1.5, color=colors[col])

ax.axhline(0, color='gray', lw=0.8, ls='--')
# 标注三段分界（可选，帮助观察不同时期）
for boundary in ["2019-01-01", "2021-01-01"]:
    ax.axvline(pd.Timestamp(boundary), color='gray', lw=0.6, ls=':', alpha=0.6)

ax.set_title("各因子累计IC曲线（2010-2025）", fontsize=13)
ax.set_xlabel("日期")
ax.set_ylabel("累计IC")
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---- 打印各因子最终累计IC ----
print("=== 各因子最终累计IC值 ===")
for col in FACTOR_COLS:
    final = cum_ic[col].iloc[-1]
    print(f"  {labels[col]}: {final:+.2f}  (斜率方向={'上升↗' if final>0 else '下降↘'})")

In [ ]:
# ============================================================
# 【第4步·C】分组收益曲线 + 多空曲线（检查单调性）
# 每日按因子值将股票分5组(Q1最低~Q5最高)，算各组未来5日平均收益
# 检查单调性：因子有效则各组收益应有序排列
# 净值：因未来5日收益重叠，除以5做日化近似后再累乘（避免重叠高估）
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

factor_table = pd.read_pickle("factor_table.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]
labels_cn = {"momentum_20d": "20日动量", "reversal_5d": "5日反转",
             "volatility_20d": "20日波动率", "volume_ratio": "成交量比"}
N_GROUPS = 5

def quantile_returns(ft, factor, n_groups=5):
    """返回每日各组平均未来收益：索引=日期，列=Q1..Q5"""
    dates_idx = ft.index.get_level_values("Date")
    records = []
    for date, group in ft.groupby(dates_idx):
        f = group[factor]; r = group["future_ret_5d"]
        mask = f.notna() & r.notna()
        if mask.sum() < n_groups * 2:
            continue
        fv, rv = f[mask], r[mask]
        try:
            lab = pd.qcut(fv, n_groups, labels=False, duplicates="drop")
        except Exception:
            continue
        if pd.Series(lab).nunique() < n_groups:
            continue
        grp = rv.groupby(lab).mean()
        grp.name = date
        records.append(grp)
    res = pd.DataFrame(records)
    res.columns = [f"Q{i+1}" for i in range(res.shape[1])]
    return res.sort_index()

# ---- 4个因子各画一张：5组净值 + 多空 ----
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

summary_rows = []
for ax, factor in zip(axes, FACTOR_COLS):
    qret = quantile_returns(factor_table, factor, N_GROUPS)

    # 各组平均未来5日收益（看单调性）
    grp_mean = qret.mean()

    # 净值（日化后累乘）
    nav = (1 + qret / 5).cumprod()
    # 多空组合 Q5-Q1（也日化）
    long_short_nav = (1 + (qret["Q5"] - qret["Q1"]) / 5).cumprod()

    # 画5组净值
    cmap = plt.cm.RdYlGn(np.linspace(0.1, 0.9, N_GROUPS))
    for i, q in enumerate(qret.columns):
        ax.plot(nav.index, nav[q], color=cmap[i], linewidth=1.2, label=q)
    # 画多空
    ax.plot(long_short_nav.index, long_short_nav, color="black", linewidth=1.8,
            ls="--", label="多空(Q5-Q1)")
    ax.set_title(f"{labels_cn[factor]} — 分组净值", fontsize=12)
    ax.set_ylabel("净值"); ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)

    # 记录单调性
    diffs = grp_mean.diff().dropna()
    mono = "单调递增↗" if (diffs > 0).all() else ("单调递减↘" if (diffs < 0).all() else "非单调")
    summary_rows.append({
        "因子": labels_cn[factor],
        "Q1收益%": round(grp_mean.iloc[0]*100, 3),
        "Q5收益%": round(grp_mean.iloc[-1]*100, 3),
        "多空Q5-Q1%": round((grp_mean.iloc[-1]-grp_mean.iloc[0])*100, 3),
        "单调性": mono,
    })

plt.tight_layout()
plt.show()

# ---- 单调性汇总表 ----
print("=== 各因子分组收益单调性 ===")
print(pd.DataFrame(summary_rows).to_string(index=False))
print("\n各组平均未来5日收益（逐因子）:")
for factor in FACTOR_COLS:
    qret = quantile_returns(factor_table, factor, N_GROUPS)
    gm = qret.mean()
    print(f"\n{labels_cn[factor]}:")
    for q, v in gm.items():
        print(f"  {q}: {v*100:+.3f}%")

In [ ]:
# ============================================================
# 【第5步·A】机器学习因子合成（网格调参 + 合并重训）
# 流程：
#   1) 训练集训练、验证集选参（RF: depth×min_samples_leaf；
#      XGB: depth×lr×n_estimators）
#   2) 选定参数后，合并训练集+验证集重训最终模型（充分利用数据）
#   3) 测试集只评估一次（embargo保证不泄漏）
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from itertools import product
import pickle

train = pd.read_pickle("factor_train.pkl")
val   = pd.read_pickle("factor_val.pkl")
test  = pd.read_pickle("factor_test.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]

def cross_sectional_zscore(df, cols):
    df = df.copy()
    dates_idx = df.index.get_level_values("Date")
    for col in cols:
        grp = df.groupby(dates_idx)[col]
        df[col] = (df[col] - grp.transform("mean")) / grp.transform("std")
    return df

train_z = cross_sectional_zscore(train, FACTOR_COLS).dropna(subset=FACTOR_COLS)
val_z   = cross_sectional_zscore(val,   FACTOR_COLS).dropna(subset=FACTOR_COLS)
test_z  = cross_sectional_zscore(test,  FACTOR_COLS).dropna(subset=FACTOR_COLS)
print(f"标准化后样本: train={len(train_z)}, val={len(val_z)}, test={len(test_z)}")

X_train, y_train = train_z[FACTOR_COLS].values, train_z["future_ret_5d"].values
X_val = val_z[FACTOR_COLS].values

def daily_ic(model, data_z, X):
    tmp = data_z.copy()
    tmp["pred"] = model.predict(X)
    dates_idx = tmp.index.get_level_values("Date")
    ics = []
    for date, g in tmp.groupby(dates_idx):
        if len(g) >= 5 and g["pred"].std() > 1e-12:
            ic, _ = spearmanr(g["pred"], g["future_ret_5d"])
            if not np.isnan(ic):
                ics.append(ic)
    ics = np.array(ics)
    return ics.mean(), ics.std(), (ics > 0).mean()

# ============================================================
# 步骤1：网格选参（train训练，val选）
# ============================================================
print("\n=== 随机森林网格调参（验证集选）===")
best_rf_params, best_rf_ic = None, -np.inf
for depth, msl, ne in product([3, 4, 5], [20, 50, 100], [100, 200]):
    m = RandomForestRegressor(n_estimators=ne, max_depth=depth,
                               min_samples_leaf=msl, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    v_ic = daily_ic(m, val_z, X_val)[0]
    if v_ic > best_rf_ic:
        best_rf_ic, best_rf_params = v_ic, (depth, msl, ne)
print(f"  最优: max_depth={best_rf_params[0]}, min_samples_leaf={best_rf_params[1]}, "
      f"n_estimators={best_rf_params[2]} (验证IC={best_rf_ic:+.4f})")

print("\n=== XGBoost网格调参（验证集选）===")
best_xgb_params, best_xgb_ic = None, -np.inf
for depth, lr, ne in product([3, 4, 5], [0.01, 0.03, 0.05, 0.1], [100, 200, 300]):
    m = xgb.XGBRegressor(n_estimators=ne, max_depth=depth, learning_rate=lr,
                          random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    v_ic = daily_ic(m, val_z, X_val)[0]
    if v_ic > best_xgb_ic:
        best_xgb_ic, best_xgb_params = v_ic, (depth, lr, ne)
print(f"  最优: max_depth={best_xgb_params[0]}, learning_rate={best_xgb_params[1]}, "
      f"n_estimators={best_xgb_params[2]} (验证IC={best_xgb_ic:+.4f})")

# ============================================================
# 步骤2：合并 train+val，用最优参数重训最终模型
# ============================================================
trainval_z = pd.concat([train_z, val_z])
X_tv, y_tv = trainval_z[FACTOR_COLS].values, trainval_z["future_ret_5d"].values
print(f"\n合并 train+val 重训: {len(trainval_z)} 样本（2010-2020）")

rf = RandomForestRegressor(n_estimators=200, max_depth=best_rf_params[0],
                            min_samples_leaf=best_rf_params[1],
                            random_state=42, n_jobs=-1).fit(X_tv, y_tv)
xgbm = xgb.XGBRegressor(n_estimators=best_xgb_params[2], max_depth=best_xgb_params[0],
                         learning_rate=best_xgb_params[1], random_state=42,
                         n_jobs=-1).fit(X_tv, y_tv)

# ============================================================
# 步骤3：测试集最终评估（只评估一次）
# ============================================================
X_test = test_z[FACTOR_COLS].values
rf_stats = daily_ic(rf, test_z, X_test)
xgb_stats = daily_ic(xgbm, test_z, X_test)

print("\n" + "="*60)
print("测试集样本外IC（2021-2025，合并重训后的最终模型）")
print("="*60)
result = pd.DataFrame({
    "IC均值": [rf_stats[0], xgb_stats[0]],
    "IC标准差": [rf_stats[1], xgb_stats[1]],
    "IR": [rf_stats[0]/rf_stats[1], xgb_stats[0]/xgb_stats[1]],
    "IC>0占比": [rf_stats[2], xgb_stats[2]],
}, index=["随机森林", "XGBoost"])
print(result.round(4).to_string())

# 对比单因子测试期IC
daily_ic_single = pd.read_pickle("daily_ic.pkl")
test_single = daily_ic_single.loc["2021-01-01":"2025-12-31"].mean()
print("\n--- 对比：单因子测试期IC均值 ---")
for f in FACTOR_COLS:
    print(f"  {f}: {test_single[f]:+.4f}")
best_single = test_single.abs().max()
print(f"\n最强单因子|IC|={best_single:.4f}  vs  模型(RF={rf_stats[0]:+.4f}, XGB={xgb_stats[0]:+.4f})")

# 保存
test_z["pred_rf"] = rf.predict(X_test)
test_z["pred_xgb"] = xgbm.predict(X_test)
test_z.to_pickle("test_predictions.pkl")
with open("ml_models.pkl", "wb") as f:
    pickle.dump({"rf": rf, "xgb": xgbm, "FACTOR_COLS": FACTOR_COLS,
                 "best_rf_params": best_rf_params, "best_xgb_params": best_xgb_params}, f)
print("\n✓ 已保存 test_predictions.pkl 和 ml_models.pkl")

In [ ]:
# ============================================================
# 【第5步·B】特征重要性 + 与单因子分析的一致性讨论
# 输出RF和XGBoost的特征重要性，和单因子测试期IC对比
# ============================================================
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读入训练好的模型
with open("ml_models.pkl", "rb") as f:
    models = pickle.load(f)
rf, xgbm = models["rf"], models["xgb"]
FACTOR_COLS = models["FACTOR_COLS"]

labels_cn = {"momentum_20d": "20日动量", "reversal_5d": "5日反转",
             "volatility_20d": "20日波动率", "volume_ratio": "成交量比"}

# ---- 提取特征重要性 ----
rf_imp = pd.Series(rf.feature_importances_, index=FACTOR_COLS)
xgb_imp = pd.Series(xgbm.feature_importances_, index=FACTOR_COLS)

# ---- 单因子测试期IC ----
daily_ic = pd.read_pickle("daily_ic.pkl")
single_ic = daily_ic.loc["2021-01-01":"2025-12-31"].mean()

# ---- 整合对比表 ----
compare = pd.DataFrame({
    "RF重要性": rf_imp,
    "XGB重要性": xgb_imp,
    "单因子IC均值": single_ic,
    "单因子|IC|": single_ic.abs(),
})
compare_display = compare.copy()
compare_display.index = [labels_cn[c] for c in compare.index]
compare_display = compare_display.sort_values("XGB重要性", ascending=False)

print("="*70)
print("特征重要性 vs 单因子有效性")
print("="*70)
print(compare_display.round(4).to_string())

# ---- 画特征重要性柱状图 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, imp, title in [(axes[0], rf_imp, "随机森林"), (axes[1], xgb_imp, "XGBoost")]:
    imp_sorted = imp.sort_values(ascending=True)
    names = [labels_cn[c] for c in imp_sorted.index]
    ax.barh(names, imp_sorted.values, color="steelblue")
    ax.set_title(f"{title} 特征重要性", fontsize=12)
    ax.set_xlabel("重要性")
    ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

# ---- 一致性分析 ----
print("\n" + "="*70)
print("一致性讨论")
print("="*70)
rf_rank = list(rf_imp.sort_values(ascending=False).index)
xgb_rank = list(xgb_imp.sort_values(ascending=False).index)
ic_rank = list(single_ic.abs().sort_values(ascending=False).index)

print(f"RF重要性排序:    {[labels_cn[c] for c in rf_rank]}")
print(f"XGB重要性排序:   {[labels_cn[c] for c in xgb_rank]}")
print(f"单因子|IC|排序:  {[labels_cn[c] for c in ic_rank]}")
print(f"\n最重要特征: RF={labels_cn[rf_rank[0]]}, XGB={labels_cn[xgb_rank[0]]}, "
      f"单因子最强={labels_cn[ic_rank[0]]}")

# ============================================================
# 【第6步】投资组合构建（RF + XGBoost + 波动率单因子，三个对比）
# 仅测试集；每5交易日调仓；选前20%等权；持仓期收益=选中股票实际未来5日收益均值
# 加入波动率单因子组合：验证"最强单因子直接选股"是否优于机器学习模型
# ============================================================
test_z = pd.read_pickle("test_predictions.pkl")

REBALANCE = 5
TOP_PCT = 0.2

all_days = test_z.index.get_level_values("Date").unique().sort_values()
rebal_days = all_days[::REBALANCE]
print(f"测试期交易日: {len(all_days)} 天，调仓 {len(rebal_days)} 次")

# ============================================================
# 构建组合：按打分选前20%
# ascending=False：选打分最高的（预测收益 / 正IC因子如波动率）
# ============================================================
def build_portfolio(test_z, score_col, ascending=False):
    period_returns = []
    for rd in rebal_days:
        day_data = test_z.xs(rd, level="Date")
        if len(day_data) < 5:
            continue
        n_top = max(1, int(len(day_data) * TOP_PCT))
        top = (day_data.nsmallest(n_top, score_col) if ascending
               else day_data.nlargest(n_top, score_col))
        period_ret = top["future_ret_5d"].mean()
        period_returns.append({"rebal_date": rd, "period_ret": period_ret, "n_stocks": n_top})
    return pd.DataFrame(period_returns).set_index("rebal_date")

def period_to_daily(pr, all_days, rebalance=5):
    daily_ret = pd.Series(0.0, index=all_days)
    rd_list = list(pr.index)
    for i, rd in enumerate(rd_list):
        period_ret = pr.loc[rd, "period_ret"]
        start_idx = all_days.get_loc(rd)
        end_idx = (all_days.get_loc(rd_list[i+1]) if i < len(rd_list)-1
                   else min(start_idx + rebalance, len(all_days)))
        n_days = end_idx - start_idx
        if n_days <= 0:
            continue
        daily_ret.iloc[start_idx:end_idx] = (1 + period_ret) ** (1/n_days) - 1
    return daily_ret

# ============================================================
# 三个组合
# 注意：波动率是正IC因子(高波动->高收益)，故 ascending=False 选波动最高的
#       模型预测的是收益，也选最高的
# ============================================================
configs = [
    ("RF",       "pred_rf",         False),
    ("XGB",      "pred_xgb",        False),
    ("波动率单因子", "volatility_20d",  False),   # 正IC：选波动率最高的前20%
]

portfolios = {}
print("\n" + "="*55)
print("三个组合表现对比")
print("="*55)
for name, col, asc in configs:
    pr = build_portfolio(test_z, col, asc)
    daily = period_to_daily(pr, all_days, REBALANCE)
    nav = (1 + daily).cumprod()
    portfolios[name] = {"period_ret": pr, "daily_ret": daily, "nav": nav}
    print(f"{name:12}: 选股{pr['n_stocks'].iloc[0]}只 | "
          f"持仓期均收益{pr['period_ret'].mean()*100:+.3f}% | "
          f"期末净值{nav.iloc[-1]:.4f}（总收益{nav.iloc[-1]-1:+.1%}）")

# 保存三个组合，第7步用
save = {}
for name in portfolios:
    save[f"{name}_daily_ret"] = portfolios[name]["daily_ret"]
    save[f"{name}_nav"] = portfolios[name]["nav"]
portfolio_df = pd.DataFrame(save)
portfolio_df.to_pickle("portfolio_returns.pkl")
print(f"\n✓ 已保存 portfolio_returns.pkl（三个组合）")

In [ ]:
# ============================================================
# 【第6步·改进】投资组合构建 —— 方案B：每天建仓、持有5天、重叠持仓
# 彻底解决几何平摊导致的波动率低估/夏普虚高问题：
#   每天用模型选前20%建一批，每批持有5天；任意一天同时持有5批（各1/5资金）；
#   组合当日收益 = 当天所有活跃批次的【真实当日收益】等权平均
# 三个组合：RF / XGBoost / 波动率单因子
# ============================================================
import numpy as np
import pandas as pd

test_z = pd.read_pickle("test_predictions.pkl")

# ---- 读入每只股票的真实每日收益（清洗阶段Cell6存的）----
daily_ret_wide = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)

# 只保留测试期的日收益
test_dates = test_z.index.get_level_values("Date").unique().sort_values()
daily_ret_wide = daily_ret_wide.reindex(test_dates)
print(f"测试期: {len(test_dates)} 天, 每日收益矩阵 {daily_ret_wide.shape}")

HOLD = 5          # 每批持有5天
TOP_PCT = 0.2     # 选前20%

# ============================================================
# 重叠持仓组合构建
# ============================================================
def build_overlap_portfolio(test_z, daily_ret_wide, score_col, hold=5, top_pct=0.2, ascending=False):
    all_days = daily_ret_wide.index

    # 1) 每天新建一批：选前20%
    daily_picks = {}
    for d in all_days:
        try:
            day_data = test_z.xs(d, level="Date")
        except KeyError:
            continue
        day_data = day_data.dropna(subset=[score_col])
        if len(day_data) < 5:
            continue
        n_top = max(1, int(len(day_data) * top_pct))
        picks = (day_data.nsmallest(n_top, score_col) if ascending
                 else day_data.nlargest(n_top, score_col)).index.tolist()
        daily_picks[d] = picks

    # 2) 组合每天收益 = 当天所有活跃批次(过去1~hold天建的)的真实当日收益平均
    portfolio_daily = pd.Series(0.0, index=all_days)
    for i, today in enumerate(all_days):
        active_returns = []
        for lag in range(1, hold + 1):          # 批次建于 today 的前 1~hold 天
            bi = i - lag
            if bi < 0:
                continue
            build_day = all_days[bi]
            if build_day not in daily_picks:
                continue
            picks = daily_picks[build_day]
            valid = [tk for tk in picks if tk in daily_ret_wide.columns]
            if valid:
                batch_ret = daily_ret_wide.loc[today, valid].mean()   # 该批今日真实收益
                if np.isfinite(batch_ret):
                    active_returns.append(batch_ret)
        if active_returns:
            portfolio_daily.iloc[i] = np.mean(active_returns)         # 等权平均各批
    return portfolio_daily

# 三个组合（波动率是正IC，选最高，ascending=False）
configs = [
    ("RF",       "pred_rf",        False),
    ("XGB",      "pred_xgb",       False),
    ("波动率单因子", "volatility_20d", False),
]

save = {}
print("\n" + "="*55)
print("三个组合（方案B：每日调仓+重叠持仓+真实日收益）")
print("="*55)
for name, col, asc in configs:
    daily = build_overlap_portfolio(test_z, daily_ret_wide, col, HOLD, TOP_PCT, asc)
    nav = (1 + daily).cumprod()
    save[f"{name}_daily_ret"] = daily
    save[f"{name}_nav"] = nav
    ann_vol = daily.std() * np.sqrt(252)
    print(f"{name:12}: 期末净值{nav.iloc[-1]:.4f}（总收益{nav.iloc[-1]-1:+.1%}）| "
          f"年化波动{ann_vol*100:.1f}%")

portfolio_df = pd.DataFrame(save)
portfolio_df.to_pickle("portfolio_returns.pkl")
print(f"\n✓ 已保存 portfolio_returns.pkl（真实日收益，无平摊）")
print("注意：年化波动率现在是真实的，夏普不会再虚高")

In [ ]:
# ============================================================
# 【第6步·基准】下载标普500指数(^GSPC)作为组合对比基准
# 用和股票数据相同的方式(urllib直连Yahoo chart API) + 重试机制
# ============================================================
import urllib.request
import datetime
import time

SPX_TICKER = "%5EGSPC"   # ^ 在URL里要转义成 %5E
# 稍往前留一点(2020-12)，保证2021-01有数据；覆盖到2026-01
t1 = int(datetime.datetime(2020, 8, 1).timestamp())
t2 = int(datetime.datetime(2026, 1, 1).timestamp())
url = (
    f"https://query1.finance.yahoo.com/v8/finance/chart/{SPX_TICKER}"
    f"?period1={t1}&period2={t2}&interval=1d&events=history"
)

# ---- 重试机制：应对偶发的网络/SSL波动 ----
MAX_RETRIES = 5
TIMEOUT = 60
data = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        resp = urllib.request.urlopen(req, timeout=TIMEOUT)
        data = json.loads(resp.read())
        print(f"第{attempt}次尝试成功")
        break
    except Exception as e:
        print(f"第{attempt}次尝试失败: {type(e).__name__}: {e}")
        if attempt < MAX_RETRIES:
            wait = attempt * 3
            print(f"  {wait}秒后重试...")
            time.sleep(wait)

if data is None:
    raise RuntimeError(
        f"重试{MAX_RETRIES}次仍失败，大概率是网络环境问题"
        f"（防火墙/VPN/代理干扰SSL握手），不是代码问题。建议：\n"
        f"  1. 检查是否连VPN/代理，尝试切换网络（如手机热点）\n"
        f"  2. 浏览器直接访问看能否打开: {url}"
    )

# ---- 解析 ----
result = data["chart"]["result"][0]
ts = pd.to_datetime(result["timestamp"], unit="s")
quote = result["indicators"]["quote"][0]
spx = pd.DataFrame({
    "Date": ts,
    "Open": quote["open"],
    "High": quote["high"],
    "Low": quote["low"],
    "Close": quote["close"],
    "Volume": quote["volume"],
})
# 指数一般没有独立复权价（不分红），adjclose不存在时用close，结果一样
if "adjclose" in result["indicators"]:
    spx["Adj Close"] = result["indicators"]["adjclose"][0]["adjclose"]
else:
    spx["Adj Close"] = quote["close"]
spx = spx.dropna(subset=["Close"]).set_index("Date").sort_index()
print(f"\n标普500指数: {spx.shape[0]} 天, {spx.index[0].date()} → {spx.index[-1].date()}")

# ---- 存收益率（第7步会读这个文件）----
spx_returns = spx["Adj Close"].pct_change().dropna()
spx_returns.name = "SPX_return"
spx_returns.to_csv("spx_returns.csv")
print("✓ 已保存 spx_returns.csv")

# 看一眼标普500这段自己的表现
spx_nav = (1 + spx_returns).cumprod()
print(f"标普500在此区间累计净值: {spx_nav.iloc[-1]:.3f} "
      f"（总收益 {spx_nav.iloc[-1]-1:+.1%}）")

In [ ]:
# ============================================================
# 【第7步·A】绩效指标计算 + 净值对比图
# 三个组合(RF/XGB/波动率单因子) vs 标普500基准
# 指标：年化收益(日均×252)、年化波动(日std×√252)、夏普(rf=0)、最大回撤
# ============================================================

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

portfolio_df = pd.read_pickle("portfolio_returns.pkl")
spx_returns = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True)["SPX_return"]

# ---- 日期对齐（组合与基准对齐到相同交易日）----
port_daily_cols = ["RF_daily_ret", "XGB_daily_ret", "波动率单因子_daily_ret"]
common_dates = portfolio_df.index.intersection(spx_returns.index)
print(f"组合天数={len(portfolio_df)}, 基准天数={len(spx_returns)}, 对齐后={len(common_dates)}")

spx_aligned = spx_returns.reindex(common_dates)

# ---- 绩效指标 ----
def performance_metrics(daily_ret):
    daily_ret = daily_ret.dropna()
    ann_ret = daily_ret.mean() * 252
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    nav = (1 + daily_ret).cumprod()
    running_max = nav.cummax()
    drawdown = (nav - running_max) / running_max
    max_dd = drawdown.min()
    return {"总收益": nav.iloc[-1]-1, "年化收益": ann_ret, "年化波动": ann_vol,
            "夏普": sharpe, "最大回撤": max_dd}

names_map = {"RF_daily_ret": "RF组合", "XGB_daily_ret": "XGB组合",
             "波动率单因子_daily_ret": "波动率单因子组合"}
rows = {}
for col in port_daily_cols:
    rows[names_map[col]] = performance_metrics(portfolio_df[col].reindex(common_dates))
rows["标普500"] = performance_metrics(spx_aligned)

perf = pd.DataFrame(rows).T
print("\n" + "="*70)
print("绩效指标对比（测试期 2021-2025）")
print("="*70)
perf_display = perf.copy()
for c in ["总收益", "年化收益", "年化波动", "最大回撤"]:
    perf_display[c] = (perf[c]*100).round(2).astype(str) + "%"
perf_display["夏普"] = perf["夏普"].round(2)
print(perf_display.to_string())

# ---- 净值对比图 ----
fig, ax = plt.subplots(figsize=(13, 6))
colors = {"RF组合": "tab:blue", "XGB组合": "tab:red",
          "波动率单因子组合": "tab:green", "标普500": "gray"}

for col in port_daily_cols:
    nav = (1 + portfolio_df[col].reindex(common_dates)).cumprod()
    ax.plot(nav.index, nav, label=names_map[col], linewidth=1.5, color=colors[names_map[col]])
spx_nav = (1 + spx_aligned).cumprod()
ax.plot(spx_nav.index, spx_nav, label="标普500", linewidth=1.8, color="gray", ls="--")

ax.axhline(1, color='black', lw=0.6, alpha=0.5)
ax.set_title("组合与标普500累计净值对比（2021-2025）", fontsize=13)
ax.set_xlabel("日期"); ax.set_ylabel("累计净值")
ax.legend(loc="best", fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 保存绩效表和对齐后的收益，第7步后续用
perf.to_pickle("performance_metrics.pkl")
aligned = portfolio_df[port_daily_cols].reindex(common_dates).copy()
aligned["SPX"] = spx_aligned
aligned.to_pickle("aligned_returns.pkl")
print("\n✓ 已保存 performance_metrics.pkl 和 aligned_returns.pkl")

In [ ]:
# ============================================================
# 【第7步·B】回撤曲线(水下曲线) + 年度收益柱状图
# 回撤曲线：直观展示各组合的下跌风险（暴露组合回撤vs标普的真实对比）
# 年度收益：逐年对比组合与标普500
# ============================================================

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

aligned = pd.read_pickle("aligned_returns.pkl")
# aligned 的列：RF_daily_ret, XGB_daily_ret, 波动率单因子_daily_ret, SPX

names_map = {"RF_daily_ret": "RF组合", "XGB_daily_ret": "XGB组合",
             "波动率单因子_daily_ret": "波动率单因子组合", "SPX": "标普500"}
colors = {"RF组合": "tab:blue", "XGB组合": "tab:red",
          "波动率单因子组合": "tab:green", "标普500": "gray"}

# ============================================================
# 1. 回撤曲线（水下曲线）
# ============================================================
def drawdown_series(daily_ret):
    nav = (1 + daily_ret).cumprod()
    running_max = nav.cummax()
    return (nav - running_max) / running_max

fig, ax = plt.subplots(figsize=(13, 6))
for col in aligned.columns:
    dd = drawdown_series(aligned[col])
    name = names_map[col]
    ax.plot(dd.index, dd * 100, label=f"{name} (最大回撤{dd.min()*100:.1f}%)",
            linewidth=1.3, color=colors[name])
    ax.fill_between(dd.index, dd * 100, 0, alpha=0.1, color=colors[name])

ax.axhline(0, color='black', lw=0.6)
ax.set_title("回撤曲线（水下曲线）2021-2025", fontsize=13)
ax.set_xlabel("日期"); ax.set_ylabel("回撤 (%)")
ax.legend(loc="lower left", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 2. 年度收益柱状图
# ============================================================
def annual_returns(daily_ret):
    return (1 + daily_ret).groupby(daily_ret.index.year).apply(lambda x: x.prod() - 1)

annual_df = pd.DataFrame({names_map[col]: annual_returns(aligned[col]) for col in aligned.columns})

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(annual_df.index))
width = 0.2
for i, name in enumerate(annual_df.columns):
    ax.bar(x + i*width, annual_df[name]*100, width, label=name, color=colors[name])

ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(annual_df.index)
ax.set_title("各年度收益对比", fontsize=13)
ax.set_xlabel("年份"); ax.set_ylabel("年度收益 (%)")
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# 打印年度收益表
print("="*70)
print("各年度收益 (%)")
print("="*70)
print((annual_df*100).round(2).to_string())

annual_df.to_pickle("annual_returns.pkl")
print("\n✓ 已保存 annual_returns.pkl")

In [ ]:
# ============================================================
# 【进阶·批量探测】测试全部股票池的基本面数据覆盖面
# 逐只拉取，统计：成功率、时间覆盖、关键字段完整性
# 用"年报"探测（季报数据量更大，先用年报摸底覆盖面）
# 注：本地跑，需联网；100只可能要几分钟
# ============================================================
import akshare as ak
import time

# 读你的股票池
tickers = pd.read_csv("valid_tickers.csv")["Ticker"].tolist()
print(f"股票池: {len(tickers)} 只，开始批量探测...")

KEY_FIELDS = ["NOTICE_DATE", "REPORT_DATE", "ROE_AVG", "ROA", "DEBT_ASSET_RATIO"]

success, fail = [], []
coverage = []

for i, tk in enumerate(tickers):
    try:
        df = ak.stock_financial_us_analysis_indicator_em(symbol=tk, indicator="年报")
        # 检查关键字段
        missing = [f for f in KEY_FIELDS if f not in df.columns]
        if missing:
            fail.append((tk, f"缺字段{missing}"))
            print(f"  [{i+1:>3}/{len(tickers)}] △ {tk:<6} 缺字段{missing}")
            continue
        if len(df) == 0:
            fail.append((tk, "空数据"))
            print(f"  [{i+1:>3}/{len(tickers)}] △ {tk:<6} 空数据")
            continue
        nd = pd.to_datetime(df["NOTICE_DATE"])
        coverage.append({
            "ticker": tk, "期数": len(df),
            "最早公告": nd.min().date(), "最晚公告": nd.max().date(),
        })
        success.append(tk)
        print(f"  [{i+1:>3}/{len(tickers)}] ✓ {tk:<6} {len(df)}期, {nd.min().date()}→{nd.max().date()}")
    except Exception as e:
        fail.append((tk, str(e)[:40]))
        print(f"  [{i+1:>3}/{len(tickers)}] ✗ {tk:<6} {str(e)[:40]}")
    time.sleep(0.5)   # 礼貌间隔，避免被封

# ============================================================
# 汇总
# ============================================================
print("\n" + "="*60)
print("批量探测汇总")
print("="*60)
print(f"成功: {len(success)}/{len(tickers)} 只")
print(f"失败/异常: {len(fail)} 只")

if coverage:
    cov_df = pd.DataFrame(coverage)
    print(f"\n--- 时间覆盖 ---")
    print(f"期数中位数: {cov_df['期数'].median():.0f} 期")
    print(f"最早公告日的范围: {cov_df['最早公告'].min()} ~ {cov_df['最早公告'].max()}")
    print(f"最晚公告日的范围: {cov_df['最晚公告'].min()} ~ {cov_df['最晚公告'].max()}")
    # 有多少只覆盖到2021年前(测试期需要)
    early_enough = (pd.to_datetime(cov_df['最早公告']) <= pd.Timestamp("2021-01-01")).sum()
    print(f"最早公告≤2021年的股票: {early_enough}/{len(cov_df)} 只（测试期回测需要）")

if fail:
    print(f"\n--- 失败列表 ---")
    for tk, reason in fail:
        print(f"  {tk}: {reason}")

# 保存成功列表和覆盖信息
if coverage:
    cov_df.to_csv("fundamental_coverage.csv", index=False)
    pd.Series(success, name="Ticker").to_csv("fundamental_valid_tickers.csv", index=False)
    print(f"\n✓ 已保存 fundamental_coverage.csv 和 fundamental_valid_tickers.csv")

In [ ]:
# ============================================================
# 【进阶·基本面·拉取】批量拉取84只非金融股的季报数据
# 接口: stock_financial_us_analysis_indicator_em (累计季报)
# 取字段: 公告日(NOTICE_DATE)、报告期(REPORT_DATE)、
#         ROE(ROE_AVG)、ROA、资产负债率(DEBT_ASSET_RATIO)、EPS(算PE用)
# 按任务清单要求用"季度更新"的基本面数据
# 注: 本地跑,需联网; 84只季报可能要5-10分钟
# ============================================================
import akshare as ak
import pandas as pd
import numpy as np
import time

# 读入84只非金融股(上一步探测保存的)
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
print(f"非金融股票池: {len(tickers)} 只，开始拉取季报...")

NEED_COLS = ["REPORT_DATE", "NOTICE_DATE", "ROE_AVG", "ROA", "DEBT_ASSET_RATIO", "BASIC_EPS"]

all_fund = {}
fetch_fail = []
for i, tk in enumerate(tickers):
    try:
        df = ak.stock_financial_us_analysis_indicator_em(symbol=tk, indicator="累计季报")
        if df is None or len(df) == 0:
            fetch_fail.append(tk)
            print(f"  [{i+1:>3}/{len(tickers)}] △ {tk:<6} 空")
            continue
        # 检查字段
        missing = [c for c in NEED_COLS if c not in df.columns]
        if missing:
            fetch_fail.append(tk)
            print(f"  [{i+1:>3}/{len(tickers)}] △ {tk:<6} 缺{missing}")
            continue
        d = df[NEED_COLS].copy()
        d["REPORT_DATE"] = pd.to_datetime(d["REPORT_DATE"])
        d["NOTICE_DATE"] = pd.to_datetime(d["NOTICE_DATE"])
        d["Ticker"] = tk
        all_fund[tk] = d
        print(f"  [{i+1:>3}/{len(tickers)}] ✓ {tk:<6} {len(d)}期季报, "
              f"{d['NOTICE_DATE'].min().date()}→{d['NOTICE_DATE'].max().date()}")
    except Exception as e:
        fetch_fail.append(tk)
        print(f"  [{i+1:>3}/{len(tickers)}] ✗ {tk:<6} {str(e)[:35]}")
    time.sleep(0.5)

# 合并保存
combined = pd.concat(all_fund.values(), ignore_index=True)
combined.to_pickle("fundamental_raw.pkl")
print(f"\n{'='*55}")
print(f"成功: {len(all_fund)}/{len(tickers)} 只 | 合并 {len(combined)} 行季报记录")
if fetch_fail:
    print(f"失败: {fetch_fail}")
print(f"✓ 已保存 fundamental_raw.pkl")

In [ ]:
# ============================================================
# 【进阶·基本面·对齐】公告日对齐 + 算PE + 整合成基本面因子表
# 1. 累计季报EPS年化(Q1×4, H1×2, 9M×4/3, FY×1)
# 2. 按公告日(NOTICE_DATE)前向填充到日频 —— 严格防前视
# 3. PE = 股价 / 年化EPS
# 4. 整合成日频因子表: ROE, ROA, 资产负债率, PE
# ============================================================
combined = pd.read_pickle("fundamental_raw.pkl")

# 股价(用清洗后的Adj Close),只保留84只非金融股
adj_close = pd.read_csv("adj_close_clean.csv", index_col=0, parse_dates=True)
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers = [t for t in tickers if t in adj_close.columns]
adj_close = adj_close[tickers]
trade_days = adj_close.index
print(f"股票池 {len(tickers)} 只, 交易日 {len(trade_days)} 天")

# ============================================================
# 1. 累计季报EPS年化
# ============================================================
def annualize_eps(row):
    month = row["REPORT_DATE"].month
    q_map = {3: 1, 6: 2, 9: 3, 12: 4}      # 3月Q1(1季),6月H1(2季),9月(3季),12月全年(4季)
    n_q = q_map.get(month, 4)
    if pd.isna(row["BASIC_EPS"]):
        return np.nan
    return row["BASIC_EPS"] * (4 / n_q)

combined["EPS_annual"] = combined.apply(annualize_eps, axis=1)

# ============================================================
# 2+3. 逐股票:公告日对齐到日频 + 算PE
# ============================================================
VALUE_COLS = ["ROE_AVG", "ROA", "DEBT_ASSET_RATIO", "EPS_annual"]

fund_panels = []
for tk in tickers:
    sub = combined[combined["Ticker"] == tk].sort_values("NOTICE_DATE")
    if len(sub) == 0:
        continue
    # 按公告日索引,前向填充到交易日(只用已公告的数据,防前视)
    f = sub.set_index("NOTICE_DATE")[VALUE_COLS]
    f = f[~f.index.duplicated(keep="last")]      # 同一公告日去重
    aligned = f.reindex(f.index.union(trade_days)).ffill().reindex(trade_days)

    # 算PE = 股价 / 年化EPS (EPS<=0时PE无意义,设NaN)
    eps = aligned["EPS_annual"]
    pe = adj_close[tk] / eps.where(eps > 0)

    panel = pd.DataFrame({
        "roe": aligned["ROE_AVG"],
        "roa": aligned["ROA"],
        "debt_ratio": aligned["DEBT_ASSET_RATIO"],
        "pe": pe,
    })
    panel["Ticker"] = tk
    panel.index.name = "Date"
    fund_panels.append(panel)

# 合并成长表
fund_factor = pd.concat(fund_panels).reset_index().set_index(["Date", "Ticker"]).sort_index()
print(f"\n基本面因子表: {fund_factor.shape[0]} 行 × {fund_factor.shape[1]} 列")
print(f"列: {fund_factor.columns.tolist()}")

# 覆盖度检查(每年有多少只股票有ROE)
print("\n--- 每年有基本面数据的股票数(检查训练期够不够) ---")
tmp = fund_factor.dropna(subset=["roe"]).reset_index()
tmp["year"] = tmp["Date"].dt.year
yearly_count = tmp.groupby("year")["Ticker"].nunique()
print(yearly_count.to_string())

# 看样例
print("\n--- 苹果基本面因子样例(近期) ---")
print(fund_factor.xs("AAPL", level="Ticker").dropna().tail(3).round(2).to_string())

fund_factor.to_pickle("fundamental_factor.pkl")
print("\n✓ 已保存 fundamental_factor.pkl")

In [ ]:
fund_factor

In [ ]:
# ============================================================
# 【进阶·技术面·TA-Lib版】RSI/MACD柱/布林带宽/ATR + ht_phase(希尔伯特相位)
# ============================================================
import numpy as np
import pandas as pd
import talib

stock_data = pd.read_csv("stock_data_clean_adjusted.csv", header=[0,1], index_col=0, parse_dates=True)
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
close_df = stock_data.xs("Close", axis=1, level=1)
high_df  = stock_data.xs("High", axis=1, level=1)
low_df   = stock_data.xs("Low", axis=1, level=1)
tickers84 = [t for t in tickers84 if t in close_df.columns]
close_df = close_df[tickers84]; high_df = high_df[tickers84]; low_df = low_df[tickers84]

print(f"计算TA-Lib技术因子(含ht_phase): {len(tickers84)}只股票")
rsi_dict, macd_dict, boll_dict, atr_dict, htphase_dict = {}, {}, {}, {}, {}

for tk in tickers84:
    close = close_df[tk].values.astype(float)
    high  = high_df[tk].values.astype(float)
    low   = low_df[tk].values.astype(float)
    rsi_dict[tk] = talib.RSI(close, timeperiod=14)
    _, _, hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    macd_dict[tk] = hist
    upper, middle, lower = talib.BBANDS(close, timeperiod=20, nbdevup=2, nbdevdn=2)
    boll_dict[tk] = (upper - lower) / middle * 100
    atr_dict[tk] = talib.ATR(high, low, close, timeperiod=14)
    htphase_dict[tk] = talib.HT_DCPHASE(close)   # ★希尔伯特主导周期相位

rsi_df  = pd.DataFrame(rsi_dict, index=close_df.index)
macd_df = pd.DataFrame(macd_dict, index=close_df.index)
boll_df = pd.DataFrame(boll_dict, index=close_df.index)
atr_df  = pd.DataFrame(atr_dict, index=close_df.index)
htphase_df = pd.DataFrame(htphase_dict, index=close_df.index)   # ★

def to_long(df, name):
    s = df.stack().rename(name); s.index.names = ["Date", "Ticker"]
    return s

technical_factor = pd.concat([
    to_long(rsi_df, "rsi"),
    to_long(macd_df, "macd_hist"),
    to_long(boll_df, "boll_width"),
    to_long(atr_df, "atr"),
    to_long(htphase_df, "ht_phase"),   # ★新增
], axis=1)

technical_factor.to_pickle("technical_factor.pkl")
print(f"\nTA-Lib技术因子: {technical_factor.shape}")
print(f"缺失率: RSI {technical_factor['rsi'].isna().mean()*100:.1f}%, "
      f"MACD {technical_factor['macd_hist'].isna().mean()*100:.1f}%, "
      f"布林 {technical_factor['boll_width'].isna().mean()*100:.1f}%, "
      f"ATR {technical_factor['atr'].isna().mean()*100:.1f}%, "
      f"ht_phase {technical_factor['ht_phase'].isna().mean()*100:.1f}%")
print("\n✓ 已保存 technical_factor.pkl (含ht_phase)")

In [ ]:
tech_factor

In [ ]:
# ============================================================
# 【进阶·特征库整合】把12个因子整合成统一特征库
# 量价(4) + 技术面(4) + 基本面(4) + 未来5日收益(目标)
# 统一到84只非金融股; 未来收益来自第一阶段的factor_table
# ============================================================
import numpy as np
import pandas as pd

# 读三个因子表
price_factor = pd.read_pickle("factor_table.pkl")      # 第一阶段:4量价因子+future_ret_5d(99只)
tech_factor  = pd.read_pickle("technical_factor.pkl")  # 技术面4因子(84只)
fund_factor  = pd.read_pickle("fundamental_factor.pkl") # 基本面4因子(84只)

# 84只非金融股
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()

# ---- 把量价因子也筛到84只 ----
price_84 = price_factor[price_factor.index.get_level_values("Ticker").isin(tickers)]
print(f"量价因子(84只): {price_84.shape}")
print(f"技术因子: {tech_factor.shape}")
print(f"基本面因子: {fund_factor.shape}")

# ============================================================
# 三表按(Date,Ticker)索引整合
# 以量价因子(含future_ret_5d目标)为主，join技术面和基本面
# ============================================================
feature_lib = price_84.join(tech_factor, how="left").join(fund_factor, how="left")

# 12个因子列
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",  # 量价
               "rsi", "macd_hist", "boll_width", "atr",                          # 技术面
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]                                 # 基本面

print(f"\n特征库: {feature_lib.shape[0]} 行 × {feature_lib.shape[1]} 列")
print(f"12个因子: {FACTOR_COLS}")

# ---- NaN情况 ----
print(f"\n各因子NaN比例:")
for c in FACTOR_COLS + ["future_ret_5d"]:
    pct = feature_lib[c].isna().mean() * 100
    print(f"  {c:16}: {pct:.1f}%")

# ---- 保留:有未来收益 + 12因子都不缺 ----
# (基本面因子早期可能缺,dropna会自动去掉那些行,和第一阶段处理IPO一样)
feature_clean = feature_lib.dropna(subset=["future_ret_5d"] + FACTOR_COLS)
print(f"\n完整样本(12因子+目标都不缺): {feature_clean.shape[0]} 行")
print(f"日期范围: {feature_clean.index.get_level_values('Date').min().date()} "
      f"→ {feature_clean.index.get_level_values('Date').max().date()}")
print(f"股票数: {feature_clean.index.get_level_values('Ticker').nunique()} 只")

# 每年样本数(看训练期够不够)
tmp = feature_clean.reset_index()
tmp["year"] = tmp["Date"].dt.year
print(f"\n各年完整样本数:")
print(tmp.groupby("year").size().to_string())

# 保存
feature_lib.to_pickle("feature_library.pkl")         # 完整版(含NaN)
feature_clean.to_pickle("feature_library_clean.pkl") # 清洗版(12因子都全)
print(f"\n✓ 已保存 feature_library.pkl 和 feature_library_clean.pkl")

In [ ]:
feature_clean

In [ ]:
# ============================================================
# 【进阶·预处理】去极值(winsorize) + 标准化(z-score)
# 正交化前的必要预处理:
#   ① 去极值: MAD法,每天横截面缩尾极端值(防极端值扭曲正交化)
#   ② 标准化: 每天横截面z-score,统一量纲(正交化才公平)
# 对12个因子做,不动future_ret_5d(目标)
# ============================================================

# 读清洗后的特征库(12因子都全的)
feature = pd.read_pickle("feature_library_clean.pkl")

FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",  # 量价
               "rsi", "macd_hist", "boll_width", "atr",                          # 技术面
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]                                 # 基本面

# ============================================================
# ① 去极值：MAD法，每天横截面缩尾
#    偏离中位数超过 n_mad×1.4826×MAD 的值，缩尾到边界
#    (1.4826使MAD在正态下≈标准差，n_mad=5相当于5倍标准差)
# ============================================================
def winsorize_mad(df, cols, n_mad=5):
    df = df.copy()
    dates_idx = df.index.get_level_values("Date")
    for col in cols:
        def _winsor(x):
            med = x.median()
            mad = (x - med).abs().median()
            if mad == 0 or np.isnan(mad):
                return x
            upper = med + n_mad * 1.4826 * mad
            lower = med - n_mad * 1.4826 * mad
            return x.clip(lower, upper)
        df[col] = df.groupby(dates_idx)[col].transform(_winsor)
    return df

# ============================================================
# ② 标准化：每天横截面 z-score
# ============================================================
def zscore_cross(df, cols):
    df = df.copy()
    dates_idx = df.index.get_level_values("Date")
    for col in cols:
        grp = df.groupby(dates_idx)[col]
        df[col] = (df[col] - grp.transform("mean")) / grp.transform("std")
    return df

print("预处理前，各因子数值范围（部分）:")
for col in ["momentum_20d", "pe", "roe", "atr"]:
    print(f"  {col:14}: [{feature[col].min():.2f}, {feature[col].max():.2f}]")

# 先去极值
feat_wins = winsorize_mad(feature, FACTOR_COLS, n_mad=5)
print("\n去极值后，各因子数值范围（部分）:")
for col in ["momentum_20d", "pe", "roe", "atr"]:
    print(f"  {col:14}: [{feat_wins[col].min():.2f}, {feat_wins[col].max():.2f}]")

# 再标准化
feat_std = zscore_cross(feat_wins, FACTOR_COLS)

# 保留目标列
feat_std["future_ret_5d"] = feature["future_ret_5d"]

# 标准化后可能因某天某因子std=0产生NaN,去掉
feat_std_clean = feat_std.dropna(subset=FACTOR_COLS)

print(f"\n预处理后特征库: {feat_std_clean.shape[0]} 行 × {feat_std_clean.shape[1]} 列")

# 验证:随机一天,各因子应均值≈0标准差≈1
sample_date = feat_std_clean.index.get_level_values("Date").unique()[100]
day_data = feat_std_clean.xs(sample_date, level="Date")
print(f"\n验证标准化(某日横截面, 应均值≈0标准差≈1):")
print(day_data[FACTOR_COLS].agg(["mean", "std"]).round(3).T.to_string())

feat_std_clean.to_pickle("feature_standardized.pkl")
print(f"\n✓ 已保存 feature_standardized.pkl（去极值+标准化后的12因子）")

In [ ]:
# ============================================================
# 【进阶·正交化·第1步】查看12个因子的相关矩阵
# 确认因子间的共线性(哪些因子高度相关)，为正交化做准备
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

feat = pd.read_pickle("feature_standardized.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
               "rsi", "macd_hist", "boll_width", "atr",
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]

# ---- 相关矩阵(Spearman秩相关,更稳健) ----
corr = feat[FACTOR_COLS].corr(method="spearman")

print("="*80)
print("12因子相关矩阵 (Spearman)")
print("="*80)
print(corr.round(2).to_string())

# ---- 找出高相关对 ----
print("\n" + "="*50)
print("高相关因子对 (|corr| > 0.5)")
print("="*50)
high_corr = []
for i in range(len(FACTOR_COLS)):
    for j in range(i+1, len(FACTOR_COLS)):
        c = corr.iloc[i, j]
        if abs(c) > 0.5:
            high_corr.append((FACTOR_COLS[i], FACTOR_COLS[j], c))
if high_corr:
    for f1, f2, c in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {f1:16} <-> {f2:16}: {c:+.2f}")
else:
    print("  无强相关对")

# ---- 热力图 ----
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(FACTOR_COLS)))
ax.set_yticks(range(len(FACTOR_COLS)))
ax.set_xticklabels(FACTOR_COLS, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(FACTOR_COLS, fontsize=9)
# 标注数值
for i in range(len(FACTOR_COLS)):
    for j in range(len(FACTOR_COLS)):
        val = corr.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if abs(val) > 0.5 else "black", fontsize=7)
plt.colorbar(im, label="相关系数")
ax.set_title("12因子相关矩阵热力图", fontsize=13)
plt.tight_layout()
plt.show()

corr.to_pickle("factor_corr.pkl")
print("\n✓ 已保存 factor_corr.pkl")

In [ ]:
# ============================================================
# 【进阶·正交化·第2步】对称正交化(Symmetric Orthogonalization)
# 每天横截面上,对12因子做对称正交化: F_orth = F · M^(-1/2), M=FᵀF
# 特点: 所有因子平等对待(不依赖顺序),去相关后各因子仍保留经济含义
# ============================================================
import numpy as np
import pandas as pd

feat = pd.read_pickle("feature_standardized.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
               "rsi", "macd_hist", "boll_width", "atr",
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]

# ============================================================
# 对称正交化(单个横截面)
# ============================================================
def symmetric_orthogonalize(F):
    """F: (n_stocks, n_factors) → 正交化后的矩阵"""
    M = F.T @ F                                  # 重叠矩阵
    eigvals, eigvecs = np.linalg.eigh(M)
    eigvals = np.maximum(eigvals, 1e-10)         # 防数值问题
    M_inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals)) @ eigvecs.T
    return F @ M_inv_sqrt

# ============================================================
# 逐日横截面正交化
# ============================================================
dates_idx = feat.index.get_level_values("Date")
orth_records = []

for date, group in feat.groupby(dates_idx):
    F = group[FACTOR_COLS].values
    # 需要股票数 > 因子数,否则正交化不稳定
    if F.shape[0] <= len(FACTOR_COLS):
        continue
    # 该日再标准化一次(确保每列均值0,正交化前提)
    F = (F - F.mean(axis=0)) / (F.std(axis=0) + 1e-12)
    F_orth = symmetric_orthogonalize(F)
    # 正交化后再标准化(方便后续用)
    F_orth = (F_orth - F_orth.mean(axis=0)) / (F_orth.std(axis=0) + 1e-12)

    df_o = pd.DataFrame(F_orth, columns=[f"{c}_orth" for c in FACTOR_COLS],
                        index=group.index)
    orth_records.append(df_o)

feat_orth = pd.concat(orth_records).sort_index()
# 带上未来收益
feat_orth["future_ret_5d"] = feat["future_ret_5d"]
feat_orth = feat_orth.dropna(subset=[f"{c}_orth" for c in FACTOR_COLS])

print(f"正交化后特征库: {feat_orth.shape[0]} 行 × {feat_orth.shape[1]} 列")
print(f"正交因子列: {[c for c in feat_orth.columns if c.endswith('_orth')]}")

# ============================================================
# 验证:正交化后因子间相关性应≈0
# ============================================================
ORTH_COLS = [f"{c}_orth" for c in FACTOR_COLS]
corr_after = feat_orth[ORTH_COLS].corr(method="pearson")
off_diag = corr_after.values[~np.eye(len(ORTH_COLS), dtype=bool)]
print(f"\n正交化后因子间相关(非对角):")
print(f"  最大绝对相关: {np.abs(off_diag).max():.4f} (应接近0)")
print(f"  平均绝对相关: {np.abs(off_diag).mean():.4f}")

# 对比正交化前(读之前的相关矩阵)
corr_before = pd.read_pickle("factor_corr.pkl")
off_before = corr_before.values[~np.eye(len(FACTOR_COLS), dtype=bool)]
print(f"\n对比 — 正交化前平均绝对相关: {np.abs(off_before).mean():.4f}")
print(f"      正交化后平均绝对相关: {np.abs(off_diag).mean():.4f}")

feat_orth.to_pickle("feature_orthogonalized.pkl")
print(f"\n✓ 已保存 feature_orthogonalized.pkl")

In [ ]:
# ============================================================
# 【进阶·正交化·第3步】正交化效果评估
# 对比正交化前 vs 后:各因子IC均值、IR、累计IC曲线(全12个)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
import pickle

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

feat_before = pd.read_pickle("feature_standardized.pkl")
feat_after  = pd.read_pickle("feature_orthogonalized.pkl")

FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
               "rsi", "macd_hist", "boll_width", "atr",
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]
ORTH_COLS = [f"{c}_orth" for c in FACTOR_COLS]

labels_cn = {"momentum_20d":"20日动量", "reversal_5d":"5日反转", "volatility_20d":"20日波动率",
             "volume_ratio":"成交量比", "rsi":"RSI", "macd_hist":"MACD柱",
             "boll_width":"布林带宽", "atr":"ATR", "roe":"ROE", "roa":"ROA",
             "debt_ratio":"资产负债率", "pe":"市盈率", "ht_phase":"希尔伯特相位"}

def compute_daily_ic(feat, factor_cols):
    dates_idx = feat.index.get_level_values("Date")
    ic_dict = {c: {} for c in factor_cols}
    for date, g in feat.groupby(dates_idx):
        r = g["future_ret_5d"].values
        for col in factor_cols:
            f = g[col].values
            mask = np.isfinite(f) & np.isfinite(r)
            if mask.sum() >= 5 and np.std(f[mask]) > 1e-12:
                ic, _ = spearmanr(f[mask], r[mask])
                if not np.isnan(ic):
                    ic_dict[col][date] = ic
    return pd.DataFrame({c: pd.Series(ic_dict[c]) for c in factor_cols}).sort_index()

ic_before = compute_daily_ic(feat_before, FACTOR_COLS)
ic_after  = compute_daily_ic(feat_after, ORTH_COLS)

# ---- IC统计对比 ----
def ic_stats(daily_ic):
    return pd.DataFrame({
        "IC均值": daily_ic.mean(),
        "IR": daily_ic.mean() / daily_ic.std(),
        "IC>0占比": (daily_ic > 0).mean(),
    })

stats_before = ic_stats(ic_before)
stats_after = ic_stats(ic_after)
stats_after.index = FACTOR_COLS

compare = pd.DataFrame({
    "IC均值_前": stats_before["IC均值"],
    "IC均值_后": stats_after["IC均值"],
    "IR_前": stats_before["IR"],
    "IR_后": stats_after["IR"],
})
compare.index = [labels_cn[c] for c in compare.index]
print("="*75)
print("正交化前后 IC/IR 对比（全12因子）")
print("="*75)
print(compare.round(4).to_string())

# ---- 全12个因子的累计IC曲线对比 (4行×3列) ----
cum_before = ic_before.cumsum()
cum_after = ic_after.cumsum()
cum_after.columns = FACTOR_COLS

fig, axes = plt.subplots(5, 3, figsize=(16, 16))
for ax, f in zip(axes.flatten(), FACTOR_COLS):
    ax.plot(cum_before.index, cum_before[f], label="正交化前", linewidth=1.3, color="tab:blue")
    ax.plot(cum_after.index, cum_after[f], label="正交化后", linewidth=1.3, ls="--", color="tab:red")
    ax.axhline(0, color="gray", lw=0.6)
    ax.set_title(f"{labels_cn[f]} 累计IC", fontsize=11)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

with open("ortho_ic_compare.pkl", "wb") as f:
    pickle.dump({"before": stats_before, "after": stats_after, "compare": compare}, f)
print("\n✓ 已保存 ortho_ic_compare.pkl")

In [ ]:
# ============================================================
# 【进阶·机器学习】用12个正交因子训练 RF/XGBoost (带进度显示)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from itertools import product
import pickle
import time

feat = pd.read_pickle("feature_orthogonalized.pkl")
ORTH_COLS = [f"{c}_orth" for c in
             ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
              "rsi", "macd_hist", "boll_width", "atr",
              "roe", "roa", "debt_ratio", "pe", "ht_phase"]]

EMBARGO = 5
date_level = feat.index.get_level_values("Date")
def slice_period(start, end):
    m = (date_level >= pd.Timestamp(start)) & (date_level <= pd.Timestamp(end))
    return feat[m]
def drop_tail(df, k):
    if len(df) == 0: return df
    uniq = df.index.get_level_values("Date").unique().sort_values()
    if len(uniq) <= k: return df
    return df[df.index.get_level_values("Date") <= uniq[-(k+1)]]

train = drop_tail(slice_period("2010-01-01", "2018-12-31"), EMBARGO)
val   = drop_tail(slice_period("2019-01-01", "2020-12-31"), EMBARGO)
test  = slice_period("2021-01-01", "2025-12-31")
print(f"划分: train={len(train)}, val={len(val)}, test={len(test)}")

X_train, y_train = train[ORTH_COLS].values, train["future_ret_5d"].values
X_val = val[ORTH_COLS].values

def daily_ic(model, data_z, X):
    tmp = data_z.copy()
    tmp["pred"] = model.predict(X)
    dates_idx = tmp.index.get_level_values("Date")
    ics = []
    for date, g in tmp.groupby(dates_idx):
        if len(g) >= 5 and g["pred"].std() > 1e-12:
            ic, _ = spearmanr(g["pred"], g["future_ret_5d"])
            if not np.isnan(ic):
                ics.append(ic)
    ics = np.array(ics)
    return ics.mean(), ics.std(), (ics > 0).mean()

# ============================================================
# 随机森林网格(带进度)
# ============================================================
rf_grid = list(product([3, 4, 5], [20, 50, 100], [100, 200, 300]))
print(f"\n随机森林网格调参 (共{len(rf_grid)}个组合)...")
best_rf, best_rf_ic = None, -np.inf
t0 = time.time()
for i, (depth, msl, ne) in enumerate(rf_grid):
    m = RandomForestRegressor(n_estimators=ne, max_depth=depth,
                               min_samples_leaf=msl, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    v = daily_ic(m, val, X_val)[0]
    if v > best_rf_ic:
        best_rf_ic, best_rf = v, (depth, msl, ne)
    elapsed = time.time() - t0
    eta = elapsed / (i+1) * (len(rf_grid) - i - 1)
    print(f"  [{i+1:>2}/{len(rf_grid)}] depth={depth},msl={msl},ne={ne} | "
          f"val IC={v:+.4f} | 用时{elapsed:.0f}s ETA{eta:.0f}s", flush=True)
print(f"  最优RF: {best_rf} (val IC={best_rf_ic:+.4f})")

# ============================================================
# XGBoost网格(带进度)
# ============================================================
xgb_grid = list(product([3, 4, 5], [0.01, 0.03, 0.05, 0.1], [100, 300]))
print(f"\nXGBoost网格调参 (共{len(xgb_grid)}个组合)...")
best_xgb, best_xgb_ic = None, -np.inf
t0 = time.time()
for i, (depth, lr, ne) in enumerate(xgb_grid):
    m = xgb.XGBRegressor(n_estimators=ne, max_depth=depth, learning_rate=lr,
                          random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    v = daily_ic(m, val, X_val)[0]
    if v > best_xgb_ic:
        best_xgb_ic, best_xgb = v, (depth, lr, ne)
    elapsed = time.time() - t0
    eta = elapsed / (i+1) * (len(xgb_grid) - i - 1)
    print(f"  [{i+1:>2}/{len(xgb_grid)}] depth={depth},lr={lr},ne={ne} | "
          f"val IC={v:+.4f} | 用时{elapsed:.0f}s ETA{eta:.0f}s", flush=True)
print(f"  最优XGB: {best_xgb} (val IC={best_xgb_ic:+.4f})")

# ============================================================
# 合并重训 → 测试集
# ============================================================
print("\n合并train+val重训最终模型...")
trainval = pd.concat([train, val])
X_tv, y_tv = trainval[ORTH_COLS].values, trainval["future_ret_5d"].values
rf = RandomForestRegressor(n_estimators=best_rf[2], max_depth=best_rf[0],
                            min_samples_leaf=best_rf[1], random_state=42, n_jobs=-1).fit(X_tv, y_tv)
xgbm = xgb.XGBRegressor(n_estimators=best_xgb[2], max_depth=best_xgb[0],
                         learning_rate=best_xgb[1], random_state=42, n_jobs=-1).fit(X_tv, y_tv)

X_test = test[ORTH_COLS].values
rf_stats = daily_ic(rf, test, X_test)
xgb_stats = daily_ic(xgbm, test, X_test)

print("\n" + "="*60)
print("12因子模型 测试集样本外IC (2021-2025)")
print("="*60)
result = pd.DataFrame({
    "IC均值": [rf_stats[0], xgb_stats[0]],
    "IR": [rf_stats[0]/rf_stats[1], xgb_stats[0]/xgb_stats[1]],
    "IC>0占比": [rf_stats[2], xgb_stats[2]],
}, index=["随机森林", "XGBoost"])
print(result.round(4).to_string())

print("\n" + "="*60)
print("对比:第一阶段4因子 vs 进阶12因子 (测试集IC)")
print("="*60)
print("  第一阶段4因子:  RF=+0.0173, XGB=+0.0178")
print(f"  进阶12因子:     RF={rf_stats[0]:+.4f}, XGB={xgb_stats[0]:+.4f}")

test_pred = test.copy()
test_pred["pred_rf"] = rf.predict(X_test)
test_pred["pred_xgb"] = xgbm.predict(X_test)
test_pred.to_pickle("test_predictions_v2.pkl")
with open("ml_models_v2.pkl", "wb") as f:
    pickle.dump({"rf": rf, "xgb": xgbm, "ORTH_COLS": ORTH_COLS,
                 "best_rf": best_rf, "best_xgb": best_xgb}, f)
print("\n✓ 已保存 test_predictions_v2.pkl 和 ml_models_v2.pkl")

In [ ]:
# ============================================================
# 【排查·合并版】RF和XGB: 未正交化12因子 vs 正交化12因子
# 一次性对比4个组合,汇总成表
# 目的:确认树模型该用哪种因子 + 12因子有没有增量价值
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from itertools import product
import time

FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
               "rsi", "macd_hist", "boll_width", "atr",
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]
ORTH_COLS = [f"{c}_orth" for c in FACTOR_COLS]

EMBARGO = 5
def make_splits(feat):
    dl = feat.index.get_level_values("Date")
    def sl(s, e): return feat[(dl >= pd.Timestamp(s)) & (dl <= pd.Timestamp(e))]
    def dt(df, k):
        if len(df) == 0: return df
        u = df.index.get_level_values("Date").unique().sort_values()
        return df[df.index.get_level_values("Date") <= u[-(k+1)]] if len(u) > k else df
    return (dt(sl("2010-01-01","2018-12-31"), EMBARGO),
            dt(sl("2019-01-01","2020-12-31"), EMBARGO),
            sl("2021-01-01","2025-12-31"))

def daily_ic(model, dz, X):
    tmp = dz.copy(); tmp["p"] = model.predict(X)
    di = tmp.index.get_level_values("Date"); ics = []
    for d, g in tmp.groupby(di):
        if len(g) >= 5 and g["p"].std() > 1e-12:
            ic, _ = spearmanr(g["p"], g["future_ret_5d"])
            if not np.isnan(ic): ics.append(ic)
    ics = np.array(ics)
    return ics.mean(), ics.std(), (ics > 0).mean()

def run_both_models(feat, cols, tag):
    """对给定因子集,跑RF和XGB网格,返回各自测试集IC"""
    train, val, test = make_splits(feat)
    Xtr, ytr = train[cols].values, train["future_ret_5d"].values
    Xv = val[cols].values
    tv = pd.concat([train, val])
    Xtv, ytv = tv[cols].values, tv["future_ret_5d"].values
    Xte = test[cols].values
    results = {}

    # --- RF ---
    print(f"\n[{tag}] 随机森林网格...")
    t0 = time.time()
    best_rf, best_rf_ic = None, -np.inf
    rf_grid = list(product([3, 4, 5], [20, 50, 100], [100, 200, 300]))
    for i, (d, msl, ne) in enumerate(rf_grid):
        m = RandomForestRegressor(n_estimators=ne, max_depth=d, min_samples_leaf=msl,
                                   random_state=42, n_jobs=-1).fit(Xtr, ytr)
        v = daily_ic(m, val, Xv)[0]
        if v > best_rf_ic: best_rf_ic, best_rf = v, (d, msl, ne)
    rf = RandomForestRegressor(n_estimators=best_rf[2], max_depth=best_rf[0],
                                min_samples_leaf=best_rf[1], random_state=42, n_jobs=-1).fit(Xtv, ytv)
    results["RF"] = daily_ic(rf, test, Xte)[0]
    print(f"  RF最优{best_rf}, 测试IC={results['RF']:+.4f} (用时{time.time()-t0:.0f}s)")

    # --- XGB ---
    print(f"[{tag}] XGBoost网格...")
    t0 = time.time()
    best_xgb, best_xgb_ic = None, -np.inf
    xgb_grid = list(product([3, 4, 5], [0.01, 0.03, 0.05, 0.1], [100, 300]))
    for i, (d, lr, ne) in enumerate(xgb_grid):
        m = xgb.XGBRegressor(n_estimators=ne, max_depth=d, learning_rate=lr,
                              random_state=42, n_jobs=-1).fit(Xtr, ytr)
        v = daily_ic(m, val, Xv)[0]
        if v > best_xgb_ic: best_xgb_ic, best_xgb = v, (d, lr, ne)
    xgbm = xgb.XGBRegressor(n_estimators=best_xgb[2], max_depth=best_xgb[0],
                             learning_rate=best_xgb[1], random_state=42, n_jobs=-1).fit(Xtv, ytv)
    results["XGB"] = daily_ic(xgbm, test, Xte)[0]
    print(f"  XGB最优{best_xgb}, 测试IC={results['XGB']:+.4f} (用时{time.time()-t0:.0f}s)")
    return results

# 跑两种因子集
feat_std = pd.read_pickle("feature_standardized.pkl")
feat_orth = pd.read_pickle("feature_orthogonalized.pkl")

res_raw = run_both_models(feat_std, FACTOR_COLS, "未正交化")
res_orth = run_both_models(feat_orth, ORTH_COLS, "正交化后")

# ============================================================
# 汇总对比表
# ============================================================
print("\n" + "="*60)
print("测试集样本外IC 汇总对比")
print("="*60)
summary = pd.DataFrame({
    "随机森林": [0.0173, res_raw["RF"], res_orth["RF"]],
    "XGBoost": [0.0178, res_raw["XGB"], res_orth["XGB"]],
}, index=["第一阶段4因子", "未正交化12因子", "正交化后12因子"])
print(summary.round(4).to_string())

print("\n--- 结论 ---")
best_raw = max(res_raw["RF"], res_raw["XGB"])
if best_raw > max(res_orth["RF"], res_orth["XGB"]) + 0.001:
    print("• 树模型用未正交化因子更好(正交化削弱了模型)")
if best_raw > 0.0178:
    print("• 未正交化12因子超过4因子: 因子扩充有增量价值")
else:
    print("• 12因子仍未超4因子: 技术面/基本面对5日预测帮助有限")

In [ ]:
# ============================================================
# 【进阶·最终预测】未正交化12因子 + XGBoost → 生成预测收益
# 用最优参数(前面网格已确定:depth=3,lr=0.01,n_est=100)
# 存下测试集预测,供组合优化(最大夏普)用
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import xgboost as xgb
import pickle

# 未正交化的12因子(去极值+标准化后)
feat = pd.read_pickle("feature_standardized.pkl")
FACTOR_COLS = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio",
               "rsi", "macd_hist", "boll_width", "atr",
               "roe", "roa", "debt_ratio", "pe", "ht_phase"]

# 划分(带embargo)
EMBARGO = 5
dl = feat.index.get_level_values("Date")
def sl(s, e): return feat[(dl >= pd.Timestamp(s)) & (dl <= pd.Timestamp(e))]
def dt(df, k):
    u = df.index.get_level_values("Date").unique().sort_values()
    return df[df.index.get_level_values("Date") <= u[-(k+1)]] if len(u) > k else df

train = dt(sl("2010-01-01", "2018-12-31"), EMBARGO)
val   = dt(sl("2019-01-01", "2020-12-31"), EMBARGO)
test  = sl("2021-01-01", "2025-12-31")

# 合并train+val,用最优参数重训
trainval = pd.concat([train, val])
X_tv, y_tv = trainval[FACTOR_COLS].values, trainval["future_ret_5d"].values

# 最优参数(前面网格搜索确定的未正交化XGB最优)
xgbm = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.01,
                         random_state=42, n_jobs=-1).fit(X_tv, y_tv)

# 测试集预测
X_test = test[FACTOR_COLS].values
test_pred = test.copy()
test_pred["pred_xgb"] = xgbm.predict(X_test)

# 验证IC(确认和之前0.0190一致)
def daily_ic(dz, pred_col):
    di = dz.index.get_level_values("Date"); ics = []
    for d, g in dz.groupby(di):
        if len(g) >= 5 and g[pred_col].std() > 1e-12:
            ic, _ = spearmanr(g[pred_col], g["future_ret_5d"])
            if not np.isnan(ic): ics.append(ic)
    return np.mean(ics), (np.array(ics) > 0).mean()

ic_mean, ic_pos = daily_ic(test_pred, "pred_xgb")
print(f"最终模型(未正交化12因子+XGB) 测试集IC={ic_mean:+.4f}, IC>0占比={ic_pos:.3f}")
print(f"(应该≈0.0190,和之前网格结果一致)")

# 保存预测(组合优化用) + 特征重要性
test_pred.to_pickle("final_predictions.pkl")

# 特征重要性(可解释,报告用)
importance = pd.Series(xgbm.feature_importances_, index=FACTOR_COLS).sort_values(ascending=False)
print(f"\n特征重要性(未正交化,可解读):")
labels_cn = {"momentum_20d":"20日动量","reversal_5d":"5日反转","volatility_20d":"20日波动率",
             "volume_ratio":"成交量比","rsi":"RSI","macd_hist":"MACD柱","boll_width":"布林带宽",
             "atr":"ATR","roe":"ROE","roa":"ROA","debt_ratio":"资产负债率","pe":"市盈率","ht_phase":"希尔伯特相位"}
for f, imp in importance.items():
    print(f"  {labels_cn[f]:8}: {imp:.4f}")

with open("final_model.pkl", "wb") as f:
    pickle.dump({"model": xgbm, "FACTOR_COLS": FACTOR_COLS, "importance": importance}, f)
print(f"\n✓ 已保存 final_predictions.pkl 和 final_model.pkl")

In [ ]:
# ============================================================
# 【进阶·组合优化 第1步】准备数据 + 优化函数 + 测试单次优化
# 用cvxpy: Ledoit-Wolf协方差 + 最大夏普 + 风险平价
# 约束:权重≤5%、不做空
# ============================================================
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.covariance import LedoitWolf

# 读数据
final_pred = pd.read_pickle("final_predictions.pkl")   # 测试集XGB预测(含pred_xgb)
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)

# 只保留84只非金融股
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers = [t for t in tickers if t in daily_returns.columns]
daily_returns = daily_returns[tickers]

print(f"预测数据: {final_pred.shape}, 股票池: {len(tickers)}只")
print(f"预测日期范围: {final_pred.index.get_level_values('Date').min().date()} → {final_pred.index.get_level_values('Date').max().date()}")

# ============================================================
# 优化函数1: 最大夏普(标准变量替换法)
# ============================================================
def max_sharpe(mu, Sigma, w_max=0.05):
    n = len(mu)
    if np.all(mu <= 0):    # 全负预测,退化为等权
        return np.ones(n) / n, "all_neg_fallback"
    y = cp.Variable(n)
    kappa = cp.Variable()
    constraints = [mu @ y == 1, cp.sum(y) == kappa, y >= 0, y <= kappa*w_max, kappa >= 0]
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)), constraints)
    try:
        prob.solve()
        if y.value is None or kappa.value is None or kappa.value < 1e-8:
            return np.ones(n)/n, "failed_fallback"
        return y.value / kappa.value, prob.status
    except Exception as e:
        return np.ones(n)/n, f"error_fallback"

# ============================================================
# 优化函数2: 风险平价(Spinu凸公式 + 权重上限后处理)
# ============================================================
def risk_parity(Sigma, w_max=0.05):
    n = Sigma.shape[0]
    w = cp.Variable(n)
    objective = 0.5 * cp.quad_form(w, Sigma) - (1.0/n) * cp.sum(cp.log(w))
    constraints = [w >= 1e-6]
    prob = cp.Problem(cp.Minimize(objective), constraints)
    try:
        prob.solve()
        if w.value is None:
            return np.ones(n)/n, "failed_fallback"
        w_norm = w.value / w.value.sum()
        # 权重上限后处理:超过w_max的截断,多余的按比例分给其他
        w_norm = np.minimum(w_norm, w_max)
        w_norm = w_norm / w_norm.sum()
        return w_norm, prob.status
    except Exception:
        return np.ones(n)/n, "error_fallback"

# ============================================================
# 测试:在测试期第一个调仓日,做一次优化
# ============================================================
test_dates = sorted(final_pred.index.get_level_values("Date").unique())
COV_WINDOW = 126
W_MAX = 0.05

# 找第一个有足够历史的调仓日
first_date = test_dates[0]
hist_end = daily_returns.index[daily_returns.index < first_date]
if len(hist_end) >= COV_WINDOW:
    # 该日的预测收益
    day_pred = final_pred.xs(first_date, level="Date")["pred_xgb"]
    # 选预测收益前50%
    n_select = len(day_pred) // 2
    selected = day_pred.nlargest(n_select).index.tolist()

    # 这些股票的历史收益(前126天)
    hist = daily_returns.loc[hist_end[-COV_WINDOW:], selected].dropna(axis=1)
    selected = hist.columns.tolist()
    mu = day_pred[selected].values

    # Ledoit-Wolf协方差
    lw = LedoitWolf().fit(hist.values)
    Sigma = lw.covariance_

    # 优化
    w_ms, st_ms = max_sharpe(mu, Sigma, W_MAX)
    w_rp, st_rp = risk_parity(Sigma, W_MAX)

    print(f"\n=== 测试单次优化({first_date.date()}) ===")
    print(f"选股数: {len(selected)}, 协方差窗口: {COV_WINDOW}天, 收缩强度: {lw.shrinkage_:.3f}")
    print(f"\n最大夏普: status={st_ms}")
    print(f"  权重[{w_ms.min():.4f},{w_ms.max():.4f}], 和={w_ms.sum():.4f}, 非零{(w_ms>0.001).sum()}只")
    print(f"风险平价: status={st_rp}")
    print(f"  权重[{w_rp.min():.4f},{w_rp.max():.4f}], 和={w_rp.sum():.4f}, 非零{(w_rp>0.001).sum()}只")
    print(f"\n✓ 单次优化测试成功,可以进行完整回测")
else:
    print(f"第一个调仓日历史不足{COV_WINDOW}天")

In [ ]:
# ============================================================
# 【进阶·组合优化 v4 修正版】修复时间对齐bug
# 关键修正:持有期收益从调仓日"之后"开始(rd < d <= end)
#   之前bug:用了调仓日当天收益(rd日还没买入,且和未来5天预测错配)
# 三频率:每天/每5天/每20天
# ============================================================
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.covariance import LedoitWolf
import matplotlib
import matplotlib.pyplot as plt
import pickle, time

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

final_pred = pd.read_pickle("final_predictions.pkl")
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers = [t for t in tickers if t in daily_returns.columns]
daily_returns = daily_returns[tickers]

def max_sharpe(mu, Sigma, w_max):
    n = len(mu)
    if np.all(mu <= 0): return np.ones(n)/n
    y = cp.Variable(n); kappa = cp.Variable()
    cons = [mu @ y == 1, cp.sum(y) == kappa, y >= 0, y <= kappa*w_max, kappa >= 0]
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)), cons)
    try:
        prob.solve()
        if y.value is None or kappa.value is None or kappa.value < 1e-8: return np.ones(n)/n
        return y.value / kappa.value
    except: return np.ones(n)/n

def risk_parity(Sigma, w_max):
    n = Sigma.shape[0]
    w = cp.Variable(n)
    obj = 0.5*cp.quad_form(w, Sigma) - (1.0/n)*cp.sum(cp.log(w))
    prob = cp.Problem(cp.Minimize(obj), [w >= 1e-6])
    try:
        prob.solve()
        if w.value is None: return np.ones(n)/n
        wn = w.value / w.value.sum()
        wn = np.minimum(wn, w_max); wn = wn / wn.sum()
        return wn
    except: return np.ones(n)/n

def run_backtest(rebal_freq, cov_window=126, w_max=0.07, top_pct=0.2):
    test_dates = sorted(final_pred.index.get_level_values("Date").unique())
    rebal_dates = [test_dates[i] for i in range(0, len(test_dates), rebal_freq)]
    ms_daily = pd.Series(0.0, index=test_dates)
    rp_daily = pd.Series(0.0, index=test_dates)
    ms_weights, rp_weights = [], []

    for i, rd in enumerate(rebal_dates):
        hist_dates = daily_returns.index[daily_returns.index < rd]
        if len(hist_dates) < cov_window:
            continue
        day_pred = final_pred.xs(rd, level="Date")["pred_xgb"]
        n_select = max(int(len(day_pred) * top_pct), 5)
        selected = day_pred.nlargest(n_select).index.tolist()
        hist = daily_returns.loc[hist_dates[-cov_window:], selected].dropna(axis=1)
        selected = hist.columns.tolist()
        if len(selected) < 5:
            continue
        mu = day_pred[selected].values
        Sigma = LedoitWolf().fit(hist.values).covariance_
        w_ms = max_sharpe(mu, Sigma, w_max)
        w_rp = risk_parity(Sigma, w_max)
        ms_weights.append(pd.Series(w_ms, index=selected, name=rd))
        rp_weights.append(pd.Series(w_rp, index=selected, name=rd))

        end_date = rebal_dates[i+1] if i+1 < len(rebal_dates) else test_dates[-1]
        # ★★★ 关键修正:rd < d <= end_date (收益从调仓日之后开始,不含当天) ★★★
        hold = [d for d in test_dates if rd < d <= end_date]
        for d in hold:
            if d in daily_returns.index:
                r = np.nan_to_num(daily_returns.loc[d, selected].values, nan=0.0)
                ms_daily[d] = r @ w_ms
                rp_daily[d] = r @ w_rp
    return {"ms": ms_daily, "rp": rp_daily,
            "ms_w": ms_weights, "rp_w": rp_weights, "rebal_dates": rebal_dates}

def perf(daily):
    daily = daily[daily != 0]
    total = (1+daily).prod() - 1
    ann = (1+daily).prod()**(252/len(daily)) - 1
    vol = daily.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+daily).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return total*100, ann*100, vol*100, sharpe, dd*100

results = {}
for freq, name in [(1, "每天"), (5, "每5天"), (20, "每20天")]:
    print(f"回测{name}调仓(修正版)...")
    t0 = time.time()
    results[name] = run_backtest(rebal_freq=freq)
    print(f"  用时{time.time()-t0:.0f}s")

print("\n" + "="*75)
print("三频率组合优化绩效 (修正时间对齐,无成本,真实日收益)")
print("="*75)
rows = []
for name in ["每天", "每5天", "每20天"]:
    for pname, key in [("最大夏普","ms"), ("风险平价","rp")]:
        t, a, v, s, d = perf(results[name][key])
        rows.append({"频率": name, "组合": pname, "年化%": a, "波动%": v, "夏普": s, "回撤%": d})
comp = pd.DataFrame(rows)
print(comp.round(2).to_string(index=False))

with open("portfolio_opt_v4.pkl", "wb") as f:
    pickle.dump({"results": results, "comparison": comp}, f)
print("\n✓ 已保存 portfolio_opt_v4.pkl")

In [ ]:
# ============================================================
# 【进阶·第4块 修正版】交易成本与换手率(三频率)
# 用修复时间对齐后的结果; 成本 = 换手率 × 5bps
# 展示"交易摩擦如何吞噬高频调仓"(任务讨论点②)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pickle

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

with open("portfolio_opt_v4.pkl", "rb") as f:
    opt = pickle.load(f)
results = opt["results"]   # {每天, 每5天, 每20天}

COST_BPS = 5

def compute_turnover(w_curr, w_prev):
    all_idx = w_curr.index.union(w_prev.index)
    wc = w_curr.reindex(all_idx, fill_value=0)
    wp = w_prev.reindex(all_idx, fill_value=0)
    return (wc - wp).abs().sum()

def turnover_series(weights_history):
    turnovers = [weights_history[0].sum()]
    for t in range(1, len(weights_history)):
        turnovers.append(compute_turnover(weights_history[t], weights_history[t-1]))
    return turnovers

def apply_cost(daily_ret, weights_history, rebal_dates, cost_bps):
    daily_net = daily_ret.copy()
    turnovers = turnover_series(weights_history)
    for i, rd in enumerate(rebal_dates[:len(turnovers)]):
        cost = turnovers[i] * cost_bps / 10000
        if rd in daily_net.index:
            daily_net[rd] -= cost
    return daily_net, turnovers

def perf(daily):
    daily = daily[daily != 0]
    total = (1+daily).prod() - 1
    ann = (1+daily).prod()**(252/len(daily)) - 1
    vol = daily.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+daily).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return total*100, ann*100, vol*100, sharpe, dd*100

freq_map = {"每天": 1, "每5天": 5, "每20天": 20}

print("="*90)
print("交易成本分析 (修正版,三频率,成本=换手率×5bps)")
print("="*90)
rows = []
for name in ["每天", "每5天", "每20天"]:
    res = results[name]
    rebal_dates = res["rebal_dates"]
    for pname, dkey, wkey in [("最大夏普","ms","ms_w"), ("风险平价","rp","rp_w")]:
        daily = res[dkey]
        weights = res[wkey]
        daily_net, turnovers = apply_cost(daily, weights, rebal_dates, COST_BPS)
        avg_to = np.mean(turnovers)
        rebals_per_year = 252 / freq_map[name]
        annual_to = avg_to * rebals_per_year
        _, a0, _, s0, d0 = perf(daily)
        _, a1, v1, s1, d1 = perf(daily_net)
        rows.append({
            "频率": name, "组合": pname,
            "年化换手": annual_to,
            "年化%(无成本)": a0, "年化%(有成本)": a1,
            "夏普(无成本)": s0, "夏普(有成本)": s1,
            "夏普损失": s0-s1,
        })

cost_comp = pd.DataFrame(rows)
print(cost_comp.round(3).to_string(index=False))

# ============================================================
# 换手率调整后夏普(任务要求的指标) + 关键对比
# ============================================================
print("\n" + "="*60)
print("交易摩擦对不同频率的吞噬效应")
print("="*60)
for name in ["每天", "每5天", "每20天"]:
    sub = cost_comp[cost_comp["频率"]==name]
    to = sub["年化换手"].mean()
    loss = sub["夏普损失"].mean()
    s0 = sub["夏普(无成本)"].mean()
    s1 = sub["夏普(有成本)"].mean()
    print(f"{name:6}: 年化换手{to:5.1f} | 夏普 {s0:.2f}→{s1:.2f} (损失{loss:.3f}, {loss/s0*100:.0f}%)")

# ============================================================
# 画图:扣成本前后夏普对比(三频率)
# ============================================================
fig, ax = plt.subplots(figsize=(11, 6))
freqs = ["每天", "每5天", "每20天"]
x = np.arange(len(freqs))
width = 0.35
s0_list = [cost_comp[cost_comp["频率"]==f]["夏普(无成本)"].mean() for f in freqs]
s1_list = [cost_comp[cost_comp["频率"]==f]["夏普(有成本)"].mean() for f in freqs]
ax.bar(x - width/2, s0_list, width, label="无成本", color="steelblue")
ax.bar(x + width/2, s1_list, width, label="有成本(5bps)", color="coral")
ax.set_xticks(x); ax.set_xticklabels(freqs)
ax.set_ylabel("夏普比率"); ax.set_title("交易成本对不同调仓频率的影响", fontsize=13)
ax.legend(); ax.grid(alpha=0.3, axis="y")
for i, (s0, s1) in enumerate(zip(s0_list, s1_list)):
    ax.text(i-width/2, s0+0.02, f"{s0:.2f}", ha="center", fontsize=9)
    ax.text(i+width/2, s1+0.02, f"{s1:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

with open("cost_analysis_v2.pkl", "wb") as f:
    pickle.dump({"cost_comp": cost_comp}, f)
print("\n✓ 已保存 cost_analysis_v2.pkl")

# ============================================================
# 【第5块·准备】第一阶段重跑:84只+4因子+每天全换+等权
# 统一口径(和进阶都84只、都每天全换),修复时序(收益从rd之后)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import xgboost as xgb
import pickle

# 用84只非金融股的4因子(从feature_standardized取前4个,或从factor_table)
# 这里用factor_table(第一阶段的4因子),筛84只
factor_table = pd.read_pickle("factor_table.pkl")
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
factor_84 = factor_table[factor_table.index.get_level_values("Ticker").isin(tickers84)]

FACTOR4 = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
tickers84 = [t for t in tickers84 if t in daily_returns.columns]
daily_returns = daily_returns[tickers84]

# 划分(带embargo,和之前一致)
EMBARGO = 5
dl = factor_84.index.get_level_values("Date")
def sl(s, e): return factor_84[(dl >= pd.Timestamp(s)) & (dl <= pd.Timestamp(e))]
def dt(df, k):
    u = df.index.get_level_values("Date").unique().sort_values()
    return df[df.index.get_level_values("Date") <= u[-(k+1)]] if len(u) > k else df
train = dt(sl("2010-01-01","2018-12-31"), EMBARGO)
val   = dt(sl("2019-01-01","2020-12-31"), EMBARGO)
test  = sl("2021-01-01","2025-12-31")
print(f"第一阶段(84只4因子): train={len(train)}, val={len(val)}, test={len(test)}")

# 训练XGB(用之前第一阶段XGB的最优参数,或简单网格)
from itertools import product
def daily_ic(model, dz, X):
    tmp = dz.copy(); tmp["p"] = model.predict(X)
    di = tmp.index.get_level_values("Date"); ics = []
    for d, g in tmp.groupby(di):
        if len(g) >= 5 and g["p"].std() > 1e-12:
            ic, _ = spearmanr(g["p"], g["future_ret_5d"])
            if not np.isnan(ic): ics.append(ic)
    return np.mean(ics)

X_train, y_train = train[FACTOR4].values, train["future_ret_5d"].values
X_val = val[FACTOR4].values
best, best_ic = None, -np.inf
for depth, lr, ne in product([3,4,5],[0.01,0.03,0.05],[100,300]):
    m = xgb.XGBRegressor(n_estimators=ne, max_depth=depth, learning_rate=lr,
                          random_state=42, n_jobs=-1).fit(X_train, y_train)
    v = daily_ic(m, val, X_val)
    if v > best_ic: best_ic, best = v, (depth, lr, ne)

trainval = pd.concat([train, val])
xgbm = xgb.XGBRegressor(n_estimators=best[2], max_depth=best[0], learning_rate=best[1],
                         random_state=42, n_jobs=-1).fit(trainval[FACTOR4].values, trainval["future_ret_5d"].values)
test_pred = test.copy()
test_pred["pred"] = xgbm.predict(test[FACTOR4].values)
test_ic = daily_ic(xgbm, test, test[FACTOR4].values)
print(f"第一阶段4因子(84只)测试IC: {test_ic:+.4f}")

# ============================================================
# 每天全换等权回测(修复时序:收益从rd之后)
# ============================================================
def backtest_daily_ew(pred, daily_ret, top_pct=0.2):
    test_dates = sorted(pred.index.get_level_values("Date").unique())
    port_daily = pd.Series(0.0, index=test_dates)
    weights_hist = []
    for i, rd in enumerate(test_dates):
        day_pred = pred.xs(rd, level="Date")["pred"]
        n_sel = max(int(len(day_pred)*top_pct), 5)
        selected = day_pred.nlargest(n_sel).index.tolist()
        selected = [s for s in selected if s in daily_ret.columns]
        if len(selected) < 5: continue
        w = pd.Series(1/len(selected), index=selected, name=rd)
        weights_hist.append(w)
        # 收益从rd之后(下一天)开始
        if i+1 < len(test_dates):
            next_d = test_dates[i+1]
            if next_d in daily_ret.index:
                r = np.nan_to_num(daily_ret.loc[next_d, selected].values, nan=0.0)
                port_daily[next_d] = r.mean()
    return port_daily, weights_hist

s1_daily, s1_weights = backtest_daily_ew(test_pred, daily_returns)

def perf(daily):
    daily = daily[daily != 0]
    total = (1+daily).prod() - 1
    ann = (1+daily).prod()**(252/len(daily)) - 1
    vol = daily.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+daily).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return total*100, ann*100, vol*100, sharpe, dd*100

t, a, v, s, d = perf(s1_daily)
print(f"\n第一阶段ML等权(84只,每天全换,无成本):")
print(f"  总收益{t:.1f}%, 年化{a:.1f}%, 波动{v:.1f}%, 夏普{s:.2f}, 回撤{d:.1f}%")

with open("stage1_84_daily.pkl", "wb") as f:
    pickle.dump({"daily": s1_daily, "weights": s1_weights, "test_ic": test_ic}, f)
print("\n✓ 已保存 stage1_84_daily.pkl")

In [ ]:
# ============================================================
# 【进阶·选做·修正版】换手率控制:变量替换法+换手惩罚
# 用和主线一致的变量替换法(λ=0时=主线最大夏普),加y空间换手惩罚
# 对比不同λ扣费后净收益,验证换手控制改善
# ============================================================
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.covariance import LedoitWolf
import pickle

final_pred = pd.read_pickle("final_predictions.pkl")
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers = [t for t in tickers if t in daily_returns.columns]
daily_returns = daily_returns[tickers]

# ============================================================
# 变量替换法最大夏普 + y空间换手惩罚(λ=0时等于主线)
# ============================================================
def max_sharpe_turnover(mu, Sigma, w_prev, w_max=0.07, lam=0.0):
    n = len(mu)
    if np.all(mu <= 0): return np.ones(n)/n
    y = cp.Variable(n); kappa = cp.Variable()
    cons = [mu @ y == 1, cp.sum(y) == kappa, y >= 0, y <= kappa*w_max, kappa >= 0]
    obj = cp.quad_form(y, Sigma)
    if w_prev is not None and lam > 0:
        obj = obj + lam * cp.norm(y - cp.multiply(w_prev, kappa), 1)  # y空间换手惩罚
    prob = cp.Problem(cp.Minimize(obj), cons)
    try:
        prob.solve()
        if y.value is None or kappa.value is None or kappa.value < 1e-8:
            return w_prev if w_prev is not None else np.ones(n)/n
        w = np.maximum(y.value / kappa.value, 0)
        return w / w.sum()
    except:
        return w_prev if w_prev is not None else np.ones(n)/n

def run_backtest_turnover(lam, rebal_freq=5, cov_window=126, w_max=0.07, top_pct=0.2):
    test_dates = sorted(final_pred.index.get_level_values("Date").unique())
    rebal_dates = [test_dates[i] for i in range(0, len(test_dates), rebal_freq)]
    port_daily = pd.Series(0.0, index=test_dates)
    weights_hist = []
    prev_w_series = None

    for i, rd in enumerate(rebal_dates):
        hist_dates = daily_returns.index[daily_returns.index < rd]
        if len(hist_dates) < cov_window:
            continue
        day_pred = final_pred.xs(rd, level="Date")["pred_xgb"]
        n_select = max(int(len(day_pred) * top_pct), 5)
        selected = day_pred.nlargest(n_select).index.tolist()
        hist = daily_returns.loc[hist_dates[-cov_window:], selected].dropna(axis=1)
        selected = hist.columns.tolist()
        if len(selected) < 5:
            continue
        mu = day_pred[selected].values
        Sigma = LedoitWolf().fit(hist.values).covariance_
        w_prev = prev_w_series.reindex(selected, fill_value=0).values if prev_w_series is not None else None
        w = max_sharpe_turnover(mu, Sigma, w_prev, w_max, lam)
        w_series = pd.Series(w, index=selected, name=rd)
        weights_hist.append(w_series)
        prev_w_series = w_series

        end_date = rebal_dates[i+1] if i+1 < len(rebal_dates) else test_dates[-1]
        hold = [d for d in test_dates if rd < d <= end_date]
        for d in hold:
            if d in daily_returns.index:
                r = np.nan_to_num(daily_returns.loc[d, selected].values, nan=0.0)
                port_daily[d] = r @ w
    return port_daily, weights_hist, rebal_dates

def compute_turnover(w_curr, w_prev):
    all_idx = w_curr.index.union(w_prev.index)
    return (w_curr.reindex(all_idx, fill_value=0) - w_prev.reindex(all_idx, fill_value=0)).abs().sum()
def turnover_series(weights):
    tos = [weights[0].sum()]
    for t in range(1, len(weights)):
        tos.append(compute_turnover(weights[t], weights[t-1]))
    return tos
def apply_cost(daily_ret, weights, rebal_dates, cost_bps=5):
    daily_net = daily_ret.copy()
    tos = turnover_series(weights)
    for i, rd in enumerate(rebal_dates[:len(tos)]):
        if rd in daily_net.index:
            daily_net[rd] -= tos[i] * cost_bps / 10000
    return daily_net, tos
def perf(daily):
    daily = daily[daily != 0]
    ann = (1+daily).prod()**(252/len(daily)) - 1
    vol = daily.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    return ann*100, sharpe

print("换手率控制(变量替换法,λ=0时=主线最大夏普,成本5bps)")
print("="*80)
rows = []
for lam in [0.0, 0.001, 0.005, 0.01, 0.02]:
    daily, weights, rebal_dates = run_backtest_turnover(lam=lam)
    daily_net, tos = apply_cost(daily, weights, rebal_dates, 5)
    annual_to = np.mean(tos) * (252/5)
    a0, s0 = perf(daily)
    a1, s1 = perf(daily_net)
    rows.append({"λ": lam, "年化换手": annual_to, "夏普(无成本)": s0,
                 "夏普(有成本)": s1, "年化%(有成本)": a1})
    print(f"λ={lam}: 换手{annual_to:.1f}, 夏普(无成本){s0:.3f}, 夏普(有成本){s1:.3f}, 年化(有成本){a1:.1f}%")

comp = pd.DataFrame(rows)
best_idx = comp["夏普(有成本)"].idxmax()
best_lam = comp.loc[best_idx, "λ"]
base = comp[comp["λ"]==0]["夏普(有成本)"].values[0]
print(f"\n扣费后夏普最高: λ={best_lam} (夏普{comp.loc[best_idx,'夏普(有成本)']:.3f})")
print(f"无控制(λ=0)扣费后夏普: {base:.3f}")
if best_lam > 0:
    print(f"→ 换手控制改善净收益: {base:.3f} → {comp.loc[best_idx,'夏普(有成本)']:.3f}")

with open("turnover_control.pkl", "wb") as f:
    pickle.dump({"comparison": comp}, f)
print("\n✓ turnover_control.pkl (变量替换法版)")

In [ ]:
# ============================================================
# 【第5块·准备】第一阶段重跑:84只+4因子+每5天调仓+等权
# 和主策略统一(都84只、都每5天),修复时序(收益从rd之后)
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import xgboost as xgb
import pickle

factor_table = pd.read_pickle("factor_table.pkl")
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
factor_84 = factor_table[factor_table.index.get_level_values("Ticker").isin(tickers84)]

FACTOR4 = ["momentum_20d", "reversal_5d", "volatility_20d", "volume_ratio"]
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
tickers84 = [t for t in tickers84 if t in daily_returns.columns]
daily_returns = daily_returns[tickers84]

EMBARGO = 5
dl = factor_84.index.get_level_values("Date")
def sl(s, e): return factor_84[(dl >= pd.Timestamp(s)) & (dl <= pd.Timestamp(e))]
def dt(df, k):
    u = df.index.get_level_values("Date").unique().sort_values()
    return df[df.index.get_level_values("Date") <= u[-(k+1)]] if len(u) > k else df
train = dt(sl("2010-01-01","2018-12-31"), EMBARGO)
val   = dt(sl("2019-01-01","2020-12-31"), EMBARGO)
test  = sl("2021-01-01","2025-12-31")
print(f"第一阶段(84只4因子): train={len(train)}, val={len(val)}, test={len(test)}")

from itertools import product
def daily_ic(model, dz, X):
    tmp = dz.copy(); tmp["p"] = model.predict(X)
    di = tmp.index.get_level_values("Date"); ics = []
    for d, g in tmp.groupby(di):
        if len(g) >= 5 and g["p"].std() > 1e-12:
            ic, _ = spearmanr(g["p"], g["future_ret_5d"])
            if not np.isnan(ic): ics.append(ic)
    return np.mean(ics)

X_train, y_train = train[FACTOR4].values, train["future_ret_5d"].values
X_val = val[FACTOR4].values
best, best_ic = None, -np.inf
for depth, lr, ne in product([3,4,5],[0.01,0.03,0.05],[100,300]):
    m = xgb.XGBRegressor(n_estimators=ne, max_depth=depth, learning_rate=lr,
                          random_state=42, n_jobs=-1).fit(X_train, y_train)
    v = daily_ic(m, val, X_val)
    if v > best_ic: best_ic, best = v, (depth, lr, ne)

trainval = pd.concat([train, val])
xgbm = xgb.XGBRegressor(n_estimators=best[2], max_depth=best[0], learning_rate=best[1],
                         random_state=42, n_jobs=-1).fit(trainval[FACTOR4].values, trainval["future_ret_5d"].values)
test_pred = test.copy()
test_pred["pred"] = xgbm.predict(test[FACTOR4].values)
test_ic = daily_ic(xgbm, test, test[FACTOR4].values)
print(f"第一阶段4因子(84只)测试IC: {test_ic:+.4f}")

# ============================================================
# 每5天调仓等权回测(修复时序:收益从rd之后; 持有到下个调仓日)
# ============================================================
def backtest_ew(pred, daily_ret, rebal_freq=5, top_pct=0.2):
    test_dates = sorted(pred.index.get_level_values("Date").unique())
    rebal_dates = [test_dates[i] for i in range(0, len(test_dates), rebal_freq)]
    port_daily = pd.Series(0.0, index=test_dates)
    weights_hist = []
    for i, rd in enumerate(rebal_dates):
        day_pred = pred.xs(rd, level="Date")["pred"]
        n_sel = max(int(len(day_pred)*top_pct), 5)
        selected = day_pred.nlargest(n_sel).index.tolist()
        selected = [s for s in selected if s in daily_ret.columns]
        if len(selected) < 5: continue
        w = pd.Series(1/len(selected), index=selected, name=rd)
        weights_hist.append(w)
        # 持有期:从rd之后到下个调仓日
        end_date = rebal_dates[i+1] if i+1 < len(rebal_dates) else test_dates[-1]
        hold = [d for d in test_dates if rd < d <= end_date]
        for d in hold:
            if d in daily_ret.index:
                r = np.nan_to_num(daily_ret.loc[d, selected].values, nan=0.0)
                port_daily[d] = r.mean()
    return port_daily, weights_hist

s1_daily, s1_weights = backtest_ew(test_pred, daily_returns, rebal_freq=5)

def perf(daily):
    daily = daily[daily != 0]
    total = (1+daily).prod() - 1
    ann = (1+daily).prod()**(252/len(daily)) - 1
    vol = daily.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+daily).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return total*100, ann*100, vol*100, sharpe, dd*100

t, a, v, s, d = perf(s1_daily)
print(f"\n第一阶段ML等权(84只,每5天,无成本):")
print(f"  总收益{t:.1f}%, 年化{a:.1f}%, 波动{v:.1f}%, 夏普{s:.2f}, 回撤{d:.1f}%")

with open("stage1_84_5d.pkl", "wb") as f:
    pickle.dump({"daily": s1_daily, "weights": s1_weights, "test_ic": test_ic}, f)
print("\n✓ stage1_84_5d.pkl")

# ============================================================
# 【第5块·最终】四条净值曲线对比 + 完整指标表
# 全部84只、每天全换,统一口径
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pickle

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读数据
with open("stage1_84_daily.pkl", "rb") as f:
    s1 = pickle.load(f)
with open("portfolio_opt_v4.pkl", "rb") as f:
    v4 = pickle.load(f)
res_daily = v4["results"]["每天"]   # 进阶每天版

spx = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True)
spx_ret = spx.iloc[:, 0] if spx.shape[1] > 0 else spx

COST_BPS = 5

# 换手率+成本函数
def compute_turnover(w_curr, w_prev):
    all_idx = w_curr.index.union(w_prev.index)
    return (w_curr.reindex(all_idx, fill_value=0) - w_prev.reindex(all_idx, fill_value=0)).abs().sum()

def turnover_series(weights):
    tos = [weights[0].sum()]
    for t in range(1, len(weights)):
        tos.append(compute_turnover(weights[t], weights[t-1]))
    return tos

def apply_cost_daily(daily_ret, weights, cost_bps):
    """每天全换:每个交易日都调仓,在收益日扣成本"""
    daily_net = daily_ret.copy()
    tos = turnover_series(weights)
    dates = daily_ret.index
    # 换手发生在调仓日,成本在下一天收益里扣(和收益时序对齐)
    for i, to in enumerate(tos):
        if i+1 < len(dates):
            cost_date = dates[i+1] if hasattr(dates, '__getitem__') else None
    # 简化:按平均换手在每个收益日扣
    avg_to = np.mean(tos)
    daily_net = daily_net - np.where(daily_net != 0, avg_to * cost_bps/10000, 0)
    return daily_net, tos

# ① 第一阶段无成本
s1_daily = s1["daily"]
s1_weights = s1["weights"]
# ② 第一阶段有成本
s1_net, s1_tos = apply_cost_daily(s1_daily, s1_weights, COST_BPS)

# ③④ 进阶(每天)有成本
ms_daily = res_daily["ms"]; ms_weights = res_daily["ms_w"]
rp_daily = res_daily["rp"]; rp_weights = res_daily["rp_w"]
ms_net, ms_tos = apply_cost_daily(ms_daily, ms_weights, COST_BPS)
rp_net, rp_tos = apply_cost_daily(rp_daily, rp_weights, COST_BPS)

# ============================================================
# 绩效函数(含信息比率)
# ============================================================
def perf_full(daily, benchmark=None):
    d = daily[daily != 0]
    total = (1+d).prod() - 1
    ann = (1+d).prod()**(252/len(d)) - 1
    vol = d.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+d).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    # 信息比率:超额收益/跟踪误差
    ir = np.nan
    if benchmark is not None:
        common = d.index.intersection(benchmark.index)
        excess = d.reindex(common) - benchmark.reindex(common)
        excess = excess.dropna()
        if len(excess) > 0 and excess.std() > 0:
            ir = (excess.mean() * 252) / (excess.std() * np.sqrt(252))
    return {"总收益%": total*100, "年化%": ann*100, "波动%": vol*100,
            "夏普": sharpe, "最大回撤%": dd*100, "信息比率": ir}

# 对齐基准到测试期
test_dates = s1_daily.index
spx_aligned = spx_ret.reindex(test_dates).dropna()

# ============================================================
# 汇总指标表
# ============================================================
rows = {}
rows["①第一阶段ML等权(无成本)"] = perf_full(s1_daily, spx_aligned)
rows["②第一阶段ML等权(有成本)"] = perf_full(s1_net, spx_aligned)
rows["③进阶最大夏普(有成本)"] = perf_full(ms_net, spx_aligned)
rows["④进阶风险平价(有成本)"] = perf_full(rp_net, spx_aligned)
rows["基准标普500"] = perf_full(spx_aligned)

metrics = pd.DataFrame(rows).T
# 加换手率信息
metrics["年化换手"] = [np.mean(s1_tos)*252, np.mean(s1_tos)*252,
                    np.mean(ms_tos)*252, np.mean(rp_tos)*252, 0]
print("="*95)
print("四策略 + 基准 核心指标对比 (测试期2021-2025,全84只每天全换)")
print("="*95)
print(metrics.round(3).to_string())

# ============================================================
# 四条净值曲线
# ============================================================
fig, ax = plt.subplots(figsize=(14, 7))
curves = [
    ("① 第一阶段ML等权(无成本)", s1_daily, "-", 1.5),
    ("② 第一阶段ML等权(有成本)", s1_net, "-", 1.5),
    ("③ 进阶最大夏普(有成本)", ms_net, "-", 1.5),
    ("④ 进阶风险平价(有成本)", rp_net, "-", 1.5),
]
for name, daily, style, lw in curves:
    nav = (1 + daily).cumprod()
    ax.plot(nav.index, nav, label=name, linewidth=lw, linestyle=style)
# 基准
spx_nav = (1 + spx_aligned).cumprod()
ax.plot(spx_nav.index, spx_nav, label="基准(标普500)", linewidth=1.8, color="gray", ls="--")

ax.set_title("策略净值曲线对比 (2021-2025, 84只非金融股)", fontsize=14)
ax.set_xlabel("日期"); ax.set_ylabel("净值")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

metrics.to_pickle("final_comparison.pkl")
print("\n✓ 已保存 final_comparison.pkl")

In [ ]:
# ============================================================
# 【第5块·最终】四条净值曲线 + 完整指标表(全84只,每5天)
# ①②第一阶段4因子等权, ③④进阶12因子优化, 基准标普500
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pickle

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读数据
with open("stage1_84_5d.pkl", "rb") as f:
    s1 = pickle.load(f)
with open("portfolio_opt_v4.pkl", "rb") as f:
    v4 = pickle.load(f)
res = v4["results"]["每5天"]   # ★进阶用每5天

spx = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True)
spx_ret = spx.iloc[:, 0]

COST_BPS = 5

# 换手率+成本
def compute_turnover(w_curr, w_prev):
    all_idx = w_curr.index.union(w_prev.index)
    return (w_curr.reindex(all_idx, fill_value=0) - w_prev.reindex(all_idx, fill_value=0)).abs().sum()

def turnover_series(weights):
    tos = [weights[0].sum()]
    for t in range(1, len(weights)):
        tos.append(compute_turnover(weights[t], weights[t-1]))
    return tos

def apply_cost(daily_ret, weights, rebal_dates, cost_bps):
    daily_net = daily_ret.copy()
    tos = turnover_series(weights)
    for i, rd in enumerate(rebal_dates[:len(tos)]):
        if rd in daily_net.index:
            daily_net[rd] -= tos[i] * cost_bps / 10000
    return daily_net, tos

# 第一阶段调仓日(每5天)
s1_daily = s1["daily"]
s1_weights = s1["weights"]
test_dates = sorted(s1_daily.index)
s1_rebal = [test_dates[i] for i in range(0, len(test_dates), 5)]

# ①第一阶段无成本, ②第一阶段有成本
s1_net, s1_tos = apply_cost(s1_daily, s1_weights, s1_rebal, COST_BPS)

# ③④进阶(每5天)有成本
ms_daily, ms_w = res["ms"], res["ms_w"]
rp_daily, rp_w = res["rp"], res["rp_w"]
rebal_dates = res["rebal_dates"]
ms_net, ms_tos = apply_cost(ms_daily, ms_w, rebal_dates, COST_BPS)
rp_net, rp_tos = apply_cost(rp_daily, rp_w, rebal_dates, COST_BPS)

# 绩效(含信息比率IR)
def perf_full(daily, benchmark=None):
    d = daily[daily != 0]
    total = (1+d).prod() - 1
    ann = (1+d).prod()**(252/len(d)) - 1
    vol = d.std() * np.sqrt(252)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1+d).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    ir = np.nan
    if benchmark is not None:
        common = d.index.intersection(benchmark.index)
        excess = (d.reindex(common) - benchmark.reindex(common)).dropna()
        if len(excess) > 0 and excess.std() > 0:
            ir = (excess.mean() * 252) / (excess.std() * np.sqrt(252))
    return {"总收益%": total*100, "年化%": ann*100, "波动%": vol*100,
            "夏普": sharpe, "最大回撤%": dd*100, "信息比率": ir}

spx_aligned = spx_ret.reindex(test_dates).dropna()

rows = {}
rows["①第一阶段ML等权(无成本)"] = perf_full(s1_daily, spx_aligned)
rows["②第一阶段ML等权(有成本)"] = perf_full(s1_net, spx_aligned)
rows["③进阶最大夏普(有成本)"] = perf_full(ms_net, spx_aligned)
rows["④进阶风险平价(有成本)"] = perf_full(rp_net, spx_aligned)
rows["基准标普500"] = perf_full(spx_aligned)

metrics = pd.DataFrame(rows).T
metrics["年化换手"] = [np.mean(s1_tos)*252/5*252/252, np.mean(s1_tos)*(252/5),
                    np.mean(ms_tos)*(252/5), np.mean(rp_tos)*(252/5), 0]
# 修正年化换手计算(每5天,一年约50次调仓)
metrics["年化换手"] = [np.mean(s1_tos)*(252/5), np.mean(s1_tos)*(252/5),
                    np.mean(ms_tos)*(252/5), np.mean(rp_tos)*(252/5), 0]

print("="*95)
print("四策略 + 基准 核心指标对比 (测试期2021-2025, 全84只, 每5天调仓)")
print("="*95)
print(metrics.round(3).to_string())

# 四条曲线
fig, ax = plt.subplots(figsize=(14, 7))
for name, daily, style in [
    ("① 第一阶段ML等权(无成本)", s1_daily, "-"),
    ("② 第一阶段ML等权(有成本)", s1_net, "-"),
    ("③ 进阶最大夏普(有成本)", ms_net, "-"),
    ("④ 进阶风险平价(有成本)", rp_net, "-"),
]:
    nav = (1 + daily).cumprod()
    ax.plot(nav.index, nav, label=name, linewidth=1.5, linestyle=style)
spx_nav = (1 + spx_aligned).cumprod()
ax.plot(spx_nav.index, spx_nav, label="基准(标普500)", linewidth=1.8, color="gray", ls="--")
ax.set_title("策略净值曲线对比 (2021-2025, 84只非金融股, 每5天调仓)", fontsize=14)
ax.set_xlabel("日期"); ax.set_ylabel("净值")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

metrics.to_pickle("final_comparison.pkl")
print("\n✓ final_comparison.pkl")

In [ ]:
# ============================================================
# 【第5块·补充】带换手控制的四条曲线(λ=0.02)
# 作为标准版的对比。①②不变(等权无优化),③④带换手惩罚
# ============================================================
import numpy as np
import pandas as pd
import cvxpy as cp
from sklearn.covariance import LedoitWolf
import matplotlib
import matplotlib.pyplot as plt
import pickle

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

final_pred = pd.read_pickle("final_predictions.pkl")
daily_returns = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
tickers = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers = [t for t in tickers if t in daily_returns.columns]
daily_returns = daily_returns[tickers]

LAM = 0.02

def max_sharpe_t(mu, Sigma, w_prev, w_max=0.07, lam=LAM):
    n = len(mu)
    if np.all(mu <= 0): return np.ones(n)/n
    y = cp.Variable(n); kappa = cp.Variable()
    cons = [mu @ y == 1, cp.sum(y) == kappa, y >= 0, y <= kappa*w_max, kappa >= 0]
    obj = cp.quad_form(y, Sigma)
    if w_prev is not None and lam > 0:
        obj = obj + lam * cp.norm(y - cp.multiply(w_prev, kappa), 1)
    prob = cp.Problem(cp.Minimize(obj), cons)
    try:
        prob.solve()
        if y.value is None or kappa.value is None or kappa.value < 1e-8:
            return w_prev if w_prev is not None else np.ones(n)/n
        w = np.maximum(y.value/kappa.value, 0); return w/w.sum()
    except: return w_prev if w_prev is not None else np.ones(n)/n

def risk_parity_t(Sigma, w_prev, w_max=0.07, lam=LAM):
    n = Sigma.shape[0]; w = cp.Variable(n)
    obj = 0.5*cp.quad_form(w, Sigma) - (1.0/n)*cp.sum(cp.log(w))
    if w_prev is not None and lam > 0:
        obj = obj + lam * cp.norm(w - w_prev, 1)
    prob = cp.Problem(cp.Minimize(obj), [w >= 1e-6])
    try:
        prob.solve()
        if w.value is None: return w_prev if w_prev is not None else np.ones(n)/n
        wn = w.value/w.value.sum(); wn = np.minimum(wn, w_max); return wn/wn.sum()
    except: return w_prev if w_prev is not None else np.ones(n)/n

def run_bt(rebal_freq=5, cov_window=126, w_max=0.07, top_pct=0.2):
    test_dates = sorted(final_pred.index.get_level_values("Date").unique())
    rebal_dates = [test_dates[i] for i in range(0, len(test_dates), rebal_freq)]
    ms_daily = pd.Series(0.0, index=test_dates); rp_daily = pd.Series(0.0, index=test_dates)
    ms_w, rp_w = [], []; prev_ms, prev_rp = None, None
    for i, rd in enumerate(rebal_dates):
        hist_dates = daily_returns.index[daily_returns.index < rd]
        if len(hist_dates) < cov_window: continue
        day_pred = final_pred.xs(rd, level="Date")["pred_xgb"]
        n_sel = max(int(len(day_pred)*top_pct), 5)
        selected = day_pred.nlargest(n_sel).index.tolist()
        hist = daily_returns.loc[hist_dates[-cov_window:], selected].dropna(axis=1)
        selected = hist.columns.tolist()
        if len(selected) < 5: continue
        mu = day_pred[selected].values
        Sigma = LedoitWolf().fit(hist.values).covariance_
        wp_ms = prev_ms.reindex(selected, fill_value=0).values if prev_ms is not None else None
        wp_rp = prev_rp.reindex(selected, fill_value=0).values if prev_rp is not None else None
        w_ms = max_sharpe_t(mu, Sigma, wp_ms); w_rp = risk_parity_t(Sigma, wp_rp)
        s_ms = pd.Series(w_ms, index=selected, name=rd); s_rp = pd.Series(w_rp, index=selected, name=rd)
        ms_w.append(s_ms); rp_w.append(s_rp); prev_ms, prev_rp = s_ms, s_rp
        end = rebal_dates[i+1] if i+1 < len(rebal_dates) else test_dates[-1]
        for d in [d for d in test_dates if rd < d <= end]:
            if d in daily_returns.index:
                r = np.nan_to_num(daily_returns.loc[d, selected].values, nan=0.0)
                ms_daily[d] = r @ w_ms; rp_daily[d] = r @ w_rp
    return {"ms": ms_daily, "rp": rp_daily, "ms_w": ms_w, "rp_w": rp_w, "rebal_dates": rebal_dates}

res = run_bt()

def compute_to(a, b):
    idx = a.index.union(b.index)
    return (a.reindex(idx, fill_value=0) - b.reindex(idx, fill_value=0)).abs().sum()
def to_series(ws):
    t = [ws[0].sum()]
    for i in range(1, len(ws)): t.append(compute_to(ws[i], ws[i-1]))
    return t
def apply_cost(daily, ws, rebals, bps=5):
    net = daily.copy(); tos = to_series(ws)
    for i, rd in enumerate(rebals[:len(tos)]):
        if rd in net.index: net[rd] -= tos[i]*bps/10000
    return net, tos

with open("stage1_84_5d.pkl", "rb") as f:
    s1 = pickle.load(f)
s1_daily, s1_weights = s1["daily"], s1["weights"]
test_dates = sorted(s1_daily.index)
s1_rebal = [test_dates[i] for i in range(0, len(test_dates), 5)]
s1_net, s1_tos = apply_cost(s1_daily, s1_weights, s1_rebal)
ms_net, ms_tos = apply_cost(res["ms"], res["ms_w"], res["rebal_dates"])
rp_net, rp_tos = apply_cost(res["rp"], res["rp_w"], res["rebal_dates"])

spx = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True)
spx_aligned = spx.iloc[:, 0].reindex(test_dates).dropna()

def perf_full(daily, bench=None):
    d = daily[daily != 0]
    total = (1+d).prod()-1; ann = (1+d).prod()**(252/len(d))-1
    vol = d.std()*np.sqrt(252); sharpe = ann/vol if vol>0 else 0
    cum = (1+d).cumprod(); dd = (cum/cum.cummax()-1).min()
    ir = np.nan
    if bench is not None:
        c = d.index.intersection(bench.index)
        ex = (d.reindex(c)-bench.reindex(c)).dropna()
        if len(ex)>0 and ex.std()>0: ir = (ex.mean()*252)/(ex.std()*np.sqrt(252))
    return {"总收益%": total*100, "年化%": ann*100, "波动%": vol*100, "夏普": sharpe, "最大回撤%": dd*100, "信息比率": ir}

rows = {}
rows["①第一阶段ML等权(无成本)"] = perf_full(s1_daily, spx_aligned)
rows["②第一阶段ML等权(有成本)"] = perf_full(s1_net, spx_aligned)
rows["③进阶最大夏普(有成本,换手控制)"] = perf_full(ms_net, spx_aligned)
rows["④进阶风险平价(有成本,换手控制)"] = perf_full(rp_net, spx_aligned)
rows["基准标普500"] = perf_full(spx_aligned)
metrics = pd.DataFrame(rows).T
metrics["年化换手"] = [np.mean(s1_tos)*(252/5), np.mean(s1_tos)*(252/5),
                    np.mean(ms_tos)*(252/5), np.mean(rp_tos)*(252/5), 0]

print("="*100)
print(f"四策略+基准 [换手控制版 λ={LAM}] (全84只,每5天)")
print("="*100)
print(metrics.round(3).to_string())

fig, ax = plt.subplots(figsize=(14, 7))
for name, daily in [("① 第一阶段ML等权(无成本)", s1_daily), ("② 第一阶段ML等权(有成本)", s1_net),
                    ("③ 进阶最大夏普(换手控制)", ms_net), ("④ 进阶风险平价(换手控制)", rp_net)]:
    ax.plot((1+daily).cumprod().index, (1+daily).cumprod(), label=name, linewidth=1.5)
ax.plot((1+spx_aligned).cumprod().index, (1+spx_aligned).cumprod(), label="基准(标普500)", linewidth=1.8, color="gray", ls="--")
ax.set_title(f"策略净值曲线 [换手控制版 λ={LAM}] (2021-2025, 84只, 每5天)", fontsize=14)
ax.set_xlabel("日期"); ax.set_ylabel("净值"); ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

metrics.to_pickle("final_comparison_turnover.pkl")
print("\n✓ final_comparison_turnover.pkl (换手控制版四条曲线)")

In [ ]:
# ============================================================
# 【探路】新因子单因子IC测试(TA-Lib成交量+周期性指标)
# 目的:看哪些新因子有预测力,值不值得正式加入
# 不改动现有框架,只测IC
# ============================================================
import numpy as np
import pandas as pd
import talib
from scipy.stats import spearmanr

# 读复权OHLCV
stock_data = pd.read_csv("stock_data_clean_adjusted.csv", header=[0,1], index_col=0, parse_dates=True)
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
close_df = stock_data.xs("Close", axis=1, level=1)
high_df  = stock_data.xs("High", axis=1, level=1)
low_df   = stock_data.xs("Low", axis=1, level=1)
vol_df   = stock_data.xs("Volume", axis=1, level=1)
tickers84 = [t for t in tickers84 if t in close_df.columns]

# 未来5天收益(和主策略一致)
future_ret_5d = close_df.shift(-5) / close_df - 1

# ============================================================
# 逐股票算新因子
# ============================================================
factors = {
    "obv_chg": {},      # OBV的20日变化率(累积量转变化)
    "mfi": {},          # 资金流量指标(0-100)
    "ad_chg": {},       # AD的20日变化率
    "adosc": {},        # 量价震荡
    "ht_period": {},    # 主导周期
    "ht_phase": {},     # 主导周期相位
}

for tk in tickers84:
    c = close_df[tk].values.astype(float)
    h = high_df[tk].values.astype(float)
    l = low_df[tk].values.astype(float)
    v = vol_df[tk].values.astype(float)
    if np.isnan(c).all() or len(c) < 100:
        continue

    # 成交量指标
    obv = talib.OBV(c, v)
    factors["obv_chg"][tk] = pd.Series(obv, index=close_df.index).pct_change(20).values  # 变化率
    factors["mfi"][tk] = talib.MFI(h, l, c, v, timeperiod=14)
    ad = talib.AD(h, l, c, v)
    factors["ad_chg"][tk] = pd.Series(ad, index=close_df.index).pct_change(20).values
    factors["adosc"][tk] = talib.ADOSC(h, l, c, v, fastperiod=3, slowperiod=10)

    # 周期性指标
    factors["ht_period"][tk] = talib.HT_DCPERIOD(c)
    factors["ht_phase"][tk] = talib.HT_DCPHASE(c)

# 转成长表 + 算IC
def to_long(dfdict, name):
    df = pd.DataFrame(dfdict, index=close_df.index)
    s = df.stack().rename(name)
    s.index.names = ["Date", "Ticker"]
    return s

fr_long = future_ret_5d.stack().rename("future_ret_5d")
fr_long.index.names = ["Date", "Ticker"]

print("="*60)
print("新因子单因子IC测试 (测试期2021-2025, 未来5天收益)")
print("="*60)
print(f"{'因子':<12} {'IC均值':>10} {'IC>0占比':>10} {'|IC|':>8}")
print("-"*45)

results = {}
for name in factors:
    fac_long = to_long(factors[name], name)
    # 合并因子和未来收益
    merged = pd.concat([fac_long, fr_long], axis=1).dropna()
    # 只看测试期
    merged = merged[merged.index.get_level_values("Date") >= "2021-01-01"]
    # 替换inf(变化率可能产生inf)
    merged = merged.replace([np.inf, -np.inf], np.nan).dropna()
    # 逐日IC
    di = merged.index.get_level_values("Date")
    ics = []
    for d, g in merged.groupby(di):
        if len(g) >= 5 and g[name].std() > 1e-12:
            ic, _ = spearmanr(g[name], g["future_ret_5d"])
            if not np.isnan(ic): ics.append(ic)
    ics = np.array(ics)
    ic_mean = ics.mean()
    results[name] = ic_mean
    print(f"{name:<12} {ic_mean:>+10.4f} {(ics>0).mean():>10.3f} {abs(ic_mean):>8.4f}")

print("\n" + "="*60)
print("对比:现有最强因子波动率IC约0.013, 12因子XGB模型IC0.019")
print("="*60)
# 排序看哪些值得加
sorted_facs = sorted(results.items(), key=lambda x: abs(x[1]), reverse=True)
print("\n按|IC|排序:")
for name, ic in sorted_facs:
    verdict = "★值得考虑" if abs(ic) > 0.008 else ("一般" if abs(ic) > 0.004 else "弱,不建议")
    print(f"  {name:<12}: |IC|={abs(ic):.4f}  {verdict}")

In [ ]:
# ============================================================
# 【验证增量·补充】加ht_phase试试(单因子IC 0.0106)
# 对比: 12因子 / 13因子(+ht_phase) / 15因子(+MFI+OBV+ht_phase)
# ============================================================
import numpy as np
import pandas as pd
import talib
from scipy.stats import spearmanr
import xgboost as xgb

stock_data = pd.read_csv("stock_data_clean_adjusted.csv", header=[0,1], index_col=0, parse_dates=True)
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
close_df = stock_data.xs("Close", axis=1, level=1)
high_df  = stock_data.xs("High", axis=1, level=1)
low_df   = stock_data.xs("Low", axis=1, level=1)
vol_df   = stock_data.xs("Volume", axis=1, level=1)
tickers84 = [t for t in tickers84 if t in close_df.columns]

mfi_dict, obv_dict, phase_dict = {}, {}, {}
for tk in tickers84:
    c = close_df[tk].values.astype(float)
    h = high_df[tk].values.astype(float)
    l = low_df[tk].values.astype(float)
    v = vol_df[tk].values.astype(float)
    if np.isnan(c).all() or len(c) < 100: continue
    mfi_dict[tk] = talib.MFI(h, l, c, v, timeperiod=14)
    obv = talib.OBV(c, v)
    obv_dict[tk] = pd.Series(obv, index=close_df.index).pct_change(20).values
    phase_dict[tk] = talib.HT_DCPHASE(c)   # 希尔伯特主导周期相位

def to_long(dd, name):
    df = pd.DataFrame(dd, index=close_df.index)
    s = df.stack().rename(name); s.index.names = ["Date","Ticker"]
    return s

new_factors = pd.concat([to_long(mfi_dict, "mfi"), to_long(obv_dict, "obv_chg"),
                         to_long(phase_dict, "ht_phase")], axis=1)

feat12 = pd.read_pickle("feature_standardized.pkl")
FACTOR12 = ["momentum_20d","reversal_5d","volatility_20d","volume_ratio",
            "rsi","macd_hist","boll_width","atr","roe","roa","debt_ratio","pe"]

feat = feat12.join(new_factors, how="left")
feat = feat.replace([np.inf, -np.inf], np.nan)
di = feat.index.get_level_values("Date")
def winsor_std(s):
    med = s.median(); mad = (s-med).abs().median()*1.4826
    if mad < 1e-12: return (s-s.mean())/(s.std()+1e-12)
    s = s.clip(med-5*mad, med+5*mad)
    return (s-s.mean())/(s.std()+1e-12)
for col in ["mfi","obv_chg","ht_phase"]:
    feat[col] = feat.groupby(di)[col].transform(winsor_std)

EMBARGO = 5
dl = feat.index.get_level_values("Date")
def sl(s,e): return feat[(dl>=pd.Timestamp(s))&(dl<=pd.Timestamp(e))]
def dt(df,k):
    u=df.index.get_level_values("Date").unique().sort_values()
    return df[df.index.get_level_values("Date")<=u[-(k+1)]] if len(u)>k else df

def daily_ic(m, dz, cols):
    t=dz.copy(); t["p"]=m.predict(dz[cols].values)
    dii=t.index.get_level_values("Date"); ics=[]
    for d,g in t.groupby(dii):
        if len(g)>=5 and g["p"].std()>1e-12:
            i,_=spearmanr(g["p"],g["future_ret_5d"])
            if not np.isnan(i): ics.append(i)
    return np.mean(ics), (np.array(ics)>0).mean()

print("="*60)
print("增量验证: 加ht_phase试试")
print("="*60)
configs = [
    ("12因子(现有)", FACTOR12),
    ("13因子(+ht_phase)", FACTOR12 + ["ht_phase"]),
    ("14因子(+MFI+OBV)", FACTOR12 + ["mfi","obv_chg"]),
    ("15因子(全加)", FACTOR12 + ["mfi","obv_chg","ht_phase"]),
]
for name, cols in configs:
    sub = feat.dropna(subset=cols + ["future_ret_5d"])
    dl2 = sub.index.get_level_values("Date")
    def sl2(s,e): return sub[(dl2>=pd.Timestamp(s))&(dl2<=pd.Timestamp(e))]
    def dt2(df,k):
        u=df.index.get_level_values("Date").unique().sort_values()
        return df[df.index.get_level_values("Date")<=u[-(k+1)]] if len(u)>k else df
    tr = dt2(sl2("2010-01-01","2018-12-31"),EMBARGO)
    va = dt2(sl2("2019-01-01","2020-12-31"),EMBARGO)
    te = sl2("2021-01-01","2025-12-31")
    tv = pd.concat([tr, va])
    m = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.01,
                         random_state=42, n_jobs=-1).fit(tv[cols].values, tv["future_ret_5d"].values)
    ic_m, ic_p = daily_ic(m, te, cols)
    print(f"{name:<20}: IC={ic_m:+.4f}, IC>0={ic_p:.3f}")

print("\n判断: 看哪个配置IC明显超过12因子的基准")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import pickle
import time

tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()

# ========== 步骤1: 对每只股票,拿历史股价+还原原始价 + 历史股数 ==========
mktcap_dict = {}
failed = []

for idx, tk in enumerate(tickers84):
    try:
        t = yf.Ticker(tk)
        # 拿历史股价(含拆股信息),范围2020中到2025(测试期+缓冲)
        hist = t.history(start="2020-07-01", end="2025-12-31", auto_adjust=False, actions=True)
        if len(hist) == 0:
            failed.append(tk); continue
        # 还原未复权原始价 = Close × 该日之后的累计拆股比例
        splits = hist["Stock Splits"].replace(0, 1)
        cumsplit = splits[::-1].cumprod()[::-1].shift(-1).fillna(1)
        hist["raw_price"] = hist["Close"] * cumsplit
        # 去时区
        hist.index = pd.to_datetime(hist.index).tz_localize(None).normalize()

        # 拿历史股数
        shares = t.get_shares_full(start="2020-01-01", end="2025-12-31")
        if shares is None or len(shares) == 0:
            failed.append(tk); continue
        shares.index = pd.to_datetime(shares.index).tz_localize(None).normalize()
        shares = shares[~shares.index.duplicated(keep="last")]

        # 把股数对齐到股价的每个交易日(前向填充:某时点股数持续到下次更新)
        shares_aligned = shares.reindex(hist.index, method="ffill")
        # 开头可能有NaN(第一个股数时点之前),用第一个已知股数向前填充
        shares_aligned = shares_aligned.bfill()

        # 市值 = 真实股数 × 真实原始价 (口径一致!)
        mktcap = shares_aligned * hist["raw_price"]
        mktcap_dict[tk] = mktcap

        if (idx+1) % 20 == 0:
            print(f"进度 {idx+1}/{len(tickers84)}")
        time.sleep(0.3)
    except Exception as e:
        failed.append(tk)
        print(f"  {tk} 失败: {type(e).__name__}: {e}")

print(f"\n成功: {len(mktcap_dict)}/{len(tickers84)}只, 失败: {failed}")

# ========== 步骤2: 合并成市值矩阵 ==========
mktcap_df = pd.DataFrame(mktcap_dict)
mktcap_df = mktcap_df.dropna(how="all")
mktcap_df.to_pickle("market_cap.pkl")
print(f"\n市值矩阵: {mktcap_df.shape}")

# 验证:苹果2020和2021的市值(看是否合理)
print("\n苹果市值验证(万亿美元):")
for d in ["2020-08-03", "2021-01-04", "2023-01-03", "2025-01-02"]:
    try:
        v = mktcap_df["AAPL"].asof(pd.Timestamp(d))
        print(f"  {d}: {v/1e12:.2f}万亿")
    except: pass

# ========== 步骤3: 算基准权重(每天,市值占比) ==========
w_benchmark = mktcap_df.div(mktcap_df.sum(axis=1), axis=0)
w_benchmark.to_pickle("benchmark_weights.pkl")
print(f"\n✓ market_cap.pkl 和 benchmark_weights.pkl")
print(f"基准权重矩阵: {w_benchmark.shape}")
print(f"权重和验证(应全≈1): {w_benchmark.sum(axis=1).dropna().round(3).unique()[:3]}")

# 看2021年初的基准权重前10大
print("\n2021-01-04 基准权重前10大:")
w0 = w_benchmark.loc[w_benchmark.index >= "2021-01-01"].iloc[0]
print((w0.sort_values(ascending=False).head(10)*100).round(2).astype(str) + "%")

In [ ]:
import pandas as pd
import requests
from io import StringIO

# ========== 1. 读维基标普500的GICS行业 ==========
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
resp = requests.get(url, headers=headers)
sp500 = pd.read_html(StringIO(resp.text))[0]

# 建立 代码→GICS行业 映射
wiki_sector = sp500.set_index("Symbol")["GICS Sector"].to_dict()

# ========== 2. 映射到你的84只股票 ==========
tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()

sector_map = {}
unmatched = []
for tk in tickers84:
    if tk in wiki_sector:
        sector_map[tk] = wiki_sector[tk]
    else:
        unmatched.append(tk)

print(f"匹配上: {len(sector_map)}/{len(tickers84)}只")
print(f"没匹配上: {unmatched}")

# ========== 3. 行业分布 ==========
sector_series = pd.Series(sector_map, name="GICS_Sector")
sector_series.index.name = "Ticker"
print(f"\n行业分布:")
print(sector_series.value_counts())

# ========== 4. 保存 ==========
sector_series.to_frame().to_pickle("sector_classification.pkl")
print(f"\n✓ sector_classification.pkl ({len(sector_series)}只)")
print("\n各股行业(前15只):")
print(sector_series.head(15))

In [ ]:
import pandas as pd

# 读行业分类
sector = pd.read_pickle("sector_classification.pkl")["GICS_Sector"]

# 构建行业哑变量矩阵(one-hot): 每只股票 × 11个行业,所属行业=1
industry_dummies = pd.get_dummies(sector).astype(int)
industry_dummies.index.name = "Ticker"

print(f"行业哑变量矩阵: {industry_dummies.shape}")
print(f"(84只股票 × {industry_dummies.shape[1]}个行业)")
print(f"\n矩阵样例(前8只):")
print(industry_dummies.head(8))

# 验证:每只股票只属于一个行业(每行和=1)
print(f"\n每行和(应全=1): {industry_dummies.sum(axis=1).unique()}")
# 每个行业的股票数(每列和)
print(f"\n各行业股票数(列和):")
print(industry_dummies.sum(axis=0).sort_values(ascending=False))

industry_dummies.to_pickle("industry_dummies.pkl")
print(f"\n✓ industry_dummies.pkl")

In [ ]:
import numpy as np
import pandas as pd
import pickle

# ========== 读数据 ==========
mktcap = pd.read_pickle("market_cap.pkl")                    # 市值(任务1)
adj_close = pd.read_csv("stock_data_clean_adjusted.csv", header=[0,1], index_col=0, parse_dates=True).xs("Close", axis=1, level=1)  # 复权价(算动量)
daily_ret = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)  # 日收益(算波动/Beta)
spx = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True).iloc[:, 0]      # 标普收益(算Beta)
fund = pd.read_pickle("fundamental_raw2.pkl")                # 基本面(净利润+ROE,算价值)
fund["NOTICE_DATE"] = pd.to_datetime(fund["NOTICE_DATE"])

tickers84 = pd.read_csv("fundamental_valid_tickers.csv")["Ticker"].tolist()
tickers84 = [t for t in tickers84 if t in mktcap.columns]

# 统一到复权价的交易日索引
common_dates = adj_close.index

# ========== 横截面去极值+标准化函数 ==========
def winsor_std_cross(df):
    result = df.copy().astype(float)
    for date in df.index:
        row = df.loc[date].dropna()
        if len(row) < 3:
            result.loc[date, :] = np.nan; continue
        med = row.median()
        mad = (row - med).abs().median() * 1.4826
        row_clip = row.clip(med - 5*mad, med + 5*mad) if mad > 1e-12 else row
        mean, std = row_clip.mean(), row_clip.std()
        result.loc[date, :] = np.nan
        result.loc[date, row.index] = (row_clip - mean)/std if std > 1e-12 else 0.0
    return result

# ========== 因子1: 规模 = log(市值) ==========
print("1. 规模因子...")
size_raw = np.log(mktcap[tickers84])

# ========== 因子2: 动量 = 60日收益 ==========
print("2. 动量因子...")
mom_raw = adj_close[tickers84] / adj_close[tickers84].shift(60) - 1

# ========== 因子3: 波动率 = 20日收益标准差 ==========
print("3. 波动率因子...")
vol_raw = daily_ret[tickers84].rolling(20).std()

# ========== 因子4: Beta = 60日滚动回归斜率 ==========
print("4. Beta因子(滚动回归)...")
beta_raw = pd.DataFrame(index=daily_ret.index, columns=tickers84, dtype=float)
for tk in tickers84:
    cov = daily_ret[tk].rolling(60).cov(spx)
    var = spx.rolling(60).var()
    beta_raw[tk] = cov / var

# ========== 因子5: 价值 = 1/(P/B), P/B=股价/每股净资产 ==========
print("5. 价值因子(反推净资产)...")
# 反推每股净资产:净资产=净利润/ROE, 每股净资产=净资产/股数
# 股数用之前反推(净利润/EPS)
fund_calc = fund.copy()
fund_calc["ROE_dec"] = fund_calc["ROE_AVG"] / 100.0  # ROE转小数(如果是百分数)
fund_calc["equity"] = fund_calc["PARENT_HOLDER_NETPROFIT"] / fund_calc["ROE_dec"]  # 净资产
fund_calc["shares"] = fund_calc["PARENT_HOLDER_NETPROFIT"] / fund_calc["BASIC_EPS"]  # 股数
fund_calc["bps"] = fund_calc["equity"] / fund_calc["shares"]  # 每股净资产
# 剔除异常(ROE<=0或EPS<=0导致的负值/异常)
fund_calc.loc[(fund_calc["ROE_dec"]<=0)|(fund_calc["BASIC_EPS"]<=0)|(fund_calc["bps"]<=0), "bps"] = np.nan

# 把每股净资产按公告日对齐到每个交易日(防前视,季度阶梯)
bps_daily = pd.DataFrame(index=common_dates, columns=tickers84, dtype=float)
for tk in tickers84:
    sub = fund_calc[fund_calc["Ticker"]==tk].dropna(subset=["bps"]).sort_values("NOTICE_DATE")
    for _, row in sub.iterrows():
        bps_daily.loc[bps_daily.index >= row["NOTICE_DATE"], tk] = row["bps"]

# P/B = 股价(未复权真实价) / 每股净资产, 价值=1/PB
# 注意:PB用真实股价(stock_data.csv)更合理,但简化用复权价影响相对排名有限
raw_close = pd.read_csv("stock_data.csv", header=[0,1], index_col=0, parse_dates=True).xs("Close", axis=1, level=1)
raw_close.index = pd.to_datetime(raw_close.index).normalize()
bps_daily.index = pd.to_datetime(bps_daily.index).normalize()
pb = raw_close[tickers84].reindex(bps_daily.index) / bps_daily
value_raw = 1.0 / pb  # B/P
value_raw = value_raw.replace([np.inf, -np.inf], np.nan)

# ========== 全部标准化 ==========
print("\n标准化5个因子...")
size_factor = winsor_std_cross(size_raw)
mom_factor = winsor_std_cross(mom_raw)
vol_factor = winsor_std_cross(vol_raw)
beta_factor = winsor_std_cross(beta_raw)
value_factor = winsor_std_cross(value_raw)

# 对齐到共同日期,保留测试期+缓冲(2020-10起)
def clip_period(df):
    df.index = pd.to_datetime(df.index).normalize()
    return df.loc[df.index >= "2020-10-01"]

styles = {
    "size": clip_period(size_factor),
    "momentum": clip_period(mom_factor),
    "volatility": clip_period(vol_factor),
    "beta": clip_period(beta_factor),
    "value": clip_period(value_factor),
}

with open("style_factors.pkl", "wb") as f:
    pickle.dump(styles, f)

print("\n✓ style_factors.pkl (5个风格因子)")
for name, df in styles.items():
    valid = df.dropna(how="all")
    print(f"  {name}: {valid.shape}, 均值≈{df.mean(axis=1).mean():.3f}, 标准差≈{df.std(axis=1).mean():.3f}")

# 验证:2021年初各因子样例
print("\n2021-01初 各风格因子(苹果/微软/某小盘):")
d0 = styles["size"].loc[styles["size"].index >= "2021-01-01"].index[0]
for name, df in styles.items():
    row = df.loc[d0]
    print(f"  {name}: AAPL={row.get('AAPL', np.nan):.2f}, MSFT={row.get('MSFT', np.nan):.2f}")

In [ ]:
import pickle
import pandas as pd

styles = pickle.load(open("style_factors.pkl","rb"))
print("各因子在测试期(2021-01-01后)的有效行数:")
for name, df in styles.items():
    test_period = df.loc[df.index >= "2021-01-01"]
    valid_days = test_period.dropna(how="all").shape[0]
    print(f"  {name}: 测试期{valid_days}天有效")

In [ ]:
import pandas as pd
import pickle

# 找第二阶段的组合权重文件
candidates = ["portfolio_opt_v4.pkl", "portfolio_weights.pkl", "final_predictions.pkl", 
              "portfolio_opt.pkl", "weights.pkl"]
for fname in candidates:
    try:
        obj = pickle.load(open(fname, "rb"))
        print(f"✓ {fname}: 类型={type(obj).__name__}")
        if isinstance(obj, dict):
            print(f"   keys: {list(obj.keys())}")
            # 看看每个key的内容类型
            for k, v in list(obj.items())[:5]:
                print(f"     {k}: {type(v).__name__}, {getattr(v, 'shape', '')}")
        elif hasattr(obj, 'shape'):
            print(f"   形状: {obj.shape}, 列: {list(obj.columns)[:8] if hasattr(obj,'columns') else ''}")
    except FileNotFoundError:
        print(f"✗ 没有 {fname}")
    except Exception as e:
        print(f"? {fname}: {type(e).__name__}")

In [ ]:
import pickle
import pandas as pd

obj = pickle.load(open("portfolio_opt_v4.pkl", "rb"))
results = obj["results"]

print("results 的 keys:", list(results.keys()))
print()
# 逐个看results里每个key的结构
for k, v in results.items():
    print(f"=== {k} ===")
    print(f"  类型: {type(v).__name__}")
    if isinstance(v, dict):
        print(f"  子keys: {list(v.keys())}")
        for k2, v2 in list(v.items())[:6]:
            print(f"    {k2}: {type(v2).__name__}, shape={getattr(v2, 'shape', 'N/A')}")
    elif hasattr(v, 'shape'):
        print(f"  形状: {v.shape}")
        if hasattr(v, 'columns'):
            print(f"  列: {list(v.columns)[:10]}")
        if hasattr(v, 'index'):
            print(f"  索引前3: {list(v.index[:3])}")
    print()

In [ ]:
import pickle
import pandas as pd

obj = pickle.load(open("portfolio_opt_v4.pkl", "rb"))
r5 = obj["results"]["每5天"]

ms_w = r5["ms_w"]
rebal_dates = r5["rebal_dates"]

print(f"调仓次数: {len(ms_w)}")
print(f"调仓日期数: {len(rebal_dates)}")
print(f"前3个调仓日: {rebal_dates[:3]}")
print()

# 看第一个调仓日的权重
w0 = ms_w[0]
print(f"第一次调仓权重(ms_w[0]):")
print(f"  类型: {type(w0).__name__}")
if hasattr(w0, 'index'):
    print(f"  持仓股票数: {len(w0)}")
    print(f"  权重和: {w0.sum():.4f}")
    print(f"  前几只:")
    print(w0.sort_values(ascending=False).head(8))
elif isinstance(w0, dict):
    print(f"  持仓数: {len(w0)}")
    print(f"  权重和: {sum(w0.values()):.4f}")
    print(f"  样例: {list(w0.items())[:5]}")

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','PingFang SC','Heiti TC','SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========== 读数据 ==========
obj = pickle.load(open("portfolio_opt_v4.pkl", "rb"))
r5 = obj["results"]["每5天"]
ms_w_list = r5["ms_w"]          # 最大夏普,每个调仓日权重
rp_w_list = r5["rp_w"]          # 风险平价
rebal_dates = r5["rebal_dates"] # 调仓日期

benchmark_weights = pd.read_pickle("benchmark_weights.pkl")
industry_dummies = pd.read_pickle("industry_dummies.pkl")
styles = pickle.load(open("style_factors.pkl", "rb"))

# 统一日期格式(去时区)
benchmark_weights.index = pd.to_datetime(benchmark_weights.index).tz_localize(None).normalize()
for k in styles:
    styles[k].index = pd.to_datetime(styles[k].index).tz_localize(None).normalize()

tickers84 = list(industry_dummies.index)

# ========== 计算每个调仓日的偏离,然后取平均 ==========
industry_names = list(industry_dummies.columns)
style_names = list(styles.keys())

ind_active_list = []   # 每个调仓日的行业偏离
style_active_list = [] # 每个调仓日的风格偏离

for i, rd in enumerate(rebal_dates):
    rd_norm = pd.Timestamp(rd).tz_localize(None).normalize()
    # 组合权重(最大夏普)
    w_port = ms_w_list[i].copy()
    w_port.index = [str(x) for x in w_port.index]  # 确保ticker是字符串
    w_port = pd.Series(w_port.values, index=[t.split()[0] if ' ' in str(t) else t for t in w_port.index])
    w_port = w_port.reindex(tickers84).fillna(0)

    # 基准权重(取该调仓日,找最近的可用日期)
    if rd_norm in benchmark_weights.index:
        w_bench = benchmark_weights.loc[rd_norm, tickers84].fillna(0)
    else:
        # 找最近的前一个交易日
        avail = benchmark_weights.index[benchmark_weights.index <= rd_norm]
        if len(avail) == 0: continue
        w_bench = benchmark_weights.loc[avail[-1], tickers84].fillna(0)
    w_bench = w_bench / w_bench.sum()  # 重新归一化

    # 行业偏离 = 组合行业权重 - 基准行业权重
    port_ind = industry_dummies.mul(w_port, axis=0).sum()
    bench_ind = industry_dummies.mul(w_bench, axis=0).sum()
    ind_active_list.append(port_ind - bench_ind)

    # 风格偏离 = 组合加权风格值 - 基准加权风格值
    style_dev = {}
    for sname in style_names:
        if rd_norm in styles[sname].index:
            sf = styles[sname].loc[rd_norm, tickers84]
        else:
            avail = styles[sname].index[styles[sname].index <= rd_norm]
            if len(avail) == 0: 
                style_dev[sname] = np.nan; continue
            sf = styles[sname].loc[avail[-1], tickers84]
        sf = sf.reindex(tickers84)
        # 加权平均(只用两者都有值的股票)
        valid = sf.notna()
        port_exp = (w_port[valid] * sf[valid]).sum()
        bench_exp = (w_bench[valid] * sf[valid]).sum()
        style_dev[sname] = port_exp - bench_exp
    style_active_list.append(pd.Series(style_dev))

# 取测试期平均偏离
ind_active_avg = pd.DataFrame(ind_active_list).mean()
style_active_avg = pd.DataFrame(style_active_list).mean()

print("="*60)
print("第二阶段组合(最大夏普) 相对基准的平均主动暴露")
print("="*60)
print("\n行业偏离(正=超配,负=低配):")
print((ind_active_avg.sort_values(ascending=False)*100).round(2).astype(str) + "%")
print("\n风格偏离(标准差单位):")
print(style_active_avg.round(3))

# ========== 画图 ==========
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 行业偏离条形图
ind_sorted = ind_active_avg.sort_values()
colors1 = ['#C0504D' if x<0 else '#4472C4' for x in ind_sorted.values]
axes[0].barh(range(len(ind_sorted)), ind_sorted.values*100, color=colors1)
axes[0].set_yticks(range(len(ind_sorted)))
axes[0].set_yticklabels(ind_sorted.index, fontsize=10)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('主动偏离 (%)')
axes[0].set_title('行业主动暴露 (组合权重 - 基准权重)', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# 风格偏离条形图
style_sorted = style_active_avg.sort_values()
colors2 = ['#C0504D' if x<0 else '#4472C4' for x in style_sorted.values]
axes[1].barh(range(len(style_sorted)), style_sorted.values, color=colors2)
axes[1].set_yticks(range(len(style_sorted)))
axes[1].set_yticklabels(['规模' if s=='size' else '价值' if s=='value' else '动量' if s=='momentum' else '波动率' if s=='volatility' else 'Beta' for s in style_sorted.index], fontsize=11)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('主动暴露 (标准差)')
axes[1].set_title('风格主动暴露 (组合 - 基准)', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('active_exposure.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n✓ 图已保存 active_exposure.png")

In [ ]:
import numpy as np
import pandas as pd
import cvxpy as cp
import pickle, time
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','PingFang SC','Heiti TC','SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ================= 原始数据 =================
DR   = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
SPX  = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True).iloc[:, 0]
BW   = pd.read_pickle("benchmark_weights.pkl")
IND  = pd.read_pickle("industry_dummies.pkl")
STY  = pickle.load(open("style_factors.pkl", "rb"))
PRED = pd.read_pickle("final_predictions.pkl")

for _x in (DR, BW): _x.index = pd.to_datetime(_x.index).tz_localize(None).normalize()
SPX.index = pd.to_datetime(SPX.index).tz_localize(None).normalize()
for _k in STY: STY[_k].index = pd.to_datetime(STY[_k].index).tz_localize(None).normalize()
PRED.index = PRED.index.set_levels(
    pd.to_datetime(PRED.index.levels[0]).tz_localize(None).normalize(), level=0)

TICKERS   = [t for t in IND.index if t in DR.columns]
IND_MAT   = IND.loc[TICKERS].values.astype(float)
STY_NAMES = list(STY.keys())
COST_BPS  = 5
TEST_IDX  = DR.loc[(DR.index >= "2021-01-01") & (DR.index <= "2025-12-31")].index

def asof(df, d, cols):
    idx = df.index[df.index <= d]
    return df.loc[idx[-1], cols] if len(idx) else pd.Series(np.nan, index=cols)

# ================= 基准构造(唯一入口,基准权重滞后一天) =================
def build_benchmarks(idx):
    R = DR.loc[idx, TICKERS].fillna(0)
    W = BW.reindex(idx).ffill()[TICKERS].fillna(0)
    W = W.div(W.sum(axis=1), axis=0).shift(1)
    mcap = (W * R).sum(axis=1)
    out = pd.DataFrame({"市值加权84": mcap, "等权84": R.mean(axis=1),
                        "真标普500": SPX.reindex(idx)})
    out.iloc[0] = np.nan          # ★三个基准统一从第2天起(首日无滞后权重)
    return out

BENCH = build_benchmarks(TEST_IDX)

# ================= 评价函数(唯一入口) =================
def perf(r, bench=None, name=""):
    r = r.dropna()
    ann = (1+r).prod()**(252/len(r)) - 1
    vol = r.std()*np.sqrt(252)
    nav = (1+r).cumprod(); mdd = (nav/nav.cummax()-1).min()
    mon = (1+r).resample("ME").prod() - 1
    out = {"策略": name, "年化收益%": ann*100, "年化波动%": vol*100,
           "夏普": ann/vol if vol > 0 else np.nan,
           "最大回撤%": mdd*100, "月度胜率%": (mon > 0).mean()*100}
    if bench is not None:
        b  = bench.reindex(r.index).dropna()
        rr = r.reindex(b.index); ex = rr - b
        te = ex.std()*np.sqrt(252)
        ann_b = (1+b).prod()**(252/len(b)) - 1
        ann_r = (1+rr).prod()**(252/len(rr)) - 1
        out.update({"年化超额%": (ann_r-ann_b)*100, "跟踪误差%": te*100,
                    "IR": (ann_r-ann_b)/te if te > 0 else np.nan})
    return out

# ================= 由权重回测(唯一入口) =================
def port_returns(wdict, cost_bps=COST_BPS):
    rl = sorted(wdict.keys())
    rets, dates, tos, prev = [], [], [], None
    for i, rd in enumerate(rl):
        w = wdict[rd].groupby(level=0).sum()
        cols = [t for t in w.index if t in DR.columns]
        w = w[cols] / w[cols].sum()
        end  = rl[i+1] if i+1 < len(rl) else TEST_IDX[-1]
        hold = TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]
        cost = 0.0
        if prev is not None:
            a  = w.index.union(prev.index)
            tv = float(np.abs(w.reindex(a).fillna(0) - prev.reindex(a).fillna(0)).sum())
            tos.append(tv); cost = tv*cost_bps/10000
        for j, d in enumerate(hold):
            rets.append(float((w * DR.loc[d, cols].fillna(0)).sum()) - (cost if j == 0 else 0))
            dates.append(d)
        prev = w
    return pd.Series(rets, index=dates), (float(np.mean(tos)) if tos else np.nan)

print(f"股票池 {len(TICKERS)} 只 | 测试期 {TEST_IDX[0].date()} ~ {TEST_IDX[-1].date()} ({len(TEST_IDX)}天)")
print("\n三个基准年化收益(基准权重已滞后一天,无前视):")
print(pd.DataFrame([perf(BENCH[c], None, c) for c in BENCH.columns]).round(3).to_string(index=False))

In [ ]:
# ================= 预计算各调仓日输入(与λ无关,只跑一次) =================
t0 = time.time()
REBAL = list(TEST_IDX[::5])
CACHE = []

for i, rd in enumerate(REBAL):
    try:
        a = PRED.xs(rd, level=0)["pred_xgb"].reindex(TICKERS)
    except KeyError:
        continue
    if a.notna().sum() < 40: continue
    alpha = a.fillna(a.median()).values

    wb = asof(BW, rd, TICKERS).fillna(0)
    if wb.sum() == 0: continue
    wb = (wb / wb.sum()).values

    sty = np.column_stack([asof(STY[s], rd, TICKERS).reindex(TICKERS).fillna(0).values
                           for s in STY_NAMES])

    hist = DR[TICKERS].loc[DR.index <= rd].tail(126).dropna(axis=1, how="any")
    if hist.shape[1] < 50: continue
    lw = LedoitWolf().fit(hist.values)
    S  = pd.DataFrame(lw.covariance_, index=hist.columns, columns=hist.columns) \
           .reindex(index=TICKERS, columns=TICKERS).values.copy()
    S[np.isnan(S)] = 0.0
    d = np.diag(S).copy(); d[d <= 0] = 4e-4; np.fill_diagonal(S, d)

    end  = REBAL[i+1] if i+1 < len(REBAL) else TEST_IDX[-1]
    hold = TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]
    if len(hold) == 0: continue

    CACHE.append(dict(rd=rd, alpha=alpha, wb=wb, Sigma=S, style=sty, hold=hold,
                      cap=np.maximum(0.05, wb + 0.02)))

print(f"✓ 缓存 {len(CACHE)} 个调仓日, 耗时 {time.time()-t0:.0f} 秒")

In [ ]:
# ================= 有效前沿(扫描λ) =================
def solve_enhance(c, lam):
    """指数增强优化: max α'w − λ·(w−w_b)'Σ(w−w_b), 五个约束"""
    n = len(TICKERS)
    w = cp.Variable(n); act = w - c["wb"]
    prob = cp.Problem(
        cp.Maximize(c["alpha"] @ w - lam * cp.quad_form(act, cp.psd_wrap(c["Sigma"]))),
        [cp.sum(w) == 1, w >= 0,                                  # ①全额投资+不做空
         act <= 0.02, act >= -0.02,                               # ②个股偏离≤2%
         IND_MAT.T @ act <= 0.05, IND_MAT.T @ act >= -0.05,       # ③行业偏离≤5%
         c["style"].T @ act <= 0.5, c["style"].T @ act >= -0.5,   # ④风格偏离≤0.5
         w <= c["cap"]])                                          # ⑤权重上限max(5%,w_b+2%)
    try:
        prob.solve()
        if prob.status == "optimal":
            v = np.clip(w.value, 0, None); return v / v.sum()
    except Exception:
        pass
    return None

def run_lambda(lam):
    """给定λ,解出全部调仓日权重 + 事前跟踪误差"""
    wdict, exante, nfail = {}, [], 0
    for c in CACHE:
        w = solve_enhance(c, lam)
        if w is None: nfail += 1; continue
        act = w - c["wb"]
        exante.append(float(np.sqrt(max(act @ c["Sigma"] @ act, 0)) * np.sqrt(252)))
        wdict[c["rd"]] = pd.Series(w, index=TICKERS)
    return wdict, float(np.mean(exante)), nfail

LAMBDAS = [0.5, 1, 2, 5, 10, 20, 50, 100, 300]
rows = {}
print(f"扫描 {len(LAMBDAS)} 个λ ...")
for lam in LAMBDAS:
    t1 = time.time()
    wdict, ex_te, nfail = run_lambda(lam)
    r, to = port_returns(wdict, COST_BPS)
    m_mc = perf(r, BENCH["市值加权84"]); m_sp = perf(r, BENCH["真标普500"])
    rows[lam] = {"年化收益%": m_mc["年化收益%"], "事前跟踪误差%": ex_te*100,
                 "超额_vs市值%": m_mc["年化超额%"], "跟踪误差_vs市值%": m_mc["跟踪误差%"],
                 "IR_vs市值": m_mc["IR"],
                 "超额_vs标普%": m_sp["年化超额%"], "跟踪误差_vs标普%": m_sp["跟踪误差%"],
                 "IR_vs标普": m_sp["IR"], "平均换手": to, "优化失败": nfail}
    print(f"  λ={lam:<5} 事前TE {ex_te*100:5.2f}%  实现TE {m_mc['跟踪误差%']:5.2f}%  "
          f"超额(vs市值) {m_mc['年化超额%']:+6.2f}%  超额(vs标普) {m_sp['年化超额%']:+6.2f}%  "
          f"换手 {to:.3f}  [{time.time()-t1:.0f}s]")

FRONTIER = pd.DataFrame(rows).T; FRONTIER.index.name = "λ"
print("\n" + "="*118); print("有效前沿:λ 扫描结果"); print("="*118)
print(FRONTIER.round(3).to_string())

# ---- 选λ:在事前TE落入3-5%目标区间的候选中,取λ最大者(风险节约原则) ----
TE_LO, TE_HI = 3.0, 5.0
cand = FRONTIER[(FRONTIER["事前跟踪误差%"] >= TE_LO) & (FRONTIER["事前跟踪误差%"] <= TE_HI)]
LAM_STAR = float(cand.index.max()) if len(cand) else 10.0

print(f"\n【选参:风险节约原则】")
print(f"  目标跟踪误差区间 {TE_LO}-{TE_HI}%,事前TE落入区间的候选λ: {list(cand.index)}")
print(f"  规则:在满足下限{TE_LO}%的候选中取λ最大者 → 相同风险预算下占用最少")
print(f"  选定 λ* = {LAM_STAR:g}  (事前TE {FRONTIER.loc[LAM_STAR,'事前跟踪误差%']:.2f}%, "
      f"实现TE {FRONTIER.loc[LAM_STAR,'跟踪误差_vs市值%']:.2f}%)")
print(f"  选参仅依据事前跟踪误差(先验风险预算),未使用测试集实现收益 → 无参数过拟合")

print(f"\n  候选λ对比:")
print(cand[["事前跟踪误差%","跟踪误差_vs市值%","超额_vs市值%","IR_vs市值"]].round(3).to_string())

# 事前 vs 实现的系统性偏差(支持留缓冲,也是第五步协方差失效的伏笔)
gap = (FRONTIER["跟踪误差_vs市值%"] - FRONTIER["事前跟踪误差%"])
print(f"\n【协方差诊断】实现跟踪误差系统性高于事前: 均值 +{gap.mean():.2f}pp "
      f"(区间 +{gap.min():.2f} ~ +{gap.max():.2f}pp)")
print(f"  → Ledoit-Wolf协方差低估实际风险约{gap.mean()/FRONTIER['事前跟踪误差%'].mean()*100:.0f}%,"
      f"故事前目标需留缓冲以确保实现值不越上限")
# ---- 画前沿 ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
for ax, (bmk, ex, te) in zip(axes, [("84只市值加权基准","超额_vs市值%","跟踪误差_vs市值%"),
                                    ("真标普500(^GSPC)","超额_vs标普%","跟踪误差_vs标普%")]):
    ax.plot(FRONTIER[te], FRONTIER[ex], "o-", lw=1.8, color="#4472C4", ms=7)
    for lam in FRONTIER.index:
        ax.annotate(f"λ={lam:g}", (FRONTIER.loc[lam,te], FRONTIER.loc[lam,ex]),
                    textcoords="offset points", xytext=(7,5), fontsize=9)
    ax.scatter([FRONTIER.loc[LAM_STAR,te]], [FRONTIER.loc[LAM_STAR,ex]],
               s=190, facecolors="none", edgecolors="#C0504D", lw=2.2,
               label=f"选定 λ*={LAM_STAR:g}", zorder=5)
    ax.axhline(0, ls="--", c="gray", lw=0.9)
    ax.axvspan(TE_LO, TE_HI, alpha=0.12, color="green", label="目标跟踪误差 3–5%")
    ax.set_xlabel("跟踪误差 (年化 %)"); ax.set_ylabel("年化超额收益 (%)")
    ax.set_title(f"有效前沿:相对{bmk}", fontsize=12, fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("efficient_frontier.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# ================= 用选定的λ*做最终回测 + 约束核查 =================
W_ENH, EXANTE_TE, NFAIL = run_lambda(LAM_STAR)
R_ENH, TO_ENH = port_returns(W_ENH, COST_BPS)
pickle.dump({"weights": W_ENH, "rebal_dates": sorted(W_ENH.keys()),
             "lam": LAM_STAR, "exante_te": EXANTE_TE, "cost_bps": COST_BPS},
            open("enhance_weights.pkl", "wb"))

rows = [perf(R_ENH, BENCH[c], f"指数增强(λ={LAM_STAR:g}) vs {c}") for c in BENCH.columns]
rows += [perf(BENCH[c], None, f"基准:{c}") for c in BENCH.columns]
print("="*126); print(f"第二步 指数增强 最终结果  (λ*={LAM_STAR:g}, 成本 turnover×{COST_BPS}bps)"); print("="*126)
print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---- 跟踪误差监控:与3-5%目标区间对比 ----
te_real = perf(R_ENH, BENCH["市值加权84"])["跟踪误差%"]
print(f"\n【跟踪误差监控】事前 {EXANTE_TE*100:.2f}%  |  实现 {te_real:.2f}%  |  目标区间 {TE_LO}-{TE_HI}%")
print(f"  实现跟踪误差{'✓落在' if TE_LO <= te_real <= TE_HI else '✗未落在'}目标区间内")
print(f"  事前-实现差距 {abs(EXANTE_TE*100-te_real):.2f}pp → "
      f"{'协方差估计合理' if abs(EXANTE_TE*100-te_real) < 1 else '协方差低估了实际风险'}")

# ---- 五个约束是否真正生效 ----
dev_i, dev_ind, dev_sty, nh, cap_bind = [], [], [], [], []
for c in CACHE:
    if c["rd"] not in W_ENH: continue
    w = W_ENH[c["rd"]].values; act = w - c["wb"]
    dev_i.append(np.abs(act).max())
    dev_ind.append(np.abs(IND_MAT.T @ act).max())
    dev_sty.append(np.abs(c["style"].T @ act).max())
    nh.append(int((w > 1e-4).sum()))
    cap_bind.append(int((w >= c["cap"] - 1e-6).sum()))
print(f"\n【约束核查】(250个调仓日均值 / 最大值)")
print(f"  ②个股偏离   均值{np.mean(dev_i):.4f}  最大{np.max(dev_i):.4f}   约束0.0200")
print(f"  ③行业偏离   均值{np.mean(dev_ind):.4f}  最大{np.max(dev_ind):.4f}   约束0.0500")
print(f"  ④风格偏离   均值{np.mean(dev_sty):.4f}  最大{np.max(dev_sty):.4f}   约束0.5000")
print(f"  ⑤权重触顶   平均{np.mean(cap_bind):.1f}只/期 (上限=max(5%, w_b+2%))")
print(f"  持仓只数     平均{np.mean(nh):.1f}只  (对比第二阶段约16只)")
print(f"  优化失败     {NFAIL}个调仓日")

# ---- 净值图 ----
plt.figure(figsize=(13, 6))
for s, lab, c, lw in [(BENCH["真标普500"], "真标普500", "gray", 1.4),
                      (BENCH["等权84"], "84只等权基准", "#7F7F7F", 1.2),
                      (BENCH["市值加权84"], "84只市值加权基准", "#4472C4", 1.6),
                      (R_ENH, f"指数增强(λ={LAM_STAR:g},扣费)", "#C0504D", 2.1)]:
    ss = s.reindex(R_ENH.index).dropna(); nav = (1+ss).cumprod()
    plt.plot(nav.index, nav, label=f"{lab}  年化{perf(ss)['年化收益%']:.2f}%", color=c, lw=lw)
plt.title(f"指数增强策略 vs 三个基准 (λ*={LAM_STAR:g})", fontsize=13, fontweight="bold")
plt.ylabel("净值"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("index_enhance_nav.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# ================= 成本与换手,与第二阶段无约束组合对比 =================
_p2 = pickle.load(open("portfolio_opt_v4.pkl", "rb"))["results"]["每5天"]
W_P2 = {pd.Timestamp(rd).tz_localize(None).normalize(): w
        for rd, w in zip(_p2["rebal_dates"], _p2["ms_w"])}

R_ENH_G, _        = port_returns(W_ENH, 0)          # 指增 费前
R_P2_G,  TO_P2    = port_returns(W_P2,  0)          # 第二阶段 费前
R_P2_N,  _        = port_returns(W_P2,  COST_BPS)   # 第二阶段 费后

def ann(s):
    s = s.dropna(); return ((1+s).prod()**(252/len(s)) - 1)*100

n_reb = len(W_ENH) - 1
tab = pd.DataFrame([
    {"组合": f"指数增强(λ={LAM_STAR:g})", "平均单期换手": TO_ENH,
     "年化换手(×252/5)": TO_ENH*252/5, "费前年化%": ann(R_ENH_G), "费后年化%": ann(R_ENH),
     "成本侵蚀pp": ann(R_ENH_G)-ann(R_ENH), "平均持仓只数": np.mean(nh)},
    {"组合": "第二阶段最大夏普(无基准约束)", "平均单期换手": TO_P2,
     "年化换手(×252/5)": TO_P2*252/5, "费前年化%": ann(R_P2_G), "费后年化%": ann(R_P2_N),
     "成本侵蚀pp": ann(R_P2_G)-ann(R_P2_N), "平均持仓只数": np.mean([len(w) for w in W_P2.values()])},
])
print("="*120); print(f"成本与换手对比  (单向{COST_BPS}bps, 每5交易日调仓, {n_reb}次换仓)"); print("="*120)
print(tab.round(3).to_string(index=False))
print(f"\n换手率之比: 指增/第二阶段 = {TO_ENH/TO_P2:.2f}倍")
print(f"成本侵蚀之比: {(ann(R_ENH_G)-ann(R_ENH))/(ann(R_P2_G)-ann(R_P2_N)):.2f}倍")
print("\n注:指增持有全部84只中的大部分并贴基准微调;第二阶段仅持约16只、每期重排,两者换手结构不同")

In [ ]:
# ================= 归因诊断 =================
_W = BW.reindex(TEST_IDX).ffill()[TICKERS].fillna(0)
_W = _W.div(_W.sum(axis=1), axis=0).shift(1)        # ★基准权重滞后一天

contrib = pd.Series(0.0, index=TICKERS)
rl = sorted(W_ENH.keys())
for i, rd in enumerate(rl):
    w   = W_ENH[rd].reindex(TICKERS).fillna(0).values
    end = rl[i+1] if i+1 < len(rl) else TEST_IDX[-1]
    for d in TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]:
        wb_d = _W.loc[d].values
        if np.isnan(wb_d).any(): continue
        contrib += (w - wb_d) * DR.loc[d, TICKERS].fillna(0).values

cs = contrib.sort_values()
print("="*70); print("个股对超额收益的累计贡献  Σ(w_组合−w_基准)×收益"); print("="*70)
print("\n贡献最多的10只:"); print((cs.tail(10)[::-1]*100).round(2).to_string())
print("\n拖累最多的10只:"); print((cs.head(10)*100).round(2).to_string())
print(f"\n累计贡献合计 {contrib.sum()*100:.2f}% (算术裸加总,与年化几何超额口径不同)")

# ---- 关键时点的权重检视 ----
watch = [t for t in ["NVDA","AAPL","MSFT","AMZN","GOOGL","META","TSLA"] if t in TICKERS]
print("\n" + "="*70); print("科技巨头权重演变 (组合 vs 基准)"); print("="*70)
for target in ["2021-01-04","2023-01-03","2024-05-01","2025-01-02"]:
    av = [r for r in rl if r <= pd.Timestamp(target)]
    if not av: continue
    rd = av[-1]; w = W_ENH[rd]
    wb = asof(BW, rd, TICKERS).fillna(0); wb = wb/wb.sum()
    print(f"\n【{rd.date()}】" + "".join(f"{t:>9}" for t in watch))
    print("  组合% " + "".join(f"{w.get(t,0)*100:>9.2f}" for t in watch))
    print("  基准% " + "".join(f"{wb.get(t,0)*100:>9.2f}" for t in watch))
    print("  偏离% " + "".join(f"{(w.get(t,0)-wb.get(t,0))*100:>+9.2f}" for t in watch))

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','PingFang SC','Heiti TC','SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========== 读数据 ==========
obj = pickle.load(open("portfolio_opt_v4.pkl", "rb"))
r5 = obj["results"]["每5天"]
ms_w_list   = r5["ms_w"]          # 多头:第二阶段最大夏普组合
rebal_dates = r5["rebal_dates"]

daily_ret = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
spx       = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True).iloc[:, 0]
daily_ret.index = pd.to_datetime(daily_ret.index).tz_localize(None).normalize()
spx.index       = pd.to_datetime(spx.index).tz_localize(None).normalize()

COST_BPS = 5
rebal_norm = [pd.Timestamp(rd).tz_localize(None).normalize() for rd in rebal_dates]
all_dates = daily_ret.loc[(daily_ret.index >= rebal_norm[0]) & (daily_ret.index <= "2025-12-31")].index

# ========== 多头组合日收益(扣5bps成本) ==========
long_rets, dates_used = [], []
prev_w = None

for i, rd in enumerate(rebal_norm):
    w = ms_w_list[i].copy()
    w.index = [str(t) for t in w.index]
    w = w.groupby(level=0).sum()                      # 防重复
    end  = rebal_norm[i+1] if i+1 < len(rebal_norm) else all_dates[-1]
    hold = all_dates[(all_dates > rd) & (all_dates <= end)]
    if len(hold) == 0: continue

    # 换手成本
    cost = 0.0
    if prev_w is not None:
        allt = w.index.union(prev_w.index)
        cost = np.abs(w.reindex(allt).fillna(0) - prev_w.reindex(allt).fillna(0)).sum() * COST_BPS/10000

    for j, d in enumerate(hold):
        cols = [t for t in w.index if t in daily_ret.columns]
        r = daily_ret.loc[d, cols].fillna(0)
        pr = (w[cols] * r).sum()
        long_rets.append(pr - (cost if j == 0 else 0))
        dates_used.append(d)
    prev_w = w

long_rets = pd.Series(long_rets, index=dates_used)
bench     = spx.reindex(dates_used).fillna(0)

# ========== 任务1: 对冲 ==========
neutral = long_rets - bench          # 对冲后收益 ≈ 组合 - 基准

# ========== 任务2: Beta监控(60日滚动) ==========
W = 60
beta_before = long_rets.rolling(W).cov(bench) / bench.rolling(W).var()
beta_after  = neutral.rolling(W).cov(bench)   / bench.rolling(W).var()

# ========== 指标 ==========
def metrics(r):
    ann = (1+r).prod()**(252/len(r)) - 1
    vol = r.std()*np.sqrt(252)
    nav = (1+r).cumprod()
    mdd = (nav/nav.cummax()-1).min()
    monthly = (1+r).resample("ME").prod() - 1
    return ann*100, vol*100, (ann/vol if vol>0 else np.nan), mdd*100, (monthly>0).mean()*100

tbl = pd.DataFrame([
    dict(zip(["年化收益%","年化波动%","夏普","最大回撤%","月度胜率%"], metrics(long_rets))),
    dict(zip(["年化收益%","年化波动%","夏普","最大回撤%","月度胜率%"], metrics(bench))),
    dict(zip(["年化收益%","年化波动%","夏普","最大回撤%","月度胜率%"], metrics(neutral))),
], index=["多头组合(对冲前,扣费)", "基准 标普500", "市场中性组合(对冲后)"]).round(3)

print("="*78)
print("第三步 市场中性策略 — 任务1&2 结果")
print("="*78)
print(tbl.to_string())

print(f"\n【Beta监控】")
print(f"  对冲前 平均Beta: {beta_before.mean():.3f}   区间[{beta_before.min():.2f}, {beta_before.max():.2f}]")
print(f"  对冲后 平均Beta: {beta_after.mean():.3f}   区间[{beta_after.min():.2f}, {beta_after.max():.2f}]")
print(f"  → 残余Beta绝对值均值: {beta_after.abs().mean():.3f} (越接近0越中性)")

print(f"\n【纯Alpha检验 — 相关系数】")
print(f"  对冲前 组合 vs 基准: {long_rets.corr(bench):.3f}")
print(f"  对冲后 中性 vs 基准: {neutral.corr(bench):.3f}   ← 应接近0")

# ========== 画图 ==========
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

nav_long = (1+long_rets).cumprod(); nav_b = (1+bench).cumprod(); nav_n = (1+neutral).cumprod()
axes[0].plot(nav_long.index, nav_long, label=f"多头组合(对冲前) 年化{metrics(long_rets)[0]:.1f}%", lw=1.6)
axes[0].plot(nav_b.index, nav_b, label=f"基准 标普500 年化{metrics(bench)[0]:.1f}%", lw=1.6, color="gray")
axes[0].plot(nav_n.index, nav_n, label=f"市场中性组合 年化{metrics(neutral)[0]:.1f}%", lw=2, color="#C0504D")
axes[0].set_title("市场中性 vs 对冲前 vs 基准 (净值)", fontsize=13, fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylabel("净值")

axes[1].plot(beta_before.index, beta_before, label="对冲前 Beta", lw=1.4)
axes[1].plot(beta_after.index, beta_after, label="对冲后 Beta", lw=1.4, color="#C0504D")
axes[1].axhline(1, ls="--", c="gray", lw=0.9); axes[1].axhline(0, ls="--", c="black", lw=0.9)
axes[1].set_title("60日滚动Beta:对冲前后对比", fontsize=13, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylabel("Beta")

plt.tight_layout()
plt.savefig("market_neutral.png", dpi=120, bbox_inches="tight")
plt.show()

pickle.dump({"long": long_rets, "bench": bench, "neutral": neutral,
             "beta_before": beta_before, "beta_after": beta_after},
            open("neutral_results.pkl","wb"))
print("\n✓ neutral_results.pkl, market_neutral.png")

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','PingFang SC','Heiti TC','SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========== 读数据 ==========
obj = pickle.load(open("portfolio_opt_v4.pkl", "rb"))
r5 = obj["results"]["每5天"]
ms_w_list, rebal_dates = r5["ms_w"], r5["rebal_dates"]

daily_ret = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
spx       = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True).iloc[:, 0]
daily_ret.index = pd.to_datetime(daily_ret.index).tz_localize(None).normalize()
spx.index       = pd.to_datetime(spx.index).tz_localize(None).normalize()

COST_BPS, W = 5, 60
rebal_norm = [pd.Timestamp(rd).tz_localize(None).normalize() for rd in rebal_dates]
all_dates  = daily_ret.loc[(daily_ret.index >= rebal_norm[0]) & (daily_ret.index <= "2025-12-31")].index

# ========== 个股原始Beta(60日滚动,未标准化) ==========
# 注意:不能用style_factors里的beta(那是z-score标准化的),对冲需要原始Beta
common = daily_ret.index.intersection(spx.index)
dr, sp = daily_ret.loc[common], spx.loc[common]
var_m  = sp.rolling(W).var()
ind_beta = pd.DataFrame({c: dr[c].rolling(W).cov(sp) / var_m for c in dr.columns}, index=common)
print(f"个股Beta矩阵: {ind_beta.shape}")

# ========== 回测:多头收益 + 每期估计组合Beta ==========
long_rets, bhat_series, dates_used = [], [], []
prev_w = None

for i, rd in enumerate(rebal_norm):
    w = ms_w_list[i].copy()
    w.index = [str(t) for t in w.index]
    w = w.groupby(level=0).sum()
    cols = [t for t in w.index if t in daily_ret.columns]
    w = w[cols] / w[cols].sum()

    # --- 自下而上估计组合Beta:只用rd当天及之前可得的个股Beta(防前视) ---
    avail = ind_beta.index[ind_beta.index <= rd]
    if len(avail) == 0:
        bhat = 1.0
    else:
        ib = ind_beta.loc[avail[-1], cols]
        ib = ib.fillna(1.0)            # 历史不足的股票,保守假设Beta=1
        bhat = float((w * ib).sum())
    bhat = float(np.clip(bhat, 0.3, 2.0))   # 防止估计噪声导致极端值

    end  = rebal_norm[i+1] if i+1 < len(rebal_norm) else all_dates[-1]
    hold = all_dates[(all_dates > rd) & (all_dates <= end)]
    if len(hold) == 0: continue

    cost = 0.0
    if prev_w is not None:
        allt = w.index.union(prev_w.index)
        cost = np.abs(w.reindex(allt).fillna(0) - prev_w.reindex(allt).fillna(0)).sum() * COST_BPS/10000

    for j, d in enumerate(hold):
        r = daily_ret.loc[d, cols].fillna(0)
        long_rets.append(float((w * r).sum()) - (cost if j == 0 else 0))
        bhat_series.append(bhat)
        dates_used.append(d)
    prev_w = w

long_rets   = pd.Series(long_rets,   index=dates_used)
bhat_series = pd.Series(bhat_series, index=dates_used)
bench       = spx.reindex(dates_used).fillna(0)

# ========== 两种对冲 ==========
neutral_static = long_rets - bench                    # 1:1静态
neutral_beta   = long_rets - bhat_series * bench      # Beta调整

# ========== 指标 ==========
def metrics(r):
    ann = (1+r).prod()**(252/len(r)) - 1
    vol = r.std()*np.sqrt(252)
    nav = (1+r).cumprod()
    mdd = (nav/nav.cummax()-1).min()
    mon = (1+r).resample("ME").prod() - 1
    return dict(年化收益=round(ann*100,2), 年化波动=round(vol*100,2),
                夏普=round(ann/vol,3) if vol>0 else np.nan,
                最大回撤=round(mdd*100,2), 月度胜率=round((mon>0).mean()*100,1))

rb = lambda s: (s.rolling(W).cov(bench) / bench.rolling(W).var())
b_before, b_static, b_beta = rb(long_rets), rb(neutral_static), rb(neutral_beta)

tbl = pd.DataFrame([metrics(long_rets), metrics(bench), metrics(neutral_static), metrics(neutral_beta)],
                   index=["多头组合(对冲前,扣费)","基准 标普500","市场中性(1:1静态对冲)","市场中性(Beta调整对冲)"])
print("="*82); print("Beta调整对冲 vs 1:1静态对冲"); print("="*82)
print(tbl.to_string())

print(f"\n【残余Beta对比】")
for name, b in [("对冲前", b_before), ("1:1静态对冲", b_static), ("Beta调整对冲", b_beta)]:
    bb = b.dropna()
    print(f"  {name:<14} 平均{bb.mean():+.3f}  区间[{bb.min():+.2f},{bb.max():+.2f}]  |残余|均值{bb.abs().mean():.3f}")

print(f"\n【与基准相关系数】(任务要求接近0)")
print(f"  对冲前:        {long_rets.corr(bench):+.3f}")
print(f"  1:1静态对冲:   {neutral_static.corr(bench):+.3f}")
print(f"  Beta调整对冲:  {neutral_beta.corr(bench):+.3f}")

print(f"\n【估计的组合Beta】均值{bhat_series.mean():.3f}, 区间[{bhat_series.min():.2f}, {bhat_series.max():.2f}]")

# ========== 画图 ==========
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
for s, lab, c, lw in [(long_rets,"多头组合(对冲前)",None,1.5),(bench,"基准 标普500","gray",1.5),
                      (neutral_static,"中性(1:1静态)","#4472C4",1.6),(neutral_beta,"中性(Beta调整)","#C0504D",2.0)]:
    nav=(1+s).cumprod(); axes[0].plot(nav.index, nav, label=f"{lab} 年化{metrics(s)['年化收益']:.1f}%", color=c, lw=lw)
axes[0].set_title("市场中性:Beta调整对冲 vs 1:1静态对冲", fontsize=13, fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylabel("净值")

axes[1].plot(b_before.index, b_before, label="对冲前", lw=1.3, color="gray")
axes[1].plot(b_static.index, b_static, label="1:1静态对冲后", lw=1.4, color="#4472C4")
axes[1].plot(b_beta.index, b_beta, label="Beta调整对冲后", lw=1.6, color="#C0504D")
axes[1].axhline(0, ls="--", c="black", lw=0.9); axes[1].axhline(1, ls="--", c="gray", lw=0.8)
axes[1].set_title("60日滚动残余Beta", fontsize=13, fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylabel("Beta")
plt.tight_layout(); plt.savefig("neutral_beta_hedge.png", dpi=120, bbox_inches="tight"); plt.show()

pickle.dump({"long":long_rets,"bench":bench,"neutral_static":neutral_static,
             "neutral_beta":neutral_beta,"bhat":bhat_series,
             "beta_before":b_before,"beta_static":b_static,"beta_beta":b_beta},
            open("neutral_results.pkl","wb"))
print("\n✓ neutral_results.pkl, neutral_beta_hedge.png")

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','PingFang SC','Heiti TC','SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========== 读数据 ==========
predictions = pd.read_pickle("final_predictions.pkl")
sector      = pd.read_pickle("sector_classification.pkl")["GICS_Sector"]
bw          = pd.read_pickle("benchmark_weights.pkl")
daily_ret   = pd.read_csv("daily_returns_clean.csv", index_col=0, parse_dates=True)
spx         = pd.read_csv("spx_returns.csv", index_col=0, parse_dates=True).iloc[:,0]
styles      = pickle.load(open("style_factors.pkl","rb"))
prev        = pickle.load(open("neutral_results.pkl","rb"))
obj         = pickle.load(open("portfolio_opt_v4.pkl","rb"))
rebal_dates = obj["results"]["每5天"]["rebal_dates"]

for df in (bw, daily_ret): df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
spx.index = pd.to_datetime(spx.index).tz_localize(None).normalize()
for k in styles: styles[k].index = pd.to_datetime(styles[k].index).tz_localize(None).normalize()
predictions.index = predictions.index.set_levels(
    pd.to_datetime(predictions.index.levels[0]).tz_localize(None).normalize(), level=0)

tickers84  = list(sector.index)
rebal_norm = [pd.Timestamp(rd).tz_localize(None).normalize() for rd in rebal_dates]
all_dates  = daily_ret.loc[(daily_ret.index>=rebal_norm[0]) & (daily_ret.index<="2025-12-31")].index
COST_BPS, W, TOP = 5, 60, 0.20
sec_names  = sorted(sector.unique())
sec_stocks = {s: list(sector.index[sector==s]) for s in sec_names}

common = daily_ret.index.intersection(spx.index)
var_m  = spx.loc[common].rolling(W).var()
ind_beta = pd.DataFrame({c: daily_ret.loc[common,c].rolling(W).cov(spx.loc[common])/var_m
                         for c in daily_ret.columns}, index=common)

def asof(df, d, cols):
    idx = df.index[df.index <= d]
    return df.loc[idx[-1], cols] if len(idx) else pd.Series(np.nan, index=cols)

# ========== 构建(已修复) ==========
sn_rets, sn_bhat, dates_used = [], [], []
ind_dev_rec, style_exp_rec, maxw_rec, nhold_rec = [], [], [], []
fallback_count = {s: 0 for s in sec_names}
n_rebal = 0
prev_w = None

for i, rd in enumerate(rebal_norm):
    try:
        alpha = predictions.xs(rd, level=0)["pred_xgb"].reindex(tickers84)
    except KeyError:
        continue
    if alpha.notna().sum() < 40: continue

    wb_d = asof(bw, rd, tickers84).fillna(0)
    if wb_d.sum() == 0: continue
    wb_d = wb_d / wb_d.sum()
    bench_sec = wb_d.groupby(sector).sum()

    w = pd.Series(0.0, index=tickers84)
    for sec in sec_names:
        stks   = sec_stocks[sec]
        target = float(bench_sec.get(sec, 0.0))       # 该板块必须配到的总权重
        if target <= 0: continue
        valid  = [t for t in stks if pd.notna(alpha.get(t))]
        if len(valid) == 0:
            # ★修复1: 板块内无有效alpha → 回退到基准板块内权重(无观点则持有基准)
            sw = wb_d[stks]
            w[stks] = target*(sw/sw.sum()) if sw.sum() > 0 else target/len(stks)
            fallback_count[sec] += 1
            continue
        k = max(1, int(round(TOP*len(valid))))        # 板块内前20%,至少1只
        picks = alpha[valid].sort_values(ascending=False).head(k).index
        w[picks] = target / k                         # 板块内等权,合计=基准板块权重

    # ★修复2: 不做全局归一化(每板块已精确对齐,总和自然=1)
    assert abs(w.sum()-1) < 1e-8, f"权重和异常 {w.sum()}"
    n_rebal += 1

    ind_dev_rec.append(w.groupby(sector).sum() - bench_sec.reindex(sec_names).fillna(0))
    style_exp_rec.append(pd.Series({
        s: float((w*asof(styles[s],rd,tickers84).fillna(0)).sum()) -
           float((wb_d*asof(styles[s],rd,tickers84).fillna(0)).sum()) for s in styles}))
    maxw_rec.append(w.max()); nhold_rec.append(int((w>1e-6).sum()))

    ib   = asof(ind_beta, rd, tickers84).fillna(1.0)
    bhat = float(np.clip((w*ib).sum(), 0.3, 2.0))

    end  = rebal_norm[i+1] if i+1 < len(rebal_norm) else all_dates[-1]
    hold = all_dates[(all_dates>rd) & (all_dates<=end)]
    if len(hold)==0: continue
    cost = 0.0 if prev_w is None else np.abs(w-prev_w).sum()*COST_BPS/10000
    for j, d in enumerate(hold):
        r = daily_ret.loc[d, tickers84].fillna(0)
        sn_rets.append(float((w*r).sum()) - (cost if j==0 else 0))
        sn_bhat.append(bhat); dates_used.append(d)
    prev_w = w

sn_long    = pd.Series(sn_rets, index=dates_used)
sn_bhat    = pd.Series(sn_bhat, index=dates_used)
bench      = spx.reindex(dates_used).fillna(0)
sn_neutral = sn_long - sn_bhat*bench

# ========== 结果 ==========
def metrics(r):
    ann=(1+r).prod()**(252/len(r))-1; vol=r.std()*np.sqrt(252)
    nav=(1+r).cumprod(); mdd=(nav/nav.cummax()-1).min(); mon=(1+r).resample("ME").prod()-1
    return dict(年化收益=round(ann*100,2), 年化波动=round(vol*100,2),
                夏普=round(ann/vol,3) if vol>0 else np.nan,
                最大回撤=round(mdd*100,2), 月度胜率=round((mon>0).mean()*100,1))

al = lambda s: s.reindex(dates_used).fillna(0)
print("="*88); print("任务3 行业中性化(bug修复后,严格按任务清单:板块内前20%等权)"); print("="*88)
print(pd.DataFrame([
    metrics(al(prev["long"])), metrics(bench), metrics(al(prev["neutral_beta"])),
    metrics(sn_long), metrics(sn_neutral),
], index=["原多头(第二阶段,对冲前)","基准 标普500","原中性(Beta调整对冲)",
          "行业中性多头(对冲前)","行业中性+Beta对冲 ★"]).to_string())

print(f"\n【行业偏离验证】平均偏离%(修复后应全为0)")
dev = pd.DataFrame(ind_dev_rec)
print((dev.mean()*100).round(8).to_string())
print(f"  最大|单期偏离| = {dev.abs().max().max()*100:.10f}%")

print(f"\n【回退诊断】共{n_rebal}个调仓日, 触发回退的板块:")
fb = {k:v for k,v in fallback_count.items() if v>0}
print(f"  {fb if fb else '无'}")
if fb:
    for s,c in fb.items(): print(f"    {s}: {c}次 ({c/n_rebal*100:.1f}%的调仓日) — 该板块共{len(sec_stocks[s])}只股票")

rb = lambda s: (s.rolling(W).cov(bench)/bench.rolling(W).var()).dropna()
print(f"\n【中性效果】")
for n,s in [("原中性(Beta调整)", al(prev["neutral_beta"])), ("行业中性+Beta对冲", sn_neutral)]:
    print(f"  {n:<20} |残余Beta|均值{rb(s).abs().mean():.3f}   与基准相关{s.corr(bench):+.3f}")

print(f"\n【风格暴露对比】(相对基准)")
print(pd.DataFrame({"原组合(任务4)": pd.Series({"size":-1.339,"momentum":-0.551,
                    "volatility":0.505,"beta":-0.032,"value":0.416}),
                    "行业中性组合": pd.DataFrame(style_exp_rec).mean()}).round(3).to_string())

print(f"\n【集中度】平均持仓{np.mean(nhold_rec):.1f}只, 平均最大单只{np.mean(maxw_rec)*100:.2f}%, "
      f"历史最大单只{np.max(maxw_rec)*100:.2f}%")

# ========== 画图 ==========
fig, axes = plt.subplots(2,1, figsize=(13,9))
for s,lab,c,lw in [(bench,"基准 标普500","gray",1.4),
                   (al(prev["neutral_beta"]),"原中性(Beta调整)","#4472C4",1.6),
                   (sn_neutral,"行业中性+Beta对冲","#C0504D",2.0)]:
    nav=(1+s).cumprod(); m=metrics(s)
    axes[0].plot(nav.index,nav,label=f"{lab} 年化{m['年化收益']:.1f}% 波动{m['年化波动']:.1f}% 夏普{m['夏普']:.2f}",color=c,lw=lw)
axes[0].set_title("行业中性化对净值稳定性的影响(bug修复后)",fontsize=13,fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylabel("净值")

for s,lab,c in [(al(prev["neutral_beta"]),"原中性","#4472C4"),(sn_neutral,"行业中性","#C0504D")]:
    nav=(1+s).cumprod(); axes[1].plot(nav.index,(nav/nav.cummax()-1)*100,label=lab,color=c,lw=1.5)
axes[1].set_title("回撤对比",fontsize=13,fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylabel("回撤 %")
plt.tight_layout(); plt.savefig("sector_neutral.png",dpi=120,bbox_inches="tight"); plt.show()

pickle.dump({"sn_long":sn_long,"sn_neutral":sn_neutral,"bench":bench,
             "ind_dev":dev,"style_exp":pd.DataFrame(style_exp_rec),
             "maxw":maxw_rec,"nhold":nhold_rec,"fallback":fallback_count},
            open("sector_neutral_results.pkl","wb"))
print("\n✓ sector_neutral_results.pkl, sector_neutral.png")

In [ ]:
# ================= 第四步 任务1:Brinson 归因(按GICS板块) =================
SECTOR = IND.loc[TICKERS].idxmax(axis=1)          # 每只股票的板块
SEC_NAMES = list(IND.columns)

# 基准权重滞后一天(日度)
_WB = BW.reindex(TEST_IDX).ffill()[TICKERS].fillna(0)
_WB = _WB.div(_WB.sum(axis=1), axis=0).shift(1)

def brinson(wdict, label):
    """逐日做Brinson三效应分解,返回(配置,选股,交互)三个 日期×板块 的DataFrame"""
    rl = sorted(wdict.keys())
    recs = {"alloc": [], "selec": [], "inter": []}
    dates = []
    for i, rd in enumerate(rl):
        w = wdict[rd].reindex(TICKERS).fillna(0).values      # 持有期内权重不变
        end = rl[i+1] if i+1 < len(rl) else TEST_IDX[-1]
        for d in TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]:
            wb = _WB.loc[d].values
            if np.isnan(wb).any(): continue
            r = DR.loc[d, TICKERS].fillna(0).values

            Wp = IND_MAT.T @ w;          Wb = IND_MAT.T @ wb
            Np = IND_MAT.T @ (w * r);    Nb = IND_MAT.T @ (wb * r)
            Rb = np.where(Wb > 1e-12, Nb / np.where(Wb > 1e-12, Wb, 1), 0.0)
            # 组合该板块无持仓时,约定 Rp = Rb (无选股效应)
            Rp = np.where(Wp > 1e-12, Np / np.where(Wp > 1e-12, Wp, 1), Rb)

            dW, dR = Wp - Wb, Rp - Rb
            recs["alloc"].append(dW * Rb)
            recs["selec"].append(Wb * dR)
            recs["inter"].append(dW * dR)
            dates.append(d)
    out = {k: pd.DataFrame(v, index=dates, columns=SEC_NAMES) for k, v in recs.items()}
    # 恒等式校验
    tot = sum(out[k].sum(axis=1) for k in out)
    print(f"  [{label}] 三效应合计 {tot.sum()*100:+.4f}%  (逐日算术加总)")
    return out

print("Brinson 分解中...")
BR_ENH = brinson(W_ENH, f"指数增强λ={LAM_STAR:g}")
BR_P2  = brinson(W_P2,  "第二阶段最大夏普")

# ---- 汇总表 ----
def summary(br, name):
    a, s, i = br["alloc"].sum().sum(), br["selec"].sum().sum(), br["inter"].sum().sum()
    t = a + s + i
    return {"组合": name, "配置效应%": a*100, "选股效应%": s*100, "交互效应%": i*100,
            "合计超额%": t*100, "配置占比%": a/t*100 if t else np.nan,
            "选股占比%": s/t*100 if t else np.nan}

print("\n" + "="*116)
print("Brinson 归因汇总(测试期累计,逐日算术加总)")
print("="*116)
print(pd.DataFrame([summary(BR_ENH, f"指数增强(λ={LAM_STAR:g})"),
                    summary(BR_P2, "第二阶段最大夏普(无约束)")]).round(3).to_string(index=False))

# ---- 按板块看(指数增强) ----
sec_tab = pd.DataFrame({"配置效应%": BR_ENH["alloc"].sum()*100,
                        "选股效应%": BR_ENH["selec"].sum()*100,
                        "交互效应%": BR_ENH["inter"].sum()*100})
sec_tab["合计%"] = sec_tab.sum(axis=1)
print(f"\n【指数增强 分板块归因】")
print(sec_tab.sort_values("合计%", ascending=False).round(3).to_string())

# ---- 按季度堆积柱状图 ----
q = pd.DataFrame({"配置效应": BR_ENH["alloc"].sum(axis=1),
                  "选股效应": BR_ENH["selec"].sum(axis=1),
                  "交互效应": BR_ENH["inter"].sum(axis=1)}).resample("QE").sum()*100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
btm_pos = np.zeros(len(q)); btm_neg = np.zeros(len(q))
xs = np.arange(len(q))
for col, c in [("配置效应","#4472C4"), ("选股效应","#70AD47"), ("交互效应","#FFC000")]:
    v = q[col].values
    pos, neg = np.where(v > 0, v, 0), np.where(v < 0, v, 0)
    axes[0].bar(xs, pos, bottom=btm_pos, color=c, label=col, width=0.72)
    axes[0].bar(xs, neg, bottom=btm_neg, color=c, width=0.72)
    btm_pos += pos; btm_neg += neg
axes[0].plot(xs, q.sum(axis=1).values, "ko-", ms=4, lw=1.2, label="季度总超额")
axes[0].axhline(0, c="black", lw=0.9)
axes[0].set_xticks(xs); axes[0].set_xticklabels([f"{d.year}Q{d.quarter}" for d in q.index], rotation=45, fontsize=8)
axes[0].set_ylabel("季度效应 (%)"); axes[0].legend(fontsize=9); axes[0].grid(axis="y", alpha=0.3)
axes[0].set_title(f"Brinson 归因:季度堆积 (指数增强 λ={LAM_STAR:g})", fontsize=13, fontweight="bold")

for col, c in [("配置效应","#4472C4"), ("选股效应","#70AD47"), ("交互效应","#FFC000")]:
    axes[1].plot(q.index, q[col].cumsum(), lw=1.9, color=c, label=f"{col} 累计 {q[col].sum():+.2f}%")
axes[1].axhline(0, c="black", lw=0.9); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylabel("累计效应 (%)"); axes[1].set_title("三效应累计路径", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("brinson_attribution.png", dpi=120, bbox_inches="tight"); plt.show()

print(f"\n【季度稳定性】选股效应为正的季度: {(q['选股效应']>0).sum()}/{len(q)}  "
      f"配置效应为正: {(q['配置效应']>0).sum()}/{len(q)}")

In [ ]:
# ================= 第四步 任务2:因子归因 =================
# ---- 步骤1:逐日横截面回归,估计5个风格因子的"因子收益" ----
# 用 t-1 的暴露解释 t 的收益 → 无前视
STY_LAG = {}
for s in STY_NAMES:
    df = STY[s].reindex(TEST_IDX).ffill()[TICKERS]
    STY_LAG[s] = df.shift(1)

f_rows, f_dates = [], []
for d in TEST_IDX:
    X_raw = np.column_stack([STY_LAG[s].loc[d].values for s in STY_NAMES])
    y_raw = DR.loc[d, TICKERS].values
    ok = ~np.isnan(X_raw).any(axis=1) & ~np.isnan(y_raw)
    if ok.sum() < 30: continue
    X = np.column_stack([np.ones(ok.sum()), X_raw[ok]])
    coef = np.linalg.lstsq(X, y_raw[ok], rcond=None)[0]
    f_rows.append(coef[1:]); f_dates.append(d)

FRET = pd.DataFrame(f_rows, index=f_dates, columns=STY_NAMES)
print("【风格因子收益】(逐日横截面回归估计, 年化)")
print((FRET.mean()*252*100).round(2).to_string())
print(f"\n(正值=该风格暴露高的股票在测试期跑赢; 负值=跑输)")

# ---- 步骤2:组合超额收益 对 因子收益 做时间序列回归 ----
def factor_attrib(r_port, bench, name):
    ex = (r_port - bench.reindex(r_port.index)).dropna()
    idx = ex.index.intersection(FRET.index)
    y = ex.loc[idx].values
    Xf = FRET.loc[idx].values
    X = np.column_stack([np.ones(len(idx)), Xf])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, p = len(y), X.shape[1]
    s2 = resid @ resid / (n - p)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    tstat = beta / se
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))

    contrib = beta[1:] * Xf.mean(axis=0) * 252 * 100          # 各风格年化贡献
    alpha_ann = beta[0] * 252 * 100                            # 纯alpha年化
    tab = pd.DataFrame({"回归系数β(暴露)": beta[1:], "t值": tstat[1:],
                        "年化贡献%": contrib}, index=STY_NAMES)
    print("\n" + "="*96)
    print(f"因子归因:{name}   (超额收益 对 风格因子收益 的时间序列回归)")
    print("="*96)
    print(tab.round(3).to_string())
    print(f"\n  截距α(纯选股alpha) 年化 {alpha_ann:+.3f}%   t值 {tstat[0]:+.2f}")
    print(f"  风格贡献合计       年化 {contrib.sum():+.3f}%")
    print(f"  实际年化超额       {((1+ex).prod()**(252/len(ex))-1)*100:+.3f}%")
    print(f"  R² = {r2*100:.1f}%  → 风格因子能解释超额收益波动的{r2*100:.1f}%,"
          f"剩余{(1-r2)*100:.1f}%为纯选股")
    return dict(alpha=alpha_ann, style=contrib.sum(), r2=r2, tab=tab, t_alpha=tstat[0])

FA_ENH = factor_attrib(R_ENH, BENCH["市值加权84"], f"指数增强(λ={LAM_STAR:g})")
FA_P2  = factor_attrib(R_P2_N, BENCH["市值加权84"], "第二阶段最大夏普(无约束)")

print("\n" + "="*96)
print("对比:基准约束对超额收益'纯净度'的影响")
print("="*96)
print(pd.DataFrame([
    {"组合": f"指数增强(λ={LAM_STAR:g})", "纯alpha年化%": FA_ENH["alpha"],
     "α的t值": FA_ENH["t_alpha"], "风格贡献年化%": FA_ENH["style"], "R²%": FA_ENH["r2"]*100},
    {"组合": "第二阶段最大夏普(无约束)", "纯alpha年化%": FA_P2["alpha"],
     "α的t值": FA_P2["t_alpha"], "风格贡献年化%": FA_P2["style"], "R²%": FA_P2["r2"]*100},
]).round(3).to_string(index=False))

In [ ]:
# ================= 第四步 任务3:风格漂移检查 =================
def style_drift(wdict, label):
    rl = sorted(wdict.keys())
    rows, dates = [], []
    for i, rd in enumerate(rl):
        w = wdict[rd].reindex(TICKERS).fillna(0).values
        end = rl[i+1] if i+1 < len(rl) else TEST_IDX[-1]
        for d in TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]:
            wb = _WB.loc[d].values
            if np.isnan(wb).any(): continue
            act = w - wb
            row = []
            for s in STY_NAMES:
                sv = STY[s].reindex(TEST_IDX).ffill().loc[d, TICKERS].values
                m = ~np.isnan(sv)
                row.append(float(act[m] @ sv[m]))
            rows.append(row); dates.append(d)
    return pd.DataFrame(rows, index=dates, columns=STY_NAMES)

print("计算风格暴露时序...")
DRIFT = style_drift(W_ENH, "指数增强")

LIM = 0.5
print("\n" + "="*104)
print(f"风格漂移检查:指数增强(λ={LAM_STAR:g}) 主动风格暴露  [约束 ±{LIM}]")
print("="*104)
chk = pd.DataFrame({"均值": DRIFT.mean(), "标准差": DRIFT.std(),
                    "最小": DRIFT.min(), "最大": DRIFT.max(),
                    "|暴露|均值": DRIFT.abs().mean(),
                    "越界天数占比%": (DRIFT.abs() > LIM).mean()*100})
print(chk.round(3).to_string())
print(f"\n  任一风格越界的天数占比: {(DRIFT.abs()>LIM).any(axis=1).mean()*100:.1f}%")
print(f"  最大越界幅度: {(DRIFT.abs().max().max()-LIM):.3f} (超出约束)")
print(f"  说明:约束在调仓日施加,持有期内基准权重与暴露值变化导致漂移")

fig, ax = plt.subplots(figsize=(14, 6))
for s, c in zip(STY_NAMES, ["#4472C4","#C0504D","#70AD47","#FFC000","#7030A0"]):
    ax.plot(DRIFT.index, DRIFT[s], lw=1.1, label=s, color=c, alpha=0.85)
ax.axhline(LIM, ls="--", c="red", lw=1.2, label=f"约束 ±{LIM}")
ax.axhline(-LIM, ls="--", c="red", lw=1.2)
ax.axhline(0, c="black", lw=0.8)
ax.set_title(f"风格主动暴露时序 (指数增强 λ={LAM_STAR:g})", fontsize=13, fontweight="bold")
ax.set_ylabel("主动暴露 (标准差)"); ax.legend(ncol=6, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("style_drift.png", dpi=120, bbox_inches="tight"); plt.show()

# ================= 第四步 任务4:归因结论 =================
a_e = BR_ENH["alloc"].sum().sum()*100
s_e = BR_ENH["selec"].sum().sum()*100
i_e = BR_ENH["inter"].sum().sum()*100
print("\n" + "="*104)
print(f"归因结论  指数增强(λ={LAM_STAR:g}) 相对 84只市值加权基准")
print("="*104)
sel_total = s_e + i_e                      # 选股+交互(业界常规:交互并入选股)
tot_arith = a_e + s_e + i_e
cost_ann  = TO_ENH * COST_BPS/10000 * (len(W_ENH)-1) / 5 * 100

print(f"""
① 超额收益主要来自哪里?   [注:Brinson为"费前、算术裸加总、全期累计"口径]
   Brinson(行业维度):  配置 {a_e:+.2f}%  |  选股 {s_e:+.2f}%  |  交互 {i_e:+.2f}%
                       选股+交互合计 {sel_total:+.2f}%   vs   配置 {a_e:+.2f}%
   → 主导来源: {'选股' if abs(sel_total) > abs(a_e)*1.2 else ('配置' if abs(a_e) > abs(sel_total)*1.2 else '配置与选股大致均衡')}
     (交互效应衡量"超配的板块恰好也选股选得好",属选股能力的一部分,不宜单列丢弃)

   因子(风格维度):     风格溢价 {FA_ENH['style']:+.2f}%/年  |  纯alpha(α) {FA_ENH['alpha']:+.2f}%/年
                       风格因子解释超额波动的 {FA_ENH['r2']*100:.1f}%
   → α的t值 {FA_ENH['t_alpha']:+.2f} → {'统计显著' if abs(FA_ENH['t_alpha'])>1.96 else '统计上不显著'}
     诚实说明:5年日频样本对IR≈{perf(R_ENH, BENCH["市值加权84"])["IR"]:.2f}量级的alpha检验功率不足,
     无法在统计上将超额与零区分,不能据此断言选股能力存在或不存在

   口径对账: 费前算术 {tot_arith:+.2f}% ÷5年 = {tot_arith/5:+.2f}%/年
             扣成本 −{cost_ann:.2f}%/年 → {tot_arith/5-cost_ann:+.2f}%/年
             再计入算术→几何的波动拖累 ≈ 几何年化超额 {perf(R_ENH, BENCH["市值加权84"])["年化超额%"]:+.2f}% ✓

② 是否承担了计划之外的风险?
   风格暴露越界(|暴露|>{LIM})的天数占比 {(DRIFT.abs()>LIM).any(axis=1).mean()*100:.1f}%,
   最大暴露 {DRIFT.abs().max().max():.3f} (超出约束 {DRIFT.abs().max().max()-LIM:.3f})
   → 约束仅在调仓日施加,持有期内因基准权重与暴露值变化产生漂移
     严格说这是"调仓日约束"而非"全时段约束",但漂移幅度温和、可接受
   → 完全未被约束/对冲的维度:个股特异风险、板块内集中度、协方差估计误差
     (实现跟踪误差系统性高于事前约{(perf(R_ENH, BENCH["市值加权84"])["跟踪误差%"]/(EXANTE_TE*100)-1)*100:.0f}%)

③ 基准约束带来了什么、放弃了什么?   (对比第二阶段无约束组合)
   得到:  风格拖累 {FA_P2['style']:+.2f}% → {FA_ENH['style']:+.2f}%,  改善 {FA_ENH['style']-FA_P2['style']:+.2f}pp
          其中小盘押注的代价从 {FA_P2['tab'].loc['size','年化贡献%']:+.2f}% 削到 {FA_ENH['tab'].loc['size','年化贡献%']:+.2f}%
          跟踪误差可控 {perf(R_ENH, BENCH["市值加权84"])["跟踪误差%"]:.2f}% (目标3-5%)、换手与成本近乎减半
   放弃:  纯alpha {FA_P2['alpha']:+.2f}% → {FA_ENH['alpha']:+.2f}%,  损失 {FA_ENH['alpha']-FA_P2['alpha']:+.2f}pp
          个股偏离被限死±2%,无法充分表达alpha观点(第二阶段单只可配至7%)
   净效果: 实际超额 {FA_P2['alpha']+FA_P2['style']:+.2f}% → {FA_ENH['alpha']+FA_ENH['style']:+.2f}%
          → 约束同时压住了"错的风格押注"和"对的选股表达",本例中代价大于收益
   R²:    {FA_P2['r2']*100:.1f}% → {FA_ENH['r2']*100:.1f}%,  几乎未变
          → 约束压缩了风格暴露的幅度(β变小),但未改变风格解释超额波动的比例
            (β与残差同步缩小),故"约束使超额更纯粹"这一预期未获支持
""")
pickle.dump({"brinson_enh": BR_ENH, "brinson_p2": BR_P2, "factor_ret": FRET,
             "fa_enh": FA_ENH, "fa_p2": FA_P2, "drift": DRIFT, "quarterly": q},
            open("attribution_results.pkl", "wb"))
print("✓ attribution_results.pkl, brinson_attribution.png, style_drift.png")

In [ ]:
# ================= 第五步 任务1:压力期分析 =================
_nr = pickle.load(open("neutral_results.pkl", "rb"))
R_NEUTRAL = _nr["neutral_beta"]              # 市场中性(Beta调整对冲)
BETA_NEU  = _nr["beta_beta"]                 # 中性组合60日滚动残余Beta

# ---- 各调仓日的事前跟踪误差(用于协方差失效诊断) ----
_ex = {}
for c in CACHE:
    if c["rd"] in W_ENH:
        act = W_ENH[c["rd"]].values - c["wb"]
        _ex[c["rd"]] = float(np.sqrt(max(act @ c["Sigma"] @ act, 0)) * np.sqrt(252))
EXANTE_S = pd.Series(_ex).sort_index()

# ---- 自动识别最深回撤区间(以真标普500为准) ----
def worst_window(s):
    nav = (1+s.dropna()).cumprod()
    dd  = nav/nav.cummax() - 1
    trough = dd.idxmin()
    peak   = nav.loc[:trough].idxmax()
    return peak, trough, dd.min()

_pk, _tr, _dd = worst_window(BENCH["真标普500"])
print(f"自动识别的最深回撤区间(真标普500): {_pk.date()} → {_tr.date()}  跌幅 {_dd*100:.2f}%")

WINDOWS = {
    "全测试期":        (TEST_IDX[0], TEST_IDX[-1]),
    "2022加息熊市":    (pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31")),
    f"最深回撤段({_pk.date()}~{_tr.date()})": (_pk, _tr),
}

STRATS = {
    "指数增强(λ=%g)" % LAM_STAR: (R_ENH,     BENCH["市值加权84"]),
    "第二阶段最大夏普":            (R_P2_N,    BENCH["市值加权84"]),
    "市场中性(Beta对冲)":          (R_NEUTRAL, BENCH["真标普500"]),
    "基准:市值加权84":             (BENCH["市值加权84"], None),
    "基准:真标普500":              (BENCH["真标普500"],  None),
}

def win_stats(r, bench, lo, hi):
    s = r.dropna()
    s = s[(s.index >= lo) & (s.index <= hi)]
    if len(s) < 5: return None
    nav = (1+s).cumprod()
    cum = nav.iloc[-1] - 1
    ann = (1+s).prod()**(252/len(s)) - 1
    vol = s.std()*np.sqrt(252)
    mdd = (nav/nav.cummax()-1).min()
    mon = (1+s).resample("ME").prod() - 1
    out = {"累计收益%": cum*100, "年化收益%": ann*100, "年化波动%": vol*100,
           "最大回撤%": mdd*100, "月度胜率%": (mon > 0).mean()*100 if len(mon) >= 3 else np.nan}
    if bench is not None:
        b = bench.reindex(s.index).dropna()
        rr = s.reindex(b.index); ex = rr - b
        ann_b = (1+b).prod()**(252/len(b)) - 1
        ann_r = (1+rr).prod()**(252/len(rr)) - 1
        te = ex.std()*np.sqrt(252)
        out.update({"年化超额%": (ann_r-ann_b)*100, "跟踪误差%": te*100,
                    "IR": (ann_r-ann_b)/te if te > 0 else np.nan})
    return out

for wname, (lo, hi) in WINDOWS.items():
    rows = []
    for sname, (r, b) in STRATS.items():
        st = win_stats(r, b, lo, hi)
        if st is None: continue
        rows.append({"策略/基准": sname, **st})
    ndays = len(TEST_IDX[(TEST_IDX >= lo) & (TEST_IDX <= hi)])
    print("\n" + "="*126)
    print(f"【{wname}】  {lo.date()} ~ {hi.date()}  ({ndays}个交易日)")
    print("="*126)
    print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---- 核心检验:市场中性在压力期是否为正收益 ----
print("\n" + "="*90)
print("核心检验:市场中性的'绝对收益'卖点在压力期是否成立")
print("="*90)
for wname, (lo, hi) in WINDOWS.items():
    if wname == "全测试期": continue
    sn = R_NEUTRAL.dropna(); sn = sn[(sn.index >= lo) & (sn.index <= hi)]
    se = R_ENH.dropna();     se = se[(se.index >= lo) & (se.index <= hi)]
    sb = BENCH["真标普500"].dropna(); sb = sb[(sb.index >= lo) & (sb.index <= hi)]
    cn = (1+sn).prod()-1; ce = (1+se).prod()-1; cb = (1+sb).prod()-1
    print(f"\n{wname}:")
    print(f"  真标普500      {cb*100:+7.2f}%")
    print(f"  指数增强        {ce*100:+7.2f}%   (相对基准 {(ce-cb)*100:+.2f}pp,但绝对亏损)")
    print(f"  市场中性        {cn*100:+7.2f}%   ← {'✓ 正收益,卖点成立' if cn > 0 else '✗ 负收益,卖点不成立'}")

# ---- 协方差失效诊断 ----
print("\n" + "="*90)
print("协方差失效诊断:事前 vs 实现跟踪误差")
print("="*90)
rows = []
ex_enh = (R_ENH - BENCH["市值加权84"].reindex(R_ENH.index)).dropna()
for wname, (lo, hi) in WINDOWS.items():
    exa = EXANTE_S[(EXANTE_S.index >= lo) & (EXANTE_S.index <= hi)]
    rea = ex_enh[(ex_enh.index >= lo) & (ex_enh.index <= hi)]
    if len(exa) < 3 or len(rea) < 10: continue
    a, b = exa.mean()*100, rea.std()*np.sqrt(252)*100
    rows.append({"窗口": wname, "事前TE%": a, "实现TE%": b,
                 "低估pp": b-a, "低估幅度%": (b/a-1)*100})
cov_tab = pd.DataFrame(rows).round(2)
print(cov_tab.to_string(index=False))
print("\n→ 若压力期的低估幅度明显大于全期,即为'协方差矩阵在极端行情下失效'的实证")

# ---- 对冲时机风险 ----
print("\n" + "="*90)
print("对冲时机风险:市场中性组合的残余Beta稳定性")
print("="*90)
rows = []
for wname, (lo, hi) in WINDOWS.items():
    bb = BETA_NEU.dropna(); bb = bb[(bb.index >= lo) & (bb.index <= hi)]
    if len(bb) < 10: continue
    rows.append({"窗口": wname, "残余Beta均值": bb.mean(), "|残余Beta|均值": bb.abs().mean(),
                 "最小": bb.min(), "最大": bb.max(), "摆幅": bb.max()-bb.min()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\n→ 压力期摆幅扩大说明:对冲比例每5天才更新,极端行情下Beta漂移更快、时机风险上升")

# ---- 画图 ----
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
for ax, wname in zip(axes, ["2022加息熊市", list(WINDOWS.keys())[2]]):
    lo, hi = WINDOWS[wname]
    for sname, (r, _) in STRATS.items():
        s = r.dropna(); s = s[(s.index >= lo) & (s.index <= hi)]
        if len(s) < 5: continue
        nav = (1+s).cumprod()
        st = "--" if sname.startswith("基准") else "-"
        lw = 1.3 if sname.startswith("基准") else 2.0
        ax.plot(nav.index, nav, st, lw=lw, label=f"{sname} {(nav.iloc[-1]-1)*100:+.1f}%")
    ax.axhline(1, c="black", lw=0.8)
    ax.set_title(f"压力期净值:{wname}", fontsize=12, fontweight="bold")
    ax.set_ylabel("净值(期初=1)"); ax.legend(fontsize=8.5); ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("stress_periods.png", dpi=120, bbox_inches="tight"); plt.show()

pickle.dump({"windows": WINDOWS, "exante": EXANTE_S, "cov_diag": cov_tab},
            open("stress_results.pkl", "wb"))
print("\n✓ stress_results.pkl, stress_periods.png")

In [ ]:
# ================= 第五步 任务2a:跟踪误差目标 3%/5%/8% 三档 =================
def solve_flex(c, lam, k=1.0):
    """约束按倍数k放松: 个股±2%k, 行业±5%k, 风格±0.5k, 上限max(5%, w_b+2%k)"""
    n = len(TICKERS)
    w = cp.Variable(n); act = w - c["wb"]
    cap = np.maximum(0.05, c["wb"] + 0.02*k)
    prob = cp.Problem(
        cp.Maximize(c["alpha"] @ w - lam * cp.quad_form(act, cp.psd_wrap(c["Sigma"]))),
        [cp.sum(w) == 1, w >= 0,
         act <= 0.02*k, act >= -0.02*k,
         IND_MAT.T @ act <= 0.05*k, IND_MAT.T @ act >= -0.05*k,
         c["style"].T @ act <= 0.5*k, c["style"].T @ act >= -0.5*k,
         w <= cap])
    try:
        prob.solve()
        if prob.status == "optimal":
            v = np.clip(w.value, 0, None); return v / v.sum()
    except Exception:
        pass
    return None

def exante_te(cache_sub, lam, k):
    tes = []
    for c in cache_sub:
        w = solve_flex(c, lam, k)
        if w is None: continue
        act = w - c["wb"]
        tes.append(float(np.sqrt(max(act @ c["Sigma"] @ act, 0)) * np.sqrt(252)))
    return float(np.mean(tes))*100 if tes else np.nan

# ---- 先确认:现有约束(k=1)下跟踪误差的上限 ----
SUB = CACHE[::5]                                    # 抽样加速网格搜索
print("① 现有约束(k=1)下,λ→0 时跟踪误差的天花板:")
for lam in [0.01, 0.05, 0.1, 0.5]:
    print(f"   λ={lam:<6} 事前TE {exante_te(SUB, lam, 1.0):.2f}%")

# ---- 网格搜索:约束倍数k × λ ----
print("\n② 网格搜索(约束倍数k × λ) → 事前跟踪误差%")
GRID_K   = [1.0, 1.5, 2.0, 3.0]
GRID_LAM = [0.1, 0.5, 2, 10, 50]
grid = pd.DataFrame(index=GRID_K, columns=GRID_LAM, dtype=float)
for k in GRID_K:
    for lam in GRID_LAM:
        grid.loc[k, lam] = exante_te(SUB, lam, k)
grid.index.name = "约束倍数k"; grid.columns.name = "λ"
print(grid.round(2).to_string())

# ---- 为每个目标TE找最接近的(k, λ),再做完整回测 ----
TARGETS = [3.0, 5.0, 8.0]
flat = grid.stack().dropna()
sens_rows = []
for tgt in TARGETS:
    k_sel, lam_sel = flat.sub(tgt).abs().idxmin()
    te_hat = flat.loc[(k_sel, lam_sel)]
    reachable = abs(te_hat - tgt) < 0.75
    wd, ex_list = {}, []
    for c in CACHE:
        w = solve_flex(c, lam_sel, k_sel)
        if w is None: continue
        act = w - c["wb"]
        ex_list.append(float(np.sqrt(max(act @ c["Sigma"] @ act, 0))*np.sqrt(252)))
        wd[c["rd"]] = pd.Series(w, index=TICKERS)
    r, to = port_returns(wd, COST_BPS)
    m = perf(r, BENCH["市值加权84"])
    dev = [np.abs(wd[c["rd"]].values - c["wb"]).max() for c in CACHE if c["rd"] in wd]
    sens_rows.append({"目标TE%": tgt, "达成?": "✓" if reachable else "✗未达成",
                      "选定k": k_sel, "选定λ": lam_sel,
                      "事前TE%": np.mean(ex_list)*100, "实现TE%": m["跟踪误差%"],
                      "年化收益%": m["年化收益%"], "年化超额%": m["年化超额%"], "IR": m["IR"],
                      "换手": to, "最大个股偏离": np.mean(dev)})

print("\n" + "="*136)
print("跟踪误差目标 3%/5%/8% 敏感性测试")
print("="*136)
print(pd.DataFrame(sens_rows).round(3).to_string(index=False))
print(f"\n注:k=1为任务清单原始约束(个股±2%/行业±5%/风格±0.5);k>1表示必须放松约束才能达到该目标")

# ================= 第五步 任务2b:调仓频率改20天 =================
def build_cache(step):
    reb = list(TEST_IDX[::step]); out = []
    for i, rd in enumerate(reb):
        try:
            a = PRED.xs(rd, level=0)["pred_xgb"].reindex(TICKERS)
        except KeyError: continue
        if a.notna().sum() < 40: continue
        wb = asof(BW, rd, TICKERS).fillna(0)
        if wb.sum() == 0: continue
        wb = (wb/wb.sum()).values
        sty = np.column_stack([asof(STY[s], rd, TICKERS).reindex(TICKERS).fillna(0).values
                               for s in STY_NAMES])
        hist = DR[TICKERS].loc[DR.index <= rd].tail(126).dropna(axis=1, how="any")
        if hist.shape[1] < 50: continue
        lw = LedoitWolf().fit(hist.values)
        S = pd.DataFrame(lw.covariance_, index=hist.columns, columns=hist.columns) \
              .reindex(index=TICKERS, columns=TICKERS).values.copy()
        S[np.isnan(S)] = 0.0
        d = np.diag(S).copy(); d[d <= 0] = 4e-4; np.fill_diagonal(S, d)
        end = reb[i+1] if i+1 < len(reb) else TEST_IDX[-1]
        hold = TEST_IDX[(TEST_IDX > rd) & (TEST_IDX <= end)]
        if len(hold) == 0: continue
        out.append(dict(rd=rd, alpha=a.fillna(a.median()).values, wb=wb, Sigma=S,
                        style=sty, hold=hold, cap=np.maximum(0.05, wb+0.02)))
    return out

print("\n构建20天调仓的缓存...")
t0 = time.time()
CACHE20 = build_cache(20)
print(f"✓ {len(CACHE20)}个调仓日, 耗时{time.time()-t0:.0f}秒")

freq_rows = []
for step, cache in [(5, CACHE), (20, CACHE20)]:
    wd = {}
    for c in cache:
        w = solve_flex(c, LAM_STAR, 1.0)
        if w is not None: wd[c["rd"]] = pd.Series(w, index=TICKERS)
    r_g, to = port_returns(wd, 0)
    r_n, _  = port_returns(wd, COST_BPS)
    m = perf(r_n, BENCH["市值加权84"])
    n_reb = len(wd) - 1
    freq_rows.append({"调仓频率": f"每{step}个交易日", "调仓次数": n_reb,
                      "平均单期换手": to, "年化换手(倍)": to*252/step,
                      "费前年化%": ((1+r_g).prod()**(252/len(r_g))-1)*100,
                      "费后年化%": m["年化收益%"],
                      "成本侵蚀pp": ((1+r_g).prod()**(252/len(r_g))-1)*100 - m["年化收益%"],
                      "年化超额%": m["年化超额%"], "跟踪误差%": m["跟踪误差%"],
                      "IR": m["IR"], "夏普": m["夏普"]})

print("\n" + "="*140)
print(f"调仓频率敏感性 (λ={LAM_STAR:g}, 原始约束k=1, 成本{COST_BPS}bps)")
print("="*140)
ftab = pd.DataFrame(freq_rows)
print(ftab.round(3).to_string(index=False))

d_gross = ftab.loc[1,"费前年化%"] - ftab.loc[0,"费前年化%"]
d_cost  = ftab.loc[0,"成本侵蚀pp"] - ftab.loc[1,"成本侵蚀pp"]
print(f"""
20天 vs 5天 拆解:
  alpha信号衰减代价(费前收益变化)  {d_gross:+.2f}pp
  交易成本节约                     {d_cost:+.2f}pp
  净效果(费后收益变化)             {ftab.loc[1,'费后年化%']-ftab.loc[0,'费后年化%']:+.2f}pp
  → {'成本节约大于信号衰减,降频有利' if d_cost > -d_gross else '信号衰减大于成本节约,维持5天更优'}
  注:alpha模型预测5日收益,持有20天存在信号老化,这是降频的固有代价""")

pickle.dump({"grid": grid, "sens": pd.DataFrame(sens_rows), "freq": ftab},
            open("sensitivity_results.pkl", "wb"))
print("\n✓ sensitivity_results.pkl")

In [ ]:
# ================= 第六步 任务1&2:四条净值曲线 + 指标汇总表 =================
_nr = pickle.load(open("neutral_results.pkl", "rb"))
R_NEUTRAL = _nr["neutral_beta"]

def turnover_series(wdict):
    rl = sorted(wdict.keys()); tos, ds, prev = [], [], None
    for rd in rl:
        w = wdict[rd].groupby(level=0).sum()
        w = w / w.sum()
        if prev is not None:
            a = w.index.union(prev.index)
            tos.append(float(np.abs(w.reindex(a).fillna(0) - prev.reindex(a).fillna(0)).sum()))
            ds.append(rd)
        prev = w
    return pd.Series(tos, index=ds)

TO_ENH_S = turnover_series(W_ENH)
TO_P2_S  = turnover_series(W_P2)

# ---- 统一到共同日期区间 ----
COMMON = R_ENH.index.intersection(R_P2_N.index).intersection(R_NEUTRAL.dropna().index)
CURVES = {
    "基准:标普500(^GSPC)":        (BENCH["真标普500"].reindex(COMMON), None,     None),
    "第二阶段最大夏普(有成本)":     (R_P2_N.reindex(COMMON),  BENCH["真标普500"], TO_P2_S),
    f"指数增强 λ={LAM_STAR:g}(有成本)": (R_ENH.reindex(COMMON),   BENCH["真标普500"], TO_ENH_S),
    "市场中性(有成本)":            (R_NEUTRAL.reindex(COMMON), None,             TO_P2_S),
}

def full_metrics(r, bench, to_s, name, absolute=False):
    r = r.dropna()
    ann = (1+r).prod()**(252/len(r)) - 1
    vol = r.std()*np.sqrt(252)
    nav = (1+r).cumprod(); mdd = (nav/nav.cummax()-1).min()
    mon = (1+r).resample("ME").prod() - 1
    row = {"策略": name, "年化收益%": ann*100, "年化波动%": vol*100,
           "夏普": ann/vol if vol > 0 else np.nan, "最大回撤%": mdd*100,
           "月度胜率%": (mon > 0).mean()*100}
    if absolute:                                   # 绝对收益策略:基准=现金(0)
        row.update({"年化超额%": ann*100, "跟踪误差%": vol*100,
                    "信息比率IR": ann/vol if vol > 0 else np.nan,
                    "与基准相关系数": r.corr(BENCH["真标普500"].reindex(r.index))})
    elif bench is not None:
        b = bench.reindex(r.index).dropna(); rr = r.reindex(b.index)
        ex = rr - b; te = ex.std()*np.sqrt(252)
        ann_b = (1+b).prod()**(252/len(b)) - 1
        ann_r = (1+rr).prod()**(252/len(rr)) - 1
        row.update({"年化超额%": (ann_r-ann_b)*100, "跟踪误差%": te*100,
                    "信息比率IR": (ann_r-ann_b)/te if te > 0 else np.nan,
                    "与基准相关系数": rr.corr(b)})
    else:
        row.update({"年化超额%": np.nan, "跟踪误差%": np.nan,
                    "信息比率IR": np.nan, "与基准相关系数": 1.0})
    row["最大单期换手率"] = to_s.max() if to_s is not None else np.nan
    row["平均单期换手率"] = to_s.mean() if to_s is not None else np.nan
    return row

rows = []
for name, (r, b, to_s) in CURVES.items():
    absolute = name.startswith("市场中性")
    rows.append(full_metrics(r, b, to_s, name, absolute=absolute))
SUMMARY = pd.DataFrame(rows)

print("="*156)
print(f"第三阶段 综合指标汇总表   测试期 {COMMON[0].date()} ~ {COMMON[-1].date()} ({len(COMMON)}个交易日, 单向{COST_BPS}bps)")
print("="*156)
print(SUMMARY.round(3).to_string(index=False))
print("""
口径说明:
  · 超额/跟踪误差/IR 的参照:第二阶段与指数增强相对标普500;市场中性为绝对收益策略,参照现金(0%)
    故其"超额=年化收益、跟踪误差=年化波动、IR=夏普",与基准相关系数才是其关键指标
  · 指数增强的优化目标是84只市值加权基准(非标普500),其相对该基准的表现另表列示
  · 市场中性的换手率沿用其多头腿(第二阶段组合),未计入空头端调整""")

# ---- 指数增强相对其真实跟踪目标 ----
print("\n" + "="*100)
print("补充:指数增强相对其优化目标(84只市值加权基准)")
print("="*100)
sup = [full_metrics(R_ENH.reindex(COMMON), BENCH[c].reindex(COMMON), TO_ENH_S,
                    f"指数增强 vs {c}") for c in ["市值加权84", "等权84", "真标普500"]]
sup.append(full_metrics(BENCH["市值加权84"].reindex(COMMON), None, None, "基准:市值加权84"))
print(pd.DataFrame(sup).round(3).to_string(index=False))

# ---- 四条净值曲线(同框) ----
fig = plt.figure(figsize=(14, 11))
gs = fig.add_gridspec(3, 1, height_ratios=[2.1, 1, 1], hspace=0.28)

ax0 = fig.add_subplot(gs[0])
styles = {"基准:标普500(^GSPC)": ("gray", 1.6, "--"),
          "第二阶段最大夏普(有成本)": ("#4472C4", 1.9, "-"),
          f"指数增强 λ={LAM_STAR:g}(有成本)": ("#70AD47", 2.1, "-"),
          "市场中性(有成本)": ("#C0504D", 2.2, "-")}
for name, (r, _, _) in CURVES.items():
    c, lw, ls = styles[name]
    nav = (1+r.dropna()).cumprod()
    m = SUMMARY.loc[SUMMARY["策略"] == name].iloc[0]
    ax0.plot(nav.index, nav, ls, color=c, lw=lw,
             label=f"{name}  年化{m['年化收益%']:.2f}%  夏普{m['夏普']:.2f}  回撤{m['最大回撤%']:.1f}%")
ax0.axhline(1, c="black", lw=0.8)
ax0.axvspan(pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31"), alpha=0.09, color="red")
ax0.text(pd.Timestamp("2022-05-01"), ax0.get_ylim()[1]*0.97, "2022加息熊市",
         fontsize=9, color="darkred", ha="center", va="top")
ax0.set_title("第三阶段 四类策略净值对比(均已扣除交易成本)", fontsize=14, fontweight="bold")
ax0.set_ylabel("净值(期初=1)"); ax0.legend(fontsize=9.5, loc="upper left"); ax0.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[2], sharex=ax0)
for name, c in [("第二阶段最大夏普(有成本)", "#4472C4"),
                (f"指数增强 λ={LAM_STAR:g}(有成本)", "#70AD47")]:
    r = CURVES[name][0].dropna()
    ex = (r - BENCH["真标普500"].reindex(r.index)).dropna()
    nav = (1+ex).cumprod()
    ax2.plot(nav.index, nav, color=c, lw=1.7,
             label=f"{name.split('(')[0]}:相对标普500累计超额 (终值{nav.iloc[-1]:.2f})")
r = CURVES["市场中性(有成本)"][0].dropna()
nav = (1+r).cumprod()
ax2.plot(nav.index, nav, color="#C0504D", lw=1.9,
         label=f"市场中性:绝对净值 (终值{nav.iloc[-1]:.2f}) — 其全部收益即为超市场收益")
ax2.axhline(1, c="black", lw=0.8)
ax2.set_title("超市场收益对比:前两者为相对标普500的累计超额,中性组合为其绝对净值",
              fontsize=11.5, fontweight="bold")
ax2.legend(fontsize=8.5); ax2.grid(alpha=0.3)
plt.savefig("final_comparison_phase3.png", dpi=125, bbox_inches="tight"); plt.show()

pickle.dump({"summary": SUMMARY, "supplement": pd.DataFrame(sup),
             "curves": {k: v[0] for k, v in CURVES.items()},
             "turnover": {"enh": TO_ENH_S, "p2": TO_P2_S}},
            open("phase3_final.pkl", "wb"))
print("\n✓ phase3_final.pkl, final_comparison_phase3.png")